# Neuronas alternativas

Neurona DC, neuronas flotantes, neuronas basadas en k-means, KNN, árbol de decisión, bayes y algoritmos genéticos.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

## Neurona DC: red ajustable

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# --- Funciones de Activación ---
sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide = lambda y: y * (1 - y)

# --- Dataclass para Neurona ---
@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- Dataclass para la Red ---
@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

# --- Inicialización de Capa ---
def crear_capa_dc(num_neuronas, num_entradas_por_neurona):
    return [NeuronaDC(np.random.randn(num_entradas_por_neurona) * 0.1, np.random.randn() * 0.1)
            for _ in range(num_neuronas)]

# --- Construcción de Red ---
def crear_red_dc(num_entradas, num_neuronas_ocultas, num_salidas):
    return RedNeuronalDC(
        capa_oculta=crear_capa_dc(num_neuronas_ocultas, num_entradas),
        capa_salida=crear_capa_dc(num_salidas, num_neuronas_ocultas)
    )

# --- Predicción ---
def predecir_red_dc(red, entradas):
    z_oculta = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    a_oculta = sigmoide(z_oculta)
    z_salida = np.array([np.dot(a_oculta, n.pesos) + n.sesgo for n in red.capa_salida])
    a_salida = sigmoide(z_salida)
    return a_salida, a_oculta, z_oculta, z_salida

# --- Evaluación de la red (para el conjunto de validación y prueba) ---
def evaluar_rendimiento_red(red, X_set, y_set):
    error_set = 0
    aciertos_set = 0
    for entrada, objetivo in zip(X_set, y_set):
        pred, _, _, _ = predecir_red_dc(red, entrada)
        pred_val = pred[0]
        error_set += 0.5 * (objetivo - pred_val) ** 2
        aciertos_set += int((pred_val >= 0.5) == objetivo)

    avg_error_set = error_set / len(X_set)
    precision_set = aciertos_set / len(X_set)
    return avg_error_set, precision_set


# --- Entrenamiento Principal con LR Adaptativo ---
def entrenar_red_lr_por_validacion(
    red, X_train, y_train, X_val, y_val, # Se pasan los datos de validación
    initial_lr, epochs,
    factor_reduccion_lr=0.1,
    paciencia_lr=10, # Número de épocas sin mejora en validación para reducir LR
    min_lr=1e-6,
):
    print("\n--- Iniciando Entrenamiento con Learning Rate Adaptativo por Validación ---")
    historial_errores_train = []
    historial_precisiones_train = []
    historial_errores_val = []
    historial_precisiones_val = []
    historial_lr = []

    current_lr = initial_lr
    mejor_error_val = float('inf')
    epocas_sin_mejora_val = 0

    for epoch in range(1, epochs + 1):
        historial_lr.append(current_lr) # Guardar el LR de esta época

        # --- Fase de Entrenamiento ---
        error_total_train = 0
        aciertos_train = 0

        # Mezclar datos de entrenamiento en cada época
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_shuffled, y_shuffled = X_train[indices], y_train[indices]

        for entrada, objetivo in zip(X_shuffled, y_shuffled):
            pred, a_oculta, _, _ = predecir_red_dc(red, entrada)
            pred_val = pred[0]
            salida_binaria = 1 if pred_val >= 0.5 else 0
            aciertos_train += int(salida_binaria == objetivo)

            error = objetivo - pred_val
            error_total_train += 0.5 * error ** 2

            delta_salida = error * derivada_sigmoide(pred_val)
            error_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_oculta * derivada_sigmoide(a_oculta)

            red.capa_salida[0].pesos += current_lr * delta_salida * a_oculta
            red.capa_salida[0].sesgo += current_lr * delta_salida

            for j, n in enumerate(red.capa_oculta):
                n.pesos += current_lr * delta_oculta[j] * entrada
                n.sesgo += current_lr * delta_oculta[j]

        avg_error_train = error_total_train / len(X_train)
        precision_train = aciertos_train / len(X_train)
        historial_errores_train.append(avg_error_train)
        historial_precisiones_train.append(precision_train)

        # --- Fase de Validación ---
        avg_error_val, precision_val = evaluar_rendimiento_red(red, X_val, y_val)
        historial_errores_val.append(avg_error_val)
        historial_precisiones_val.append(precision_val)

        # --- Lógica de Ajuste del Learning Rate ---
        # Monitoreamos el error de validación para el ajuste
        if avg_error_val < mejor_error_val:
            mejor_error_val = avg_error_val
            epocas_sin_mejora_val = 0
        else:
            epocas_sin_mejora_val += 1

        if epocas_sin_mejora_val >= paciencia_lr:
            new_lr = current_lr * factor_reduccion_lr
            if new_lr >= min_lr:
                current_lr = new_lr
                print(f"\n¡LR Reducido! Época {epoch}/{epochs}. Nuevo LR: {current_lr:.6f} (Error Val sin mejora por {paciencia_lr} épocas)")
                epocas_sin_mejora_val = 0 # Resetear paciencia
            else:
                current_lr = min_lr # No bajar del mínimo
                print(f"\nLR ha alcanzado el mínimo ({min_lr:.6f}) en Época {epoch}/{epoch}. No más reducciones.")

        # Impresión de progreso
        if epoch % (epochs // 10 if epochs // 10 > 0 else 1) == 0 or epoch == 1:
            print(f"Época {epoch}/{epochs} | Train (Error: {avg_error_train:.5f}, Acc: {precision_train:.5f}) | Val (Error: {avg_error_val:.5f}, Acc: {precision_val:.5f}) | LR: {current_lr:.6f}")

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores_train, historial_precisiones_train, historial_errores_val, historial_precisiones_val, historial_lr

# --- Visualización ---
def graficar_metricas_completas(historial_errores_train, historial_precisiones_train,
                                 historial_errores_val, historial_precisiones_val, historial_lr):
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.plot(historial_errores_train, label="Error Entrenamiento")
    plt.plot(historial_errores_val, label="Error Validación", linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Error")
    plt.title("Error por Época (Train vs Val)")
    plt.grid()
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(historial_precisiones_train, label="Precisión Entrenamiento")
    plt.plot(historial_precisiones_val, label="Precisión Validación", linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Precisión")
    plt.title("Precisión por Época (Train vs Val)")
    plt.grid()
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(historial_lr, color='red', linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Learning Rate", color='red')
    plt.tick_params(axis='y', labelcolor='red')
    plt.title('Evolución del Learning Rate')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# --- Visualización de la Frontera de Decisión ---
def graficar_frontera_decision(red, X, y):
    h = .02  # tamaño de paso en la malla
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Predecir sobre la malla
    Z = np.array([predecir_red_dc(red, np.array([x, y]))[0][0] for x, y in np.c_[xx.ravel(), yy.ravel()]])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k')
    plt.title("Frontera de Decisión de la Red Neuronal")
    plt.xlabel("Característica 1")
    plt.ylabel("Característica 2")
    plt.show()


# --- Main Execution ---
if __name__ == "__main__":
    # Generar el dataset principal
    n_samples_total = 1000
    X, y = make_circles(n_samples=n_samples_total, factor=0.5, noise=0.1, random_state=1)

    # 1. Separar en Entrenamiento y Test (80% / 20%)
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 2. De la parte de entrenamiento, separar en Entrenamiento y Validación (80% / 20%)
    # Esto significa que el 80% de X_train_full será para entrenamiento y el 20% para validación
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)
    # Aquí test_size=0.25 porque 0.25 * 0.8 (X_train_full) = 0.2 (del total original)
    # Así, la distribución es: 60% entrenamiento, 20% validación, 20% prueba

    print(f"Dataset de entrenamiento: {len(X_train)} muestras")
    print(f"Dataset de validación: {len(X_val)} muestras")
    print(f"Dataset de prueba: {len(X_test)} muestras")

    # Crear la red neuronal
    red = crear_red_dc(num_entradas=2, num_neuronas_ocultas=8, num_salidas=1)

    # Entrenar la red, pasando explícitamente los conjuntos de validación
    red_entrenada, err_train, acc_train_hist, err_val, acc_val_hist, lr_hist = \
        entrenar_red_lr_por_validacion(
            red, X_train, y_train, X_val, y_val, # Ahora se pasan X_val, y_val
            initial_lr=0.5,  # Empezar con una tasa relativamente alta
            epochs=500,     # Número fijo de épocas
            factor_reduccion_lr=0.2, # Reducir el LR al 20%
            paciencia_lr=20,  # Esperar 20 épocas sin mejora en validación
            min_lr=1e-5    # LR mínimo
        )

    # Evaluación final en el conjunto de prueba (nunca visto)
    print("\n--- Evaluación Final del Modelo Entrenado en el conjunto de Prueba ---")
    final_error_test, final_precision_test = evaluar_rendimiento_red(red_entrenada, X_test, y_test)

    print(f"Error final (prueba): {final_error_test:.4f}")
    print(f"Precisión final (prueba): {final_precision_test:.4f}")

    # Visualización de todas las métricas
    graficar_metricas_completas(err_train, acc_train_hist, err_val, acc_val_hist, lr_hist)
    graficar_frontera_decision(red_entrenada, X, y) # Usar X total para la visualización de la frontera

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# --- Funciones de Activación ---
sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide = lambda y: y * (1 - y)

# --- Dataclass para Neurona ---
@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- Dataclass para la Red ---
@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

# --- Inicialización de Capa ---
def crear_capa_dc(num_neuronas, num_entradas_por_neurona):
    return [NeuronaDC(np.random.randn(num_entradas_por_neurona) * 0.1, np.random.randn() * 0.1)
            for _ in range(num_neuronas)]

# --- Construcción de Red ---
def crear_red_dc(num_entradas, num_neuronas_ocultas, num_salidas):
    return RedNeuronalDC(
        capa_oculta=crear_capa_dc(num_neuronas_ocultas, num_entradas),
        capa_salida=crear_capa_dc(num_salidas, num_neuronas_ocultas)
    )

# --- Predicción ---
def predecir_red_dc(red, entradas):
    z_oculta = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    a_oculta = sigmoide(z_oculta)
    z_salida = np.array([np.dot(a_oculta, n.pesos) + n.sesgo for n in red.capa_salida])
    a_salida = sigmoide(z_salida)
    return a_salida, a_oculta, z_oculta, z_salida

# --- Evaluación de Rendimiento General (para cualquier conjunto) ---
def evaluar_rendimiento_red(red, X_set, y_set):
    error_total = 0
    aciertos = 0
    for entrada, objetivo in zip(X_set, y_set):
        pred, _, _, _ = predecir_red_dc(red, entrada)
        pred_val = pred[0]
        error_total += 0.5 * (objetivo - pred_val) ** 2
        aciertos += int((pred_val >= 0.5) == objetivo)

    avg_error = error_total / len(X_set)
    precision = aciertos / len(X_set)
    return avg_error, precision

# --- Generador de Datos de Validación Únicos ---
def generar_datos_validacion_unicos(n_samples_val, factor, noise):
    """Genera un nuevo set de datos make_circles para validación."""
    # Usamos un random_state diferente o None para asegurar datos nuevos cada vez
    return make_circles(n_samples=n_samples_val, factor=factor, noise=noise, random_state=None)


# --- Entrenamiento Principal con LR Adaptativo ---
def entrenar_red_lr_por_validacion(
    red, X_train, y_train, X_test, y_test, # Ahora recibimos X_test y y_test
    initial_lr, epochs,
    factor_reduccion_lr=0.1,
    paciencia_lr=10, # Número de épocas sin mejora en validación para reducir LR
    min_lr=1e-6,
    val_data_params={'n_samples': 50, 'factor': 0.5, 'noise': 0.1} # Parámetros para generar validación
):
    print("\n--- Iniciando Entrenamiento con Learning Rate Adaptativo por Validación ---")
    historial_errores_train = []
    historial_precisiones_train = []
    historial_errores_val = []
    historial_precisiones_val = []
    historial_errores_test = [] # Nuevo historial para test
    historial_precisiones_test = [] # Nuevo historial para test
    historial_lr = []

    current_lr = initial_lr
    mejor_error_val = float('inf')
    epocas_sin_mejora_val = 0

    for epoch in range(1, epochs + 1):
        historial_lr.append(current_lr) # Guardar el LR de esta época

        # --- Fase de Entrenamiento ---
        error_total_train = 0
        aciertos_train = 0

        # Mezclar datos de entrenamiento en cada época
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_shuffled, y_shuffled = X_train[indices], y_train[indices]

        for entrada, objetivo in zip(X_shuffled, y_shuffled):
            pred, a_oculta, _, _ = predecir_red_dc(red, entrada)
            pred_val = pred[0]
            salida_binaria = 1 if pred_val >= 0.5 else 0
            aciertos_train += int(salida_binaria == objetivo)

            error = objetivo - pred_val
            error_total_train += 0.5 * error ** 2

            delta_salida = error * derivada_sigmoide(pred_val)
            error_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_oculta * derivada_sigmoide(a_oculta)

            red.capa_salida[0].pesos += current_lr * delta_salida * a_oculta
            red.capa_salida[0].sesgo += current_lr * delta_salida

            for j, n in enumerate(red.capa_oculta):
                n.pesos += current_lr * delta_oculta[j] * entrada
                n.sesgo += current_lr * delta_oculta[j]

        avg_error_train = error_total_train / len(X_train)
        precision_train = aciertos_train / len(X_train)
        historial_errores_train.append(avg_error_train)
        historial_precisiones_train.append(precision_train)

        # --- Fase de Validación ---
        # Generar nuevos datos sintéticos para validación en cada época
        X_val_new, y_val_new = generar_datos_validacion_unicos(
            val_data_params['n_samples'], val_data_params['factor'], val_data_params['noise']
        )
        avg_error_val, precision_val = evaluar_rendimiento_red(red, X_val_new, y_val_new)
        historial_errores_val.append(avg_error_val)
        historial_precisiones_val.append(precision_val)

        # --- Fase de Prueba (monitoreo, la red NO aprende de estos datos) ---
        avg_error_test, precision_test = evaluar_rendimiento_red(red, X_test, y_test)
        historial_errores_test.append(avg_error_test)
        historial_precisiones_test.append(precision_test)

        # --- Lógica de Ajuste del Learning Rate ---
        # Monitoreamos el error de validación para el ajuste
        if avg_error_val < mejor_error_val:
            mejor_error_val = avg_error_val
            epocas_sin_mejora_val = 0
        else:
            epocas_sin_mejora_val += 1

        if epocas_sin_mejora_val >= paciencia_lr:
            new_lr = current_lr * factor_reduccion_lr
            if new_lr >= min_lr:
                current_lr = new_lr
                print(f"\n¡LR Reducido! Época {epoch}/{epochs}. Nuevo LR: {current_lr:.6f} (Error Val sin mejora por {paciencia_lr} épocas)")
                epocas_sin_mejora_val = 0 # Resetear paciencia
            else:
                current_lr = min_lr # No bajar del mínimo
                print(f"\nLR ha alcanzado el mínimo ({min_lr:.6f}) en Época {epoch}/{epochs}. No más reducciones.")

        # Impresión de progreso
        if epoch % (epochs // 10 if epochs // 10 > 0 else 1) == 0 or epoch == 1:
            print(f"Época {epoch}/{epochs} | Train (Err: {avg_error_train:.4f}, Acc: {precision_train:.4f}) | Val (Err: {avg_error_val:.4f}, Acc: {precision_val:.4f}) | Test (Err: {avg_error_test:.4f}, Acc: {precision_test:.4f}) | LR: {current_lr:.6f}")

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores_train, historial_precisiones_train, \
           historial_errores_val, historial_precisiones_val, \
           historial_errores_test, historial_precisiones_test, historial_lr

# --- Visualización ---
def graficar_metricas_completas(err_train, acc_train, err_val, acc_val, err_test, acc_test, lr_hist):
    plt.figure(figsize=(22, 6))

    # Gráfico de Errores
    plt.subplot(1, 3, 1)
    plt.plot(err_train, label="Error Entrenamiento")
    plt.plot(err_val, label="Error Validación", linestyle='--')
    plt.plot(err_test, label="Error Prueba", linestyle=':') # Nuevo para test
    plt.xlabel("Época")
    plt.ylabel("Error")
    plt.title("Error por Época (Train vs Val vs Test)")
    plt.grid()
    plt.legend()

    # Gráfico de Precisiones
    plt.subplot(1, 3, 2)
    plt.plot(acc_train, label="Precisión Entrenamiento")
    plt.plot(acc_val, label="Precisión Validación", linestyle='--')
    plt.plot(acc_test, label="Precisión Prueba", linestyle=':') # Nuevo para test
    plt.xlabel("Época")
    plt.ylabel("Precisión")
    plt.title("Precisión por Época (Train vs Val vs Test)")
    plt.grid()
    plt.legend()

    # Gráfico de Learning Rate
    plt.subplot(1, 3, 3)
    plt.plot(lr_hist, color='red', linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Learning Rate", color='red')
    plt.tick_params(axis='y', labelcolor='red')
    plt.title('Evolución del Learning Rate')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


# --- Visualización de Decisión ---
def graficar_frontera_decision(red, X, y, h=0.02):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = np.array([predecir_red_dc(red, np.array([x, y]))[0][0] for x, y in zip(xx.ravel(), yy.ravel())])
    Z = (Z >= 0.5).astype(int).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette='coolwarm', s=100, alpha=0.8, legend='full')
    plt.title("Frontera de Decisión - Red Neuronal")
    plt.xlabel("Característica 1")
    plt.ylabel("Característica 2")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- Main Execution ---
if __name__ == "__main__":
    # Generar el dataset principal
    n_samples_total = 1000
    X, y = make_circles(n_samples=n_samples_total, factor=0.5, noise=0.1, random_state=1)

    # Separar en entrenamiento y prueba (el conjunto de validación será dinámico)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Dataset de entrenamiento: {len(X_train)} muestras")
    print(f"Dataset de prueba: {len(X_test)} muestras")

    # Crear la red neuronal
    red = crear_red_dc(num_entradas=2, num_neuronas_ocultas=8, num_salidas=1)

    # Parámetros para la generación de datos de validación dinámica
    val_params = {'n_samples': 100, 'factor': 0.5, 'noise': 0.1}

    # Entrenar la red con el nuevo enfoque de LR adaptativo por validación
    red_entrenada, err_train, acc_train_hist, err_val, acc_val_hist, err_test_hist, acc_test_hist, lr_hist = \
        entrenar_red_lr_por_validacion(
            red, X_train, y_train, X_test, y_test, # Pasamos los datos de prueba
            initial_lr=0.5,
            epochs=500,
            factor_reduccion_lr=0.2,
            paciencia_lr=20,
            min_lr=1e-5,
            val_data_params=val_params
        )

    # Evaluación final en el conjunto de prueba (nunca visto)
    print("\n--- Evaluación Final del Modelo Entrenado ---")
    final_acc_train = evaluar_rendimiento_red(red_entrenada, X_train, y_train)[1]
    final_acc_test = evaluar_rendimiento_red(red_entrenada, X_test, y_test)[1] # Usar la misma función
    print(f"Precisión final (entrenamiento): {final_acc_train:.4f}")
    print(f"Precisión final (prueba): {final_acc_test:.4f}")

    # Visualización de todas las métricas
    graficar_metricas_completas(err_train, acc_train_hist, err_val, acc_val_hist, err_test_hist, acc_test_hist, lr_hist)
    graficar_frontera_decision(red_entrenada, X, y) # Usar X total para la visualización de la frontera

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# --- Funciones de Activación ---
sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide = lambda y: y * (1 - y)

# --- Dataclass para Neurona ---
@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- Dataclass para la Red ---
@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

# --- Inicialización de Capa ---
def crear_capa_dc(num_neuronas, num_entradas_por_neurona):
    return [NeuronaDC(np.random.randn(num_entradas_por_neurona) * 0.1, np.random.randn() * 0.1)
            for _ in range(num_neuronas)]

# --- Construcción de Red ---
def crear_red_dc(num_entradas, num_neuronas_ocultas, num_salidas):
    return RedNeuronalDC(
        capa_oculta=crear_capa_dc(num_neuronas_ocultas, num_entradas),
        capa_salida=crear_capa_dc(num_salidas, num_neuronas_ocultas)
    )

# --- Predicción ---
def predecir_red_dc(red, entradas):
    z_oculta = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    a_oculta = sigmoide(z_oculta)
    z_salida = np.array([np.dot(a_oculta, n.pesos) + n.sesgo for n in red.capa_salida])
    a_salida = sigmoide(z_salida)
    return a_salida, a_oculta, z_oculta, z_salida

# --- Evaluación de la red (para el conjunto de validación) ---
def evaluar_rendimiento_red(red, X_val, y_val):
    error_val = 0
    aciertos_val = 0
    for entrada, objetivo in zip(X_val, y_val):
        pred, _, _, _ = predecir_red_dc(red, entrada)
        pred_val = pred[0]
        error_val += 0.5 * (objetivo - pred_val) ** 2
        aciertos_val += int((pred_val >= 0.5) == objetivo)

    avg_error_val = error_val / len(X_val)
    precision_val = aciertos_val / len(X_val)
    return avg_error_val, precision_val

# --- Generador de Datos de Validación Únicos ---
def generar_datos_validacion_unicos(n_samples_val, factor, noise):
    """Genera un nuevo set de datos make_circles para validación."""
    # Usamos un random_state diferente o None para asegurar datos nuevos cada vez
    return make_circles(n_samples=n_samples_val, factor=factor, noise=noise, random_state=None)


# --- Entrenamiento Principal con LR Adaptativo ---
def entrenar_red_lr_por_validacion(
    red, X_train, y_train,
    initial_lr, epochs,
    factor_reduccion_lr=0.1,
    paciencia_lr=10, # Número de épocas sin mejora en validación para reducir LR
    min_lr=1e-6,
    val_data_params={'n_samples': 50, 'factor': 0.5, 'noise': 0.1} # Parámetros para generar validación
):
    print("\n--- Iniciando Entrenamiento con Learning Rate Adaptativo por Validación ---")
    historial_errores_train = []
    historial_precisiones_train = []
    historial_errores_val = []
    historial_precisiones_val = []
    historial_lr = []

    current_lr = initial_lr
    mejor_error_val = float('inf')
    epocas_sin_mejora_val = 0

    for epoch in range(1, epochs + 1):
        historial_lr.append(current_lr) # Guardar el LR de esta época

        # --- Fase de Entrenamiento ---
        error_total_train = 0
        aciertos_train = 0

        # Mezclar datos de entrenamiento en cada época
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_shuffled, y_shuffled = X_train[indices], y_train[indices]

        for entrada, objetivo in zip(X_shuffled, y_shuffled):
            pred, a_oculta, _, _ = predecir_red_dc(red, entrada)
            pred_val = pred[0]
            salida_binaria = 1 if pred_val >= 0.5 else 0
            aciertos_train += int(salida_binaria == objetivo)

            error = objetivo - pred_val
            error_total_train += 0.5 * error ** 2

            delta_salida = error * derivada_sigmoide(pred_val)
            error_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_oculta * derivada_sigmoide(a_oculta)

            red.capa_salida[0].pesos += current_lr * delta_salida * a_oculta
            red.capa_salida[0].sesgo += current_lr * delta_salida

            for j, n in enumerate(red.capa_oculta):
                n.pesos += current_lr * delta_oculta[j] * entrada
                n.sesgo += current_lr * delta_oculta[j]

        avg_error_train = error_total_train / len(X_train)
        precision_train = aciertos_train / len(X_train)
        historial_errores_train.append(avg_error_train)
        historial_precisiones_train.append(precision_train)

        # --- Fase de Validación ---
        # Generar nuevos datos sintéticos para validación en cada época
        X_val_new, y_val_new = generar_datos_validacion_unicos(
            val_data_params['n_samples'], val_data_params['factor'], val_data_params['noise']
        )
        avg_error_val, precision_val = evaluar_rendimiento_red(red, X_val_new, y_val_new)
        historial_errores_val.append(avg_error_val)
        historial_precisiones_val.append(precision_val)

        # --- Lógica de Ajuste del Learning Rate ---
        # Monitoreamos el error de validación para el ajuste
        if avg_error_val < mejor_error_val:
            mejor_error_val = avg_error_val
            epocas_sin_mejora_val = 0
        else:
            epocas_sin_mejora_val += 1

        if epocas_sin_mejora_val >= paciencia_lr:
            new_lr = current_lr * factor_reduccion_lr
            if new_lr >= min_lr:
                current_lr = new_lr
                print(f"\n¡LR Reducido! Época {epoch}/{epochs}. Nuevo LR: {current_lr:.6f} (Error Val sin mejora por {paciencia_lr} épocas)")
                epocas_sin_mejora_val = 0 # Resetear paciencia
            else:
                current_lr = min_lr # No bajar del mínimo
                print(f"\nLR ha alcanzado el mínimo ({min_lr:.6f}) en Época {epoch}/{epochs}. No más reducciones.")

        # Impresión de progreso
        if epoch % (epochs // 10 if epochs // 10 > 0 else 1) == 0 or epoch == 1:
            print(f"Época {epoch}/{epochs} | Train (Error: {avg_error_train:.5f}, Acc: {precision_train:.5f}) | Val (Error: {avg_error_val:.5f}, Acc: {precision_val:.5f}) | LR: {current_lr:.6f}")

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores_train, historial_precisiones_train, historial_errores_val, historial_precisiones_val, historial_lr

# --- Visualización ---
def graficar_metricas_completas(historial_errores_train, historial_precisiones_train,
                                 historial_errores_val, historial_precisiones_val, historial_lr):
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.plot(historial_errores_train, label="Error Entrenamiento")
    plt.plot(historial_errores_val, label="Error Validación", linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Error")
    plt.title("Error por Época (Train vs Val)")
    plt.grid()
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(historial_precisiones_train, label="Precisión Entrenamiento")
    plt.plot(historial_precisiones_val, label="Precisión Validación", linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Precisión")
    plt.title("Precisión por Época (Train vs Val)")
    plt.grid()
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(historial_lr, color='red', linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Learning Rate", color='red')
    plt.tick_params(axis='y', labelcolor='red')
    plt.title('Evolución del Learning Rate')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


# --- Main Execution ---
if __name__ == "__main__":
    # Generar el dataset principal para entrenamiento y prueba
    n_samples_total = 1000 # Un dataset más grande para tener más variabilidad
    X, y = make_circles(n_samples=n_samples_total, factor=0.5, noise=0.1, random_state=1)

    # Separar en entrenamiento y prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Dataset de entrenamiento: {len(X_train)} muestras")
    print(f"Dataset de prueba: {len(X_test)} muestras")

    # Crear la red neuronal
    red = crear_red_dc(num_entradas=2, num_neuronas_ocultas=8, num_salidas=1)

    # Parámetros para la generación de datos de validación dinámica
    val_params = {'n_samples': 100, 'factor': 0.5, 'noise': 0.1}

    # Entrenar la red con el nuevo enfoque de LR adaptativo por validación
    red_entrenada, err_train, acc_train_hist, err_val, acc_val_hist, lr_hist = \
        entrenar_red_lr_por_validacion(
            red, X_train, y_train,
            initial_lr=0.5,  # Empezar con una tasa relativamente alta
            epochs=500,     # Número fijo de épocas
            factor_reduccion_lr=0.2, # Reducir el LR al 20%
            paciencia_lr=20,  # Esperar 20 épocas sin mejora en validación
            min_lr=1e-5,    # LR mínimo
            val_data_params=val_params
        )

    # Evaluación final en el conjunto de prueba (nunca visto)
    print("\n--- Evaluación Final del Modelo Entrenado ---")
    final_acc_train = evaluar_rendimiento_red(red_entrenada, X_train, y_train)[1]
    final_acc_test = evaluar_red(red_entrenada, X_test, y_test)
    print(f"Precisión final (entrenamiento): {final_acc_train:.4f}")
    print(f"Precisión final (prueba): {final_acc_test:.4f}")

    # Visualización de todas las métricas
    graficar_metricas_completas(err_train, acc_train_hist, err_val, acc_val_hist, lr_hist)
    graficar_frontera_decision(red_entrenada, X, y) # Usar X total para la visualización de la frontera

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# --- Funciones de Activación ---
sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide = lambda y: y * (1 - y)

# --- Dataclass para Neurona ---
@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- Dataclass para la Red ---
@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

# --- Inicialización de Capa ---
def crear_capa_dc(num_neuronas, num_entradas_por_neurona):
    return [NeuronaDC(np.random.randn(num_entradas_por_neurona) * 0.1, np.random.randn() * 0.1)
            for _ in range(num_neuronas)]

# --- Construcción de Red ---
def crear_red_dc(num_entradas, num_neuronas_ocultas, num_salidas):
    return RedNeuronalDC(
        capa_oculta=crear_capa_dc(num_neuronas_ocultas, num_entradas),
        capa_salida=crear_capa_dc(num_salidas, num_neuronas_ocultas)
    )

# --- Predicción ---
def predecir_red_dc(red, entradas):
    z_oculta = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    a_oculta = sigmoide(z_oculta)
    z_salida = np.array([np.dot(a_oculta, n.pesos) + n.sesgo for n in red.capa_salida])
    a_salida = sigmoide(z_salida)
    return a_salida, a_oculta, z_oculta, z_salida

# --- Learning Rate por Segmento ---
def obtener_lr_segmentado(epoch_actual, total_epochs, initial_lr, num_segmentos, factor_reduccion):
    segmento = int((epoch_actual - 1) // (total_epochs / num_segmentos))
    return max(initial_lr * (factor_reduccion ** segmento), 1e-6)

# --- Entrenamiento ---
def entrenar_red_dc_lr_segmentos(red, X_train, y_train, initial_lr, epochs,
                                  num_segmentos=5, factor_reduccion=0.5):
    historial_errores = []
    historial_precisiones = []
    historial_lr = []

    for epoch in range(1, epochs + 1):
        lr = obtener_lr_segmentado(epoch, epochs, initial_lr, num_segmentos, factor_reduccion)
        historial_lr.append(lr)
        error_total = 0
        aciertos = 0

        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_train, y_train = X_train[indices], y_train[indices]

        for entrada, objetivo in zip(X_train, y_train):
            pred, a_oculta, _, _ = predecir_red_dc(red, entrada)
            pred_val = pred[0]
            salida_binaria = 1 if pred_val >= 0.5 else 0
            aciertos += int(salida_binaria == objetivo)

            error = objetivo - pred_val
            error_total += 0.5 * error ** 2

            delta_salida = error * derivada_sigmoide(pred_val)
            error_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_oculta * derivada_sigmoide(a_oculta)

            red.capa_salida[0].pesos += lr * delta_salida * a_oculta
            red.capa_salida[0].sesgo += lr * delta_salida

            for j, n in enumerate(red.capa_oculta):
                n.pesos += lr * delta_oculta[j] * entrada
                n.sesgo += lr * delta_oculta[j]

        historial_errores.append(error_total / len(X_train))
        historial_precisiones.append(aciertos / len(X_train))

        if epoch % (epochs // 10) == 0 or epoch == 1:
            print(f"Época {epoch}/{epochs} - Error: {historial_errores[-1]:.5f}, Precisión: {historial_precisiones[-1]:.5f}, LR: {lr:.6f}")

    return red, historial_errores, historial_precisiones, historial_lr

# --- Evaluación ---
def evaluar_red(red, X, y):
    aciertos = 0
    for entrada, objetivo in zip(X, y):
        salida, _, _, _ = predecir_red_dc(red, entrada)
        pred = 1 if salida[0] >= 0.5 else 0
        aciertos += int(pred == objetivo)
    return aciertos / len(X)

# --- Visualización ---
def graficar_metricas(historial_errores, historial_precisiones, historial_lr):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(historial_errores, label="Error")
    plt.xlabel("Época")
    plt.ylabel("Error")
    plt.title("Error por Época")
    plt.grid()
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(historial_precisiones, label="Precisión")
    plt.xlabel("Época")
    plt.ylabel("Precisión")
    plt.title("Precisión por Época")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Gráfico combinado con LR
    plt.figure(figsize=(10, 5))
    plt.plot(historial_errores, color='blue', label='Error promedio')
    ax2 = plt.gca().twinx()
    ax2.plot(historial_lr, color='red', linestyle='--', label='Learning Rate')
    ax2.set_ylabel('Learning Rate', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    plt.title('Error y Learning Rate por Época')
    plt.xlabel('Época')
    plt.grid(True, linestyle='--', alpha=0.6)
    h1, l1 = plt.gca().get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc='upper right')
    plt.tight_layout()
    plt.show()

# --- Visualización de Decisión ---
def graficar_frontera_decision(red, X, y, h=0.02):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = np.array([predecir_red_dc(red, np.array([x, y]))[0][0] for x, y in zip(xx.ravel(), yy.ravel())])
    Z = (Z >= 0.5).astype(int).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette='coolwarm', s=100, alpha=0.8, legend='full')
    plt.title("Frontera de Decisión - Red Neuronal")
    plt.xlabel("Característica 1")
    plt.ylabel("Característica 2")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- Main ---
if __name__ == "__main__":
    X, y = make_circles(n_samples=500, factor=0.5, noise=0.1, random_state=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    red = crear_red_dc(num_entradas=2, num_neuronas_ocultas=8, num_salidas=1)

    red, errores, precisiones, historial_lr = entrenar_red_dc_lr_segmentos(
        red, X_train, y_train,
        initial_lr=0.2, epochs=500,
        num_segmentos=5, factor_reduccion=0.5
    )

    acc_train = evaluar_red(red, X_train, y_train)
    acc_test = evaluar_red(red, X_test, y_test)
    print(f"\nPrecisión final (train): {acc_train:.4f}")
    print(f"Precisión final (test): {acc_test:.4f}")

    graficar_metricas(errores, precisiones, historial_lr)
    graficar_frontera_decision(red, X, y)

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (necesaria para el backpropagation)
derivada_sigmoide_de_salida = lambda y_pred: y_pred * (1 - y_pred)

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---
# (predecir_neurona_dc_simple no se usa directamente en este flujo)

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

# predecir_red_dc se modifica para devolver las 'z' y 'a' para el backprop
def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray):
    """
    Realiza una predicción con la RedNeuronalDC (propagación hacia adelante)
    y devuelve las activaciones 'a' y pre-activaciones 'z' de las capas.
    """
    # Capa oculta
    salidas_capa_oculta_z = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    salidas_capa_oculta_a = funcion_activacion_sigmoide(salidas_capa_oculta_z)

    # Capa de salida
    salidas_capa_salida_z = np.array([np.dot(salidas_capa_oculta_a, n.pesos) + n.sesgo for n in red.capa_salida])
    salida_final_a = funcion_activacion_sigmoide(salidas_capa_salida_z)

    return salida_final_a, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z

# --- Funciones de Entrenamiento con Learning Rate Adaptativo ---

def entrenar_red_dc_lr_adaptativo(
    red: RedNeuronalDC,
    X_train: np.ndarray,
    y_train: np.ndarray,
    initial_learning_rate: float,
    epochs: int, # Número fijo de épocas
    factor_reduccion_lr: float = 0.1, # Factor por el que se reduce el LR
    paciencia_lr: int = 20, # Número de épocas sin mejora antes de reducir LR
    min_lr: float = 1e-6 # Mínimo learning rate permitido
) -> tuple[RedNeuronalDC, list[float], list[float]]:
    """
    Entrena la RedNeuronalDC por un número fijo de épocas,
    ajustando el learning_rate si el error no mejora.
    """
    print("\n--- Iniciando Entrenamiento con Learning Rate Adaptativo ---")
    historial_errores = []
    historial_learning_rate = []

    current_learning_rate = initial_learning_rate
    mejor_error = float('inf')
    epocas_sin_mejora = 0

    for epoch in range(epochs):
        error_total_epoca = 0

        # Mezclar los datos para cada época
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        for i in range(X_shuffled.shape[0]):
            entrada = X_shuffled[i]
            objetivo = y_shuffled[i]

            prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)
            prediccion_final_valor = prediccion_final[0]

            error_salida = objetivo - prediccion_final_valor
            error_total_epoca += 0.5 * (error_salida ** 2)

            delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)
            error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)

            # Actualización de Pesos y Sesgos con el current_learning_rate
            red.capa_salida[0].pesos += current_learning_rate * delta_salida * salidas_capa_oculta_a
            red.capa_salida[0].sesgo += current_learning_rate * delta_salida

            for j, neurona_oculta in enumerate(red.capa_oculta):
                neurona_oculta.pesos += current_learning_rate * delta_oculta[j] * entrada
                neurona_oculta.sesgo += current_learning_rate * delta_oculta[j]

        avg_error_epoca = error_total_epoca / X_train.shape[0]
        historial_errores.append(avg_error_epoca)
        historial_learning_rate.append(current_learning_rate) # Registrar el LR actual

        # Lógica de reducción del learning rate
        if avg_error_epoca < mejor_error:
            mejor_error = avg_error_epoca
            epocas_sin_mejora = 0
        else:
            epocas_sin_mejora += 1

        if epocas_sin_mejora >= paciencia_lr:
            new_lr = current_learning_rate * factor_reduccion_lr
            if new_lr >= min_lr:
                current_learning_rate = new_lr
                print(f"  LR Reducido en Época {epoch + 1}: Nuevo LR = {current_learning_rate:.6f}")
                epocas_sin_mejora = 0 # Reiniciar contador de paciencia
            else:
                current_learning_rate = min_lr # Asegurar que no baje del mínimo
                print(f"  LR ha alcanzado el mínimo permitido ({min_lr:.6f}) en Época {epoch + 1}. No más reducciones.")


        if (epoch + 1) % (epochs // 10 if epochs // 10 > 0 else 1) == 0 or epoch == 0:
            print(f"Época {epoch + 1}/{epochs}, Error promedio: {avg_error_epoca:.6f}, LR: {current_learning_rate:.6f}")

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores, historial_learning_rate

# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.08
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")

    # --- Definir la arquitectura de la red para make_circles ---
    num_caracteristicas_entrada = X_circles.shape[1]
    neuronas_en_capa_oculta = 8
    num_clases_salida = 1

    print("\n--- Creando la Red Neuronal Inicial ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # --- Entrenar la Red con LR Adaptativo ---
    num_epocas_fijas = 500 # Épocas fijas de entrenamiento
    tasa_aprendizaje_inicial = 0.5 # Tasa de aprendizaje inicial

    red_entrenada_dc, historial_errores, historial_lr = entrenar_red_dc_lr_adaptativo(
        mi_red_dc, X_circles, y_circles,
        initial_learning_rate=tasa_aprendizaje_inicial,
        epochs=num_epocas_fijas,
        factor_reduccion_lr=0.5, # Reduce el LR a la mitad
        paciencia_lr=30, # Espera 30 épocas sin mejora antes de reducir
        min_lr=0.0001 # LR mínimo para evitar que se haga demasiado pequeño
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones_raw = []
    for entrada_muestra in X_circles:
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, entrada_muestra)
        predicciones_raw.append(final_pred_val[0])

    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (después del entrenamiento con LR adaptativo): {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(16, 7))

    # Gráfico del error a lo largo de las épocas
    plt.subplot(1, 2, 1)
    plt.plot(range(len(historial_errores)), historial_errores, color='blue', label='Error promedio')
    # Nuevo: graficar el learning rate
    ax2 = plt.gca().twinx()
    ax2.plot(range(len(historial_lr)), historial_lr, color='red', linestyle='--', label='Learning Rate')
    ax2.set_ylabel('Learning Rate', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    plt.title('Error Total y Learning Rate por Época (Entrenamiento Adaptativo)')
    plt.xlabel('Época')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)
    lines, labels = ax2.get_legend_handles_labels()
    lines2, labels2 = plt.gca().get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper right')

    # Gráfico de la clasificación (Datos originales vs. predicciones de la red entrenada)
    plt.subplot(1, 2, 2)
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    h = .02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z_raw = []
    for i in range(xx.ravel().shape[0]):
        point = np.array([xx.ravel()[i], yy.ravel()[i]])
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, point)
        Z_raw.append(final_pred_val[0])
    Z = np.array([1 if p >= 0.5 else 0 for p in Z_raw])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.title(f'Clasificación por la Red Entrenada (LR Adaptativo)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend(title='Clase', loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    print("\n--- Ejemplo de predicción para un punto individual ---")
    test_point = np.array([0.0, 0.0])
    final_pred_test, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point)
    print(f"Predicción para el punto {test_point}: {final_pred_test[0]:.4f} (Clase: {1 if final_pred_test[0] >= 0.5 else 0})")

    test_point_2 = np.array([1.0, 0.0])
    final_pred_test_2, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point_2)
    print(f"Predicción para el punto {test_point_2}: {final_pred_test_2[0]:.4f} (Clase: {1 if final_pred_test_2[0] >= 0.5 else 0})")

In [ ]:

import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Definiciones de Funciones ---

funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide_de_salida = lambda y_pred: y_pred * (1 - y_pred)

# --- 2. Dataclass para la Neurona ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona ---
# (No se modifica para este cambio)

# --- 4. Funciones para la Capa ---

def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

def predecir_red_dc(red: 'RedNeuronalDC', entradas: np.ndarray):
    salidas_capa_oculta_z = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    salidas_capa_oculta_a = funcion_activacion_sigmoide(salidas_capa_oculta_z)

    salidas_capa_salida_z = np.array([np.dot(salidas_capa_oculta_a, n.pesos) + n.sesgo for n in red.capa_salida])
    salida_final_a = funcion_activacion_sigmoide(salidas_capa_salida_z)

    return salida_final_a, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z

# --- Función para Calcular Learning Rate por Segmento ---

def obtener_learning_rate_por_segmento(
    epoch_actual: int,
    total_epochs: int,
    initial_lr: float,
    num_segmentos: int,
    factor_reduccion: float
) -> float:
    """
    Calcula el learning rate para la época actual basándose en segmentos.

    Args:
        epoch_actual (int): La época actual (empezando desde 1).
        total_epochs (int): El número total de épocas de entrenamiento.
        initial_lr (float): La tasa de aprendizaje inicial.
        num_segmentos (int): En cuántos segmentos se divide el total de épocas.
        factor_reduccion (float): El factor por el cual se reduce el LR en cada nuevo segmento.

    Returns:
        float: El learning rate ajustado para la época actual.
    """
    if num_segmentos <= 0:
        return initial_lr # No hay segmentos, retorna el LR inicial

    # Asegurarse de que epoch_actual no sea 0 para evitar problemas con divisiones
    epoch_para_calculo = max(1, epoch_actual)

    # Calcular el tamaño de cada segmento
    tamano_segmento = total_epochs / num_segmentos

    # Determinar en qué segmento estamos (base 0)
    # Por ejemplo, si tamano_segmento=100:
    # Epoca 1-100 -> segmento 0
    # Epoca 101-200 -> segmento 1
    # ...
    segmento_actual_idx = int((epoch_para_calculo - 1) // tamano_segmento)

    # El LR se reduce factor_reduccion elevado a la potencia del segmento actual
    # Por ejemplo, en el segmento 0: initial_lr * (factor_reduccion^0) = initial_lr
    # En el segmento 1: initial_lr * (factor_reduccion^1)
    # En el segmento 2: initial_lr * (factor_reduccion^2)
    adjusted_lr = initial_lr * (factor_reduccion ** segmento_actual_idx)

    # Opcional: Establecer un learning rate mínimo para evitar valores extremadamente pequeños
    min_lr = 1e-6
    return max(adjusted_lr, min_lr)


# --- Funciones de Entrenamiento con LR por Segmento ---

def entrenar_red_dc_lr_segmentos(
    red: 'RedNeuronalDC',
    X_train: np.ndarray,
    y_train: np.ndarray,
    initial_learning_rate: float,
    epochs: int, # Número fijo de épocas
    num_segmentos: int = 5, # Ej. dividir 500 épocas en 5 segmentos de 100 épocas
    factor_reduccion: float = 0.5 # Reducir el LR a la mitad en cada segmento
) -> tuple['RedNeuronalDC', list[float], list[float]]:
    """
    Entrena la RedNeuronalDC por un número fijo de épocas,
    ajustando el learning_rate en base a segmentos predefinidos.
    """
    print("\n--- Iniciando Entrenamiento con Learning Rate por Segmentos ---")
    historial_errores = []
    historial_learning_rate = []

    for epoch in range(1, epochs + 1): # Épocas de 1 a 'epochs'
        # Calcular el LR para la época actual usando la nueva función
        current_learning_rate = obtener_learning_rate_por_segmento(
            epoch, epochs, initial_learning_rate, num_segmentos, factor_reduccion
        )
        historial_learning_rate.append(current_learning_rate)

        error_total_epoca = 0

        # Mezclar los datos para cada época
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        for i in range(X_shuffled.shape[0]):
            entrada = X_shuffled[i]
            objetivo = y_shuffled[i]

            prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)
            prediccion_final_valor = prediccion_final[0]

            error_salida = objetivo - prediccion_final_valor
            error_total_epoca += 0.5 * (error_salida ** 2)

            delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)
            error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)

            # Actualización de Pesos y Sesgos con el current_learning_rate
            red.capa_salida[0].pesos += current_learning_rate * delta_salida * salidas_capa_oculta_a
            red.capa_salida[0].sesgo += current_learning_rate * delta_salida

            for j, neurona_oculta in enumerate(red.capa_oculta):
                neurona_oculta.pesos += current_learning_rate * delta_oculta[j] * entrada
                neurona_oculta.sesgo += current_learning_rate * delta_oculta[j]

        avg_error_epoca = error_total_epoca / X_train.shape[0]
        historial_errores.append(avg_error_epoca)

        if epoch % (epochs // 10 if epochs // 10 > 0 else 1) == 0 or epoch == 1:
            print(f"Época {epoch}/{epochs}, Error promedio: {avg_error_epoca:.6f}, LR actual: {current_learning_rate:.6f}")

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores, historial_learning_rate

# --- Contenedor para la Red Neuronal (Necesario para los type hints) ---
# Si no tienes esta dataclass ya definida, debes añadirla
@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

# Función auxiliar para crear la red completa
def crear_red_dc(num_entradas: int, num_neuronas_ocultas: int, num_salidas: int) -> RedNeuronalDC:
    capa_oculta = crear_capa_dc(num_neuronas_ocultas, num_entradas)
    capa_salida = crear_capa_dc(num_salidas, num_neuronas_ocultas) # La capa de salida recibe entradas de la capa oculta
    return RedNeuronalDC(capa_oculta=capa_oculta, capa_salida=capa_salida)


# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.08
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")

    # --- Definir la arquitectura de la red para make_circles ---
    num_caracteristicas_entrada = X_circles.shape[1]
    neuronas_en_capa_oculta = 8
    num_clases_salida = 1

    print("\n--- Creando la Red Neuronal Inicial ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # --- Entrenar la Red con LR por Segmentos ---
    num_epocas_fijas = 500 # Épocas fijas de entrenamiento
    tasa_aprendizaje_inicial = 0.5 # Tasa de aprendizaje inicial

    red_entrenada_dc, historial_errores, historial_lr = entrenar_red_dc_lr_segmentos(
        mi_red_dc, X_circles, y_circles,
        initial_learning_rate=tasa_aprendizaje_inicial,
        epochs=num_epocas_fijas,
        num_segmentos=5, # Dividir las 500 épocas en 5 segmentos de 100 épocas
        factor_reduccion=0.5 # Reducir el LR a la mitad en cada nuevo segmento
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones_raw = []
    for entrada_muestra in X_circles:
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, entrada_muestra)
        predicciones_raw.append(final_pred_val[0])

    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (después del entrenamiento con LR por segmentos): {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(16, 7))

    # Gráfico del error y Learning Rate a lo largo de las épocas
    plt.subplot(1, 2, 1)
    plt.plot(range(len(historial_errores)), historial_errores, color='blue', label='Error promedio')
    ax2 = plt.gca().twinx()
    ax2.plot(range(len(historial_lr)), historial_lr, color='red', linestyle='--', label='Learning Rate')
    ax2.set_ylabel('Learning Rate', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    plt.title('Error Total y Learning Rate por Época (Segmentado)')
    plt.xlabel('Época')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)
    lines, labels = ax2.get_legend_handles_labels()
    lines2, labels2 = plt.gca().get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper right')

    # Gráfico de la clasificación (Datos originales vs. predicciones de la red entrenada)
    plt.subplot(1, 2, 2)
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    h = .02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z_raw = []
    for i in range(xx.ravel().shape[0]):
        point = np.array([xx.ravel()[i], yy.ravel()[i]])
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, point)
        Z_raw.append(final_pred_val[0])
    Z = np.array([1 if p >= 0.5 else 0 for p in Z_raw])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.title(f'Clasificación por la Red Entrenada (LR Segmentado)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend(title='Clase', loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    print("\n--- Ejemplo de predicción para un punto individual ---")
    test_point = np.array([0.0, 0.0])
    final_pred_test, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point)
    print(f"Predicción para el punto {test_point}: {final_pred_test[0]:.4f} (Clase: {1 if final_pred_test[0] >= 0.5 else 0})")

    test_point_2 = np.array([1.0, 0.0])
    final_pred_test_2, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point_2)
    print(f"Predicción para el punto {test_point_2}: {final_pred_test_2[0]:.4f} (Clase: {1 if final_pred_test_2[0] >= 0.5 else 0})")

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# --- Funciones de Activación ---
sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide = lambda y: y * (1 - y)

# --- Dataclass para Neurona ---
@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- Dataclass para la Red ---
@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

# --- Inicialización de Capa ---
def crear_capa_dc(num_neuronas, num_entradas_por_neurona):
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.randn(num_entradas_por_neurona) * 0.1
        sesgo = np.random.randn() * 0.1
        capa.append(NeuronaDC(pesos, sesgo))
    return capa

# --- Construcción de Red ---
def crear_red_dc(num_entradas, num_neuronas_ocultas, num_salidas):
    capa_oculta = crear_capa_dc(num_neuronas_ocultas, num_entradas)
    capa_salida = crear_capa_dc(num_salidas, num_neuronas_ocultas)
    return RedNeuronalDC(capa_oculta, capa_salida)

# --- Predicción ---
def predecir_red_dc(red, entradas):
    z_oculta = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    a_oculta = sigmoide(z_oculta)

    z_salida = np.array([np.dot(a_oculta, n.pesos) + n.sesgo for n in red.capa_salida])
    a_salida = sigmoide(z_salida)

    return a_salida, a_oculta, z_oculta, z_salida

# --- Learning Rate por Segmento ---
def obtener_learning_rate_por_segmento(epoch_actual, total_epochs, initial_lr, num_segmentos, factor_reduccion):
    segmento = int((epoch_actual - 1) // (total_epochs / num_segmentos))
    return max(initial_lr * (factor_reduccion ** segmento), 1e-6)

# --- Entrenamiento ---
def entrenar_red_dc_lr_segmentos(
    red, X_train, y_train,
    initial_lr, epochs,
    num_segmentos=5, factor_reduccion=0.5
):
    historial_errores = []
    historial_precisiones = []

    for epoch in range(1, epochs + 1):
        lr = obtener_learning_rate_por_segmento(epoch, epochs, initial_lr, num_segmentos, factor_reduccion)
        error_total = 0
        aciertos = 0

        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_train = X_train[indices]
        y_train = y_train[indices]

        for entrada, objetivo in zip(X_train, y_train):
            pred, a_oculta, _, _ = predecir_red_dc(red, entrada)
            pred_val = pred[0]
            salida_binaria = 1 if pred_val >= 0.5 else 0

            if salida_binaria == objetivo:
                aciertos += 1

            error = objetivo - pred_val
            error_total += 0.5 * error ** 2

            delta_salida = error * derivada_sigmoide(pred_val)
            error_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_oculta * derivada_sigmoide(a_oculta)

            # Actualizar salida
            red.capa_salida[0].pesos += lr * delta_salida * a_oculta
            red.capa_salida[0].sesgo += lr * delta_salida

            # Actualizar capa oculta
            for j, n in enumerate(red.capa_oculta):
                n.pesos += lr * delta_oculta[j] * entrada
                n.sesgo += lr * delta_oculta[j]

        error_prom = error_total / len(X_train)
        precision = aciertos / len(X_train)
        historial_errores.append(error_prom)
        historial_precisiones.append(precision)

        if epoch % (epochs // 10) == 0 or epoch == 1:
            print(f"Época {epoch}/{epochs} - Error: {error_prom:.5f}, Precisión: {precision:.5f}, LR: {lr:.6f}")

    return red, historial_errores, historial_precisiones

# --- Evaluación ---
def evaluar_red(red, X, y):
    aciertos = 0
    for entrada, objetivo in zip(X, y):
        salida, _, _, _ = predecir_red_dc(red, entrada)
        pred = 1 if salida[0] >= 0.5 else 0
        if pred == objetivo:
            aciertos += 1
    return aciertos / len(X)

# --- Visualización ---
def graficar_metricas(historial_errores, historial_precisiones):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(historial_errores, label="Error")
    plt.title("Error por Época")
    plt.xlabel("Época")
    plt.ylabel("Error")
    plt.grid()
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(historial_precisiones, label="Precisión")
    plt.title("Precisión por Época")
    plt.xlabel("Época")
    plt.ylabel("Precisión")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- Main ---
if __name__ == "__main__":
    X, y = make_circles(n_samples=500, factor=0.5, noise=0.1, random_state=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    red = crear_red_dc(num_entradas=2, num_neuronas_ocultas=8, num_salidas=1)

    red, errores, precisiones = entrenar_red_dc_lr_segmentos(
        red, X_train, y_train,
        initial_lr=0.2, epochs=500,
        num_segmentos=5, factor_reduccion=0.5
    )

    acc_train = evaluar_red(red, X_train, y_train)
    acc_test = evaluar_red(red, X_test, y_test)
    print(f"\nPrecisión final (train): {acc_train:.4f}")
    print(f"Precisión final (test): {acc_test:.4f}")

    graficar_metricas(errores, precisiones)

## Neurona DC: control de hiperparámetros

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (necesaria para el backpropagation)
derivada_sigmoide_de_salida = lambda y_pred: y_pred * (1 - y_pred)

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---

def predecir_neurona_dc_simple(neurona: NeuronaDC, entradas: np.ndarray) -> float:
    """Calcula la salida de una NeuronaDC (versión simplificada para propagación)."""
    z = np.dot(entradas, neurona.pesos) + neurona.sesgo
    return funcion_activacion_sigmoide(z)

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

# NOTA: predecir_red_dc se modifica para devolver las 'z' y 'a' para el backprop
def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray):
    """
    Realiza una predicción con la RedNeuronalDC (propagación hacia adelante)
    y devuelve las activaciones 'a' y pre-activaciones 'z' de las capas.
    """
    # Capa oculta
    salidas_capa_oculta_z = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    salidas_capa_oculta_a = funcion_activacion_sigmoide(salidas_capa_oculta_z)

    # Capa de salida
    salidas_capa_salida_z = np.array([np.dot(salidas_capa_oculta_a, n.pesos) + n.sesgo for n in red.capa_salida])
    salida_final_a = funcion_activacion_sigmoide(salidas_capa_salida_z)

    return salida_final_a, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z

# --- Funciones de Entrenamiento Adaptativo ---

def entrenar_red_dc_adaptativo(
    red: RedNeuronalDC,
    X_train: np.ndarray,
    y_train: np.ndarray,
    learning_rate: float,
    error_umbral: float = 0.05, # Umbral para considerar el error "alto"
    max_iter_adaptativas: int = 5 # Límite en el número de incrementos adaptativos
) -> tuple[RedNeuronalDC, list[float]]:
    """
    Entrena la RedNeuronalDC usando el algoritmo de retropropagación
    con un número de épocas adaptativo.
    """
    print("\n--- Iniciando Entrenamiento Adaptativo de la Red Neuronal ---")
    historial_errores_total = []

    epocas_actuales = 1 # Empieza en 1
    iteracion_adaptativa = 0
    error_promedio_anterior = float('inf') # Para la primera iteración, asumimos un error alto

    while error_promedio_anterior > error_umbral and iteracion_adaptativa < max_iter_adaptativas:
        print(f"\n--- Iteración Adaptativa {iteracion_adaptativa + 1}: Entrenando por {epocas_actuales} épocas ---")

        for epoch in range(epocas_actuales):
            error_total_epoca = 0

            # Iterar sobre cada ejemplo en el dataset
            for i in range(X_train.shape[0]):
                entrada = X_train[i]
                objetivo = y_train[i]

                # Propagación Hacia Adelante (Forward Pass)
                prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)
                prediccion_final_valor = prediccion_final[0]

                # Cálculo del Error
                error_salida = objetivo - prediccion_final_valor
                error_total_epoca += 0.5 * (error_salida ** 2)

                # Retropropagación (Backpropagation)
                delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)
                error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos
                delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)

                # Actualización de Pesos y Sesgos
                red.capa_salida[0].pesos += learning_rate * delta_salida * salidas_capa_oculta_a
                red.capa_salida[0].sesgo += learning_rate * delta_salida

                for j, neurona_oculta in enumerate(red.capa_oculta):
                    neurona_oculta.pesos += learning_rate * delta_oculta[j] * entrada
                    neurona_oculta.sesgo += learning_rate * delta_oculta[j]

            avg_error_epoca = error_total_epoca / X_train.shape[0]
            historial_errores_total.append(avg_error_epoca)

            if (epoch + 1) % (epocas_actuales // 5 if epocas_actuales // 5 > 0 else 1) == 0:
                print(f"  Época {epoch + 1}/{epocas_actuales}, Error promedio: {avg_error_epoca:.6f}")

        error_promedio_anterior = historial_errores_total[-1] if historial_errores_total else float('inf')
        print(f"Error promedio al final de la iteración {iteracion_adaptativa + 1}: {error_promedio_anterior:.6f}")

        if error_promedio_anterior > error_umbral:
            print(f"Error {error_promedio_anterior:.6f} es alto (>{error_umbral}). Aumentando épocas.")
            epocas_actuales = epocas_actuales * epocas_actuales # Elevando al cuadrado
            if epocas_actuales == 0: # Para evitar bucle infinito si epocas_actuales por alguna razón se vuelve 0
                epocas_actuales = 1
        else:
            print(f"Error {error_promedio_anterior:.6f} está por debajo del umbral ({error_umbral}). Terminando entrenamiento adaptativo.")

        iteracion_adaptativa += 1

    print("--- Entrenamiento Adaptativo Finalizado ---")
    return red, historial_errores_total

# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.08 # Un poco más de ruido para hacerlo interesante
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42 # Para reproducibilidad
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")

    # --- Definir la arquitectura de la red para make_circles ---
    num_caracteristicas_entrada = X_circles.shape[1] # 2 características
    neuronas_en_capa_oculta = 8                      # Puedes experimentar con este número
    num_clases_salida = 1                            # Para clasificación binaria (0 o 1)

    print("\n--- Creando la Red Neuronal Inicial ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # --- Entrenar la Red de forma Adaptativa ---
    tasa_aprendizaje = 0.5
    error_objetivo = 0.01 # Queremos que el error baje de este valor
    max_iteraciones_adaptativas = 7 # Por ejemplo, para evitar un ciclo infinito si el error nunca baja

    red_entrenada_dc, historial_errores = entrenar_red_dc_adaptativo(
        mi_red_dc, X_circles, y_circles,
        learning_rate=tasa_aprendizaje,
        error_umbral=error_objetivo,
        max_iter_adaptativas=max_iteraciones_adaptativas
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones_raw = []
    for entrada_muestra in X_circles:
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, entrada_muestra)
        predicciones_raw.append(final_pred_val[0])

    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    # Calcular la precisión
    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (después del entrenamiento adaptativo): {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(16, 7))

    # Gráfico del error a lo largo de las épocas
    plt.subplot(1, 2, 1)
    plt.plot(range(len(historial_errores)), historial_errores, color='blue')
    plt.title('Error Total por Época (Entrenamiento Adaptativo)')
    plt.xlabel('Época Total')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)

    # Gráfico de la clasificación (Datos originales vs. predicciones de la red entrenada)
    plt.subplot(1, 2, 2)
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    # Graficar la frontera de decisión
    h = .02  # step size in the mesh
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z_raw = []
    for i in range(xx.ravel().shape[0]):
        point = np.array([xx.ravel()[i], yy.ravel()[i]])
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, point)
        Z_raw.append(final_pred_val[0])
    Z = np.array([1 if p >= 0.5 else 0 for p in Z_raw])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.title(f'Clasificación por la Red Entrenada (Adaptativo)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend(title='Clase', loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    print("\n--- Ejemplo de predicción para un punto individual ---")
    test_point = np.array([0.0, 0.0])
    final_pred_test, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point)
    print(f"Predicción para el punto {test_point}: {final_pred_test[0]:.4f} (Clase: {1 if final_pred_test[0] >= 0.5 else 0})")

    test_point_2 = np.array([1.0, 0.0])
    final_pred_test_2, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point_2)
    print(f"Predicción para el punto {test_point_2}: {final_pred_test_2[0]:.4f} (Clase: {1 if final_pred_test_2[0] >= 0.5 else 0})")

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (necesaria para el backpropagation)
derivada_sigmoide_de_salida = lambda y_pred: y_pred * (1 - y_pred)

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---

# predecir_neurona_dc_simple no se usa directamente en este flujo

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

# predecir_red_dc se modifica para devolver las 'z' y 'a' para el backprop
def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray):
    """
    Realiza una predicción con la RedNeuronalDC (propagación hacia adelante)
    y devuelve las activaciones 'a' y pre-activaciones 'z' de las capas.
    """
    # Capa oculta
    salidas_capa_oculta_z = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    salidas_capa_oculta_a = funcion_activacion_sigmoide(salidas_capa_oculta_z)

    # Capa de salida
    salidas_capa_salida_z = np.array([np.dot(salidas_capa_oculta_a, n.pesos) + n.sesgo for n in red.capa_salida])
    salida_final_a = funcion_activacion_sigmoide(salidas_capa_salida_z)

    return salida_final_a, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z

# --- Funciones de Entrenamiento Continuo Adaptativo ---

def entrenar_red_dc_continuo(
    red: RedNeuronalDC,
    X_train: np.ndarray,
    y_train: np.ndarray,
    learning_rate: float,
    epocas_iniciales: int = 100, # Épocas base para cada ciclo de entrenamiento
    error_umbral_mejora: float = 0.001, # Qué tan poco debe mejorar el error para seguir
    error_max_para_continuar: float = 0.1, # Si el error sigue por encima de este, agregamos épocas
    max_epocas_total: int = 1000 # Máximo total de épocas permitidas
) -> tuple[RedNeuronalDC, list[float]]:
    """
    Entrena la RedNeuronalDC continuamente, añadiendo épocas si el error es alto
    o no mejora lo suficiente, hasta un máximo total.
    """
    print("\n--- Iniciando Entrenamiento Continuo de la Red Neuronal ---")
    historial_errores_total = []

    epocas_completadas = 0

    # Inicialmente, el error de la última verificación es muy alto
    ultimo_error_promedio_verificado = float('inf')

    while epocas_completadas < max_epocas_total:
        epocas_para_este_paso = min(epocas_iniciales, max_epocas_total - epocas_completadas)

        if epocas_para_este_paso <= 0: # No hay más épocas que entrenar
            break

        print(f"\n--- Entrenando desde Época {epocas_completadas + 1} por {epocas_para_este_paso} épocas ---")

        for epoch_in_step in range(epocas_para_este_paso):
            current_epoch_global = epocas_completadas + epoch_in_step + 1
            error_total_epoca = 0

            for i in range(X_train.shape[0]):
                entrada = X_train[i]
                objetivo = y_train[i]

                prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)
                prediccion_final_valor = prediccion_final[0]

                error_salida = objetivo - prediccion_final_valor
                error_total_epoca += 0.5 * (error_salida ** 2)

                delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)
                error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos
                delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)

                red.capa_salida[0].pesos += learning_rate * delta_salida * salidas_capa_oculta_a
                red.capa_salida[0].sesgo += learning_rate * delta_salida

                for j, neurona_oculta in enumerate(red.capa_oculta):
                    neurona_oculta.pesos += learning_rate * delta_oculta[j] * entrada
                    neurona_oculta.sesgo += learning_rate * delta_oculta[j]

            avg_error_this_epoch = error_total_epoca / X_train.shape[0]
            historial_errores_total.append(avg_error_this_epoch)

            if current_epoch_global % (epocas_iniciales // 5 if epocas_iniciales // 5 > 0 else 1) == 0 or current_epoch_global == 1:
                print(f"  Época {current_epoch_global}/{max_epocas_total}, Error promedio: {avg_error_this_epoch:.6f}")

        epocas_completadas += epocas_para_este_paso

        # Verificación del error al final del bloque de épocas_iniciales
        if historial_errores_total:
            error_actual_promedio = historial_errores_total[-1]
            print(f"Error promedio al final de {epocas_completadas} épocas: {error_actual_promedio:.6f}")

            # Condición de parada temprana: si el error ya es muy bajo
            if error_actual_promedio <= error_umbral_mejora * 2: # Por ejemplo, el doble del umbral de mejora deseado
                print(f"Error promedio ({error_actual_promedio:.6f}) es suficientemente bajo. Terminando entrenamiento.")
                break

            # Si el error es alto O no ha mejorado significativamente
            if error_actual_promedio > error_max_para_continuar and (ultimo_error_promedio_verificado - error_actual_promedio < error_umbral_mejora):
                print(f"Error alto ({error_actual_promedio:.6f}) y/o poca mejora. Continuando entrenamiento.")
                # Aquí no se incrementa épocas_iniciales, simplemente se usa el loop while hasta max_epocas_total.
                # Si quieres que se añadan más épocas por ciclo, podrías hacer epocas_iniciales *= 2 por ejemplo.
                # Pero con un `max_epocas_total` esto ya gestiona el límite.
            elif error_actual_promedio <= error_max_para_continuar:
                print(f"Error promedio ({error_actual_promedio:.6f}) está por debajo del umbral de continuidad. Terminando entrenamiento.")
                break # Si el error ya es bajo, no tiene sentido seguir

            ultimo_error_promedio_verificado = error_actual_promedio

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores_total

# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.08
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")

    # --- Definir la arquitectura de la red para make_circles ---
    num_caracteristicas_entrada = X_circles.shape[1]
    neuronas_en_capa_oculta = 8
    num_clases_salida = 1

    print("\n--- Creando la Red Neuronal Inicial ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # --- Entrenar la Red de forma Continua Adaptativa ---
    tasa_aprendizaje = 0.5
    epocas_por_ciclo_base = 50 # Cuántas épocas se entrena antes de verificar el error
    max_epocas_permitidas = 1000 # Límite absoluto de épocas

    red_entrenada_dc, historial_errores = entrenar_red_dc_continuo(
        mi_red_dc, X_circles, y_circles,
        learning_rate=tasa_aprendizaje,
        epocas_iniciales=epocas_por_ciclo_base,
        error_umbral_mejora=0.0001, # Qué tan poca mejora se considera estancamiento
        error_max_para_continuar=0.02, # Si el error baja de esto, paramos
        max_epocas_total=max_epocas_permitidas
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones_raw = []
    for entrada_muestra in X_circles:
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, entrada_muestra)
        predicciones_raw.append(final_pred_val[0])

    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (después del entrenamiento continuo): {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(16, 7))

    plt.subplot(1, 2, 1)
    plt.plot(range(len(historial_errores)), historial_errores, color='blue')
    plt.title('Error Total por Época (Entrenamiento Continuo)')
    plt.xlabel('Época Total')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.subplot(1, 2, 2)
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    h = .02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z_raw = []
    for i in range(xx.ravel().shape[0]):
        point = np.array([xx.ravel()[i], yy.ravel()[i]])
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, point)
        Z_raw.append(final_pred_val[0])
    Z = np.array([1 if p >= 0.5 else 0 for p in Z_raw])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.title(f'Clasificación por la Red Entrenada (Continuo)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend(title='Clase', loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    print("\n--- Ejemplo de predicción para un punto individual ---")
    test_point = np.array([0.0, 0.0])
    final_pred_test, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point)
    print(f"Predicción para el punto {test_point}: {final_pred_test[0]:.4f} (Clase: {1 if final_pred_test[0] >= 0.5 else 0})")

    test_point_2 = np.array([1.0, 0.0])
    final_pred_test_2, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point_2)
    print(f"Predicción para el punto {test_point_2}: {final_pred_test_2[0]:.4f} (Clase: {1 if final_pred_test_2[0] >= 0.5 else 0})")

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (necesaria para el backpropagation)
derivada_sigmoide_de_salida = lambda y_pred: y_pred * (1 - y_pred)

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---

# predecir_neurona_dc_simple no se usa directamente en este flujo

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

# predecir_red_dc se modifica para devolver las 'z' y 'a' para el backprop
def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray):
    """
    Realiza una predicción con la RedNeuronalDC (propagación hacia adelante)
    y devuelve las activaciones 'a' y pre-activaciones 'z' de las capas.
    """
    # Capa oculta
    salidas_capa_oculta_z = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    salidas_capa_oculta_a = funcion_activacion_sigmoide(salidas_capa_oculta_z)

    # Capa de salida
    salidas_capa_salida_z = np.array([np.dot(salidas_capa_oculta_a, n.pesos) + n.sesgo for n in red.capa_salida])
    salida_final_a = funcion_activacion_sigmoide(salidas_capa_salida_z)

    return salida_final_a, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z

# --- Funciones de Entrenamiento Continuo Adaptativo ---

def entrenar_red_dc_continuo(
    red: RedNeuronalDC,
    X_train: np.ndarray,
    y_train: np.ndarray,
    learning_rate: float,
    epocas_iniciales: int = 100, # Épocas base para cada ciclo de entrenamiento
    error_umbral_mejora: float = 0.001, # Qué tan poco debe mejorar el error para seguir
    error_max_para_continuar: float = 0.1, # Si el error sigue por encima de este, agregamos épocas
    max_epocas_total: int = 10000 # Máximo total de épocas permitidas
) -> tuple[RedNeuronalDC, list[float]]:
    """
    Entrena la RedNeuronalDC continuamente, añadiendo épocas si el error es alto
    o no mejora lo suficiente, hasta un máximo total.
    """
    print("\n--- Iniciando Entrenamiento Continuo de la Red Neuronal ---")
    historial_errores_total = []

    epocas_completadas = 0

    # Inicialmente, el error de la última verificación es muy alto
    ultimo_error_promedio_verificado = float('inf')

    while epocas_completadas < max_epocas_total:
        epocas_para_este_paso = min(epocas_iniciales, max_epocas_total - epocas_completadas)

        if epocas_para_este_paso <= 0: # No hay más épocas que entrenar
            break

        print(f"\n--- Entrenando desde Época {epocas_completadas + 1} por {epocas_para_este_paso} épocas ---")

        for epoch_in_step in range(epocas_para_este_paso):
            current_epoch_global = epocas_completadas + epoch_in_step + 1
            error_total_epoca = 0

            for i in range(X_train.shape[0]):
                entrada = X_train[i]
                objetivo = y_train[i]

                prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)
                prediccion_final_valor = prediccion_final[0]

                error_salida = objetivo - prediccion_final_valor
                error_total_epoca += 0.5 * (error_salida ** 2)

                delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)
                error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos
                delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)

                red.capa_salida[0].pesos += learning_rate * delta_salida * salidas_capa_oculta_a
                red.capa_salida[0].sesgo += learning_rate * delta_salida

                for j, neurona_oculta in enumerate(red.capa_oculta):
                    neurona_oculta.pesos += learning_rate * delta_oculta[j] * entrada
                    neurona_oculta.sesgo += learning_rate * delta_oculta[j]

            avg_error_this_epoch = error_total_epoca / X_train.shape[0]
            historial_errores_total.append(avg_error_this_epoch)

            if current_epoch_global % (epocas_iniciales // 5 if epocas_iniciales // 5 > 0 else 1) == 0 or current_epoch_global == 1:
                print(f"  Época {current_epoch_global}/{max_epocas_total}, Error promedio: {avg_error_this_epoch:.6f}")

        epocas_completadas += epocas_para_este_paso

        # Verificación del error al final del bloque de épocas_iniciales
        if historial_errores_total:
            error_actual_promedio = historial_errores_total[-1]
            print(f"Error promedio al final de {epocas_completadas} épocas: {error_actual_promedio:.6f}")

            # Condición de parada temprana: si el error ya es muy bajo
            if error_actual_promedio <= error_umbral_mejora * 2: # Por ejemplo, el doble del umbral de mejora deseado
                print(f"Error promedio ({error_actual_promedio:.6f}) es suficientemente bajo. Terminando entrenamiento.")
                break

            # Si el error es alto O no ha mejorado significativamente
            if error_actual_promedio > error_max_para_continuar and (ultimo_error_promedio_verificado - error_actual_promedio < error_umbral_mejora):
                print(f"Error alto ({error_actual_promedio:.6f}) y/o poca mejora. Continuando entrenamiento.")
                # Aquí no se incrementa épocas_iniciales, simplemente se usa el loop while hasta max_epocas_total.
                # Si quieres que se añadan más épocas por ciclo, podrías hacer epocas_iniciales *= 2 por ejemplo.
                # Pero con un `max_epocas_total` esto ya gestiona el límite.
            elif error_actual_promedio <= error_max_para_continuar:
                print(f"Error promedio ({error_actual_promedio:.6f}) está por debajo del umbral de continuidad. Terminando entrenamiento.")
                break # Si el error ya es bajo, no tiene sentido seguir

            ultimo_error_promedio_verificado = error_actual_promedio

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores_total

# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.08
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")

    # --- Definir la arquitectura de la red para make_circles ---
    num_caracteristicas_entrada = X_circles.shape[1]
    neuronas_en_capa_oculta = 8
    num_clases_salida = 1

    print("\n--- Creando la Red Neuronal Inicial ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # --- Entrenar la Red de forma Continua Adaptativa ---
    tasa_aprendizaje = 9.0
    epocas_por_ciclo_base = 50 # Cuántas épocas se entrena antes de verificar el error
    max_epocas_permitidas = 1000000 # Límite absoluto de épocas

    red_entrenada_dc, historial_errores = entrenar_red_dc_continuo(
        mi_red_dc, X_circles, y_circles,
        learning_rate=tasa_aprendizaje,
        epocas_iniciales=epocas_por_ciclo_base,
        error_umbral_mejora=0.0001, # Qué tan poca mejora se considera estancamiento
        error_max_para_continuar=0.02, # Si el error baja de esto, paramos
        max_epocas_total=max_epocas_permitidas
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones_raw = []
    for entrada_muestra in X_circles:
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, entrada_muestra)
        predicciones_raw.append(final_pred_val[0])

    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (después del entrenamiento continuo): {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(16, 7))

    plt.subplot(1, 2, 1)
    plt.plot(range(len(historial_errores)), historial_errores, color='blue')
    plt.title('Error Total por Época (Entrenamiento Continuo)')
    plt.xlabel('Época Total')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.subplot(1, 2, 2)
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    h = .02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z_raw = []
    for i in range(xx.ravel().shape[0]):
        point = np.array([xx.ravel()[i], yy.ravel()[i]])
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, point)
        Z_raw.append(final_pred_val[0])
    Z = np.array([1 if p >= 0.5 else 0 for p in Z_raw])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.title(f'Clasificación por la Red Entrenada (Continuo)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend(title='Clase', loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    print("\n--- Ejemplo de predicción para un punto individual ---")
    test_point = np.array([0.0, 0.0])
    final_pred_test, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point)
    print(f"Predicción para el punto {test_point}: {final_pred_test[0]:.4f} (Clase: {1 if final_pred_test[0] >= 0.5 else 0})")

    test_point_2 = np.array([1.0, 0.0])
    final_pred_test_2, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point_2)
    print(f"Predicción para el punto {test_point_2}: {final_pred_test_2[0]:.4f} (Clase: {1 if final_pred_test_2[0] >= 0.5 else 0})")

## Neurona DC: propagación hacia adelante

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (necesaria para el backpropagation)
# x aquí es la salida de la sigmoide (y_pred)
derivada_sigmoide_de_salida = lambda y_pred: y_pred * (1 - y_pred)

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---

def predecir_neurona_dc(neurona: NeuronaDC, entradas: np.ndarray) -> float:
    """Calcula la salida de una NeuronaDC."""
    z = np.dot(entradas, neurona.pesos) + neurona.sesgo
    return funcion_activacion_sigmoide(z)

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        # Inicializamos los pesos de forma más pequeña para un mejor entrenamiento
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

def propagar_adelante_capa_dc(capa: list[NeuronaDC], entradas_a_la_capa: np.ndarray) -> np.ndarray:
    """Propaga las entradas a través de una capa de NeuronasDC."""
    salidas = [predecir_neurona_dc(neurona, entradas_a_la_capa) for neurona in capa]
    return np.array(salidas)

# --- 5. Funciones para la Red Neuronal (con dataclasses para capas) ---

@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

def crear_red_dc(num_entradas: int, num_neuronas_capa_oculta: int, num_salidas: int) -> RedNeuronalDC:
    """Crea una RedNeuronalDC."""
    capa_oculta = crear_capa_dc(num_neuronas_capa_oculta, num_entradas)
    capa_salida = crear_capa_dc(num_salidas, num_neuronas_capa_oculta)
    return RedNeuronalDC(capa_oculta=capa_oculta, capa_salida=capa_salida)

def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray) -> np.ndarray:
    """Realiza una predicción con la RedNeuronalDC (propagación hacia adelante)."""
    # Guardar las salidas de la capa oculta (se necesitan para el backprop)
    salidas_capa_oculta_z = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    salidas_capa_oculta_a = funcion_activacion_sigmoide(salidas_capa_oculta_z)

    # Guardar las salidas de la capa de salida (se necesitan para el backprop)
    salidas_capa_salida_z = np.array([np.dot(salidas_capa_oculta_a, n.pesos) + n.sesgo for n in red.capa_salida])
    salida_final_a = funcion_activacion_sigmoide(salidas_capa_salida_z)

    # Retornamos ambas, activaciones (a) y pre-activaciones (z) para backprop
    return salida_final_a, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z


# --- Funciones de Entrenamiento (corregido) ---

def entrenar_red_dc(
    red: RedNeuronalDC,
    X_train: np.ndarray,
    y_train: np.ndarray,
    learning_rate: float,
    epochs: int
) -> RedNeuronalDC:
    """
    Entrena la RedNeuronalDC usando el algoritmo de retropropagación.
    Modifica los pesos y sesgos de la red directamente.
    """
    print("\n--- Iniciando Entrenamiento de la Red Neuronal ---")
    historial_errores = []

    for epoch in range(epochs):
        error_total_epoca = 0

        # Iterar sobre cada ejemplo en el dataset
        for i in range(X_train.shape[0]):
            entrada = X_train[i]
            objetivo = y_train[i]

            # --- 1. Propagación Hacia Adelante (Forward Pass) ---
            # Ahora predecir_red_dc devuelve todas las salidas necesarias para el backprop
            prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)

            prediccion_final_valor = prediccion_final[0] # Es un array de 1 elemento

            # --- 2. Cálculo del Error (en la capa de salida) ---
            error_salida = objetivo - prediccion_final_valor
            error_total_epoca += 0.5 * (error_salida ** 2) # Usamos MSE para el monitoreo

            # --- 3. Retropropagación (Backpropagation) ---
            # 3.1. Calcular delta para la capa de salida
            # El delta de la salida usa la derivada de la sigmoide de la SALIDA REAL (prediccion_final_valor)
            delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)

            # 3.2. Calcular delta para la capa oculta
            # El error propagado a la capa oculta es el delta de la salida multiplicado por los pesos
            # que conectan la capa oculta a la de salida.
            # En este caso, red.capa_salida[0].pesos es el vector de pesos que vienen de la capa oculta.
            error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos # Esto es un vector

            # El delta final de la capa oculta es el error_propagado_a_oculta
            # multiplicado por la derivada de la sigmoide de las SALIDAS REALES de la capa oculta (salidas_capa_oculta_a)
            delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)


            # --- 4. Actualización de Pesos y Sesgos ---
            # Actualizar pesos y sesgos de la capa de salida
            # El gradiente de los pesos de salida es delta_salida * entradas_a_capa_salida (que son salidas_capa_oculta_a)
            red.capa_salida[0].pesos += learning_rate * delta_salida * salidas_capa_oculta_a
            red.capa_salida[0].sesgo += learning_rate * delta_salida

            # Actualizar pesos y sesgos de la capa oculta
            for j, neurona_oculta in enumerate(red.capa_oculta):
                # El gradiente de los pesos ocultos es delta_oculta[j] * entrada original
                neurona_oculta.pesos += learning_rate * delta_oculta[j] * entrada
                neurona_oculta.sesgo += learning_rate * delta_oculta[j]

        avg_error_epoca = error_total_epoca / X_train.shape[0]
        historial_errores.append(avg_error_epoca)
        if (epoch + 1) % (epochs // 10) == 0 or epoch == 0:
            print(f"Época {epoch + 1}/{epochs}, Error promedio: {avg_error_epoca:.6f}")

    print("--- Entrenamiento Finalizado ---")
    return red, historial_errores # Retornamos la red con los pesos y sesgos actualizados

# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.08 # Un poco más de ruido para hacerlo interesante
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42 # Para reproducibilidad
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")

    # --- Definir la arquitectura de la red para make_circles ---
    num_caracteristicas_entrada = X_circles.shape[1] # 2 características
    neuronas_en_capa_oculta = 8                      # Puedes experimentar con este número
    num_clases_salida = 1                            # Para clasificación binaria (0 o 1)

    print("\n--- Creando la Red Neuronal Inicial ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # --- Entrenar la Red ---
    tasa_aprendizaje = 0.5 # Tasa de aprendizaje
    num_epocas = 500      # Número de épocas de entrenamiento

    red_entrenada_dc, historial_errores = entrenar_red_dc(
        mi_red_dc, X_circles, y_circles, learning_rate=tasa_aprendizaje, epochs=num_epocas
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones_raw = []
    # Para la evaluación, solo necesitamos la predicción final
    for entrada_muestra in X_circles:
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, entrada_muestra)
        predicciones_raw.append(final_pred_val[0])

    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    # Calcular la precisión
    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (después de {num_epocas} épocas): {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(16, 7))

    # Gráfico del error a lo largo de las épocas
    plt.subplot(1, 2, 1)
    plt.plot(range(num_epocas), historial_errores, color='blue')
    plt.title('Error Total por Época (Entrenamiento)')
    plt.xlabel('Época')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)

    # Gráfico de la clasificación (Datos originales vs. predicciones de la red entrenada)
    plt.subplot(1, 2, 2)
    # Puntos originales con sus clases verdaderas
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    # Graficar la frontera de decisión
    h = .02  # step size in the mesh
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Predecir sobre la malla de puntos (solo necesitamos la salida final)
    Z_raw = []
    for i in range(xx.ravel().shape[0]):
        point = np.array([xx.ravel()[i], yy.ravel()[i]])
        final_pred_val, _, _, _ = predecir_red_dc(red_entrenada_dc, point)
        Z_raw.append(final_pred_val[0])
    Z = np.array([1 if p >= 0.5 else 0 for p in Z_raw])
    Z = Z.reshape(xx.shape)

    # Dibujar los contornos de la frontera de decisión
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.title(f'Clasificación por la Red Entrenada ({num_epocas} épocas)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend(title='Clase', loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    print("\n--- Ejemplo de predicción para un punto individual ---")
    test_point = np.array([0.0, 0.0]) # Un punto en el centro (círculo interior)
    final_pred_test, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point)
    print(f"Predicción para el punto {test_point}: {final_pred_test[0]:.4f} (Clase: {1 if final_pred_test[0] >= 0.5 else 0})")

    test_point_2 = np.array([1.0, 0.0]) # Un punto fuera (círculo exterior)
    final_pred_test_2, _, _, _ = predecir_red_dc(red_entrenada_dc, test_point_2)
    print(f"Predicción para el punto {test_point_2}: {final_pred_test_2[0]:.4f} (Clase: {1 if final_pred_test_2[0] >= 0.5 else 0})")

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.datasets import make_circles # Importamos make_circles
import matplotlib.pyplot as plt
import seaborn as sns # Para una mejor visualización

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (lambda)
derivada_sigmoide_de_salida = lambda y: y * (1 - y) # 'y' es la salida de la sigmoide

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---

def predecir_neurona_dc(neurona: NeuronaDC, entradas: np.ndarray) -> float:
    """Calcula la salida de una NeuronaDC."""
    z = np.dot(entradas, neurona.pesos) + neurona.sesgo
    return funcion_activacion_sigmoide(z)

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

# Una capa será una lista de NeuronaDC
def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

def propagar_adelante_capa_dc(capa: list[NeuronaDC], entradas_a_la_capa: np.ndarray) -> np.ndarray:
    """Propaga las entradas a través de una capa de NeuronasDC."""
    salidas = [predecir_neurona_dc(neurona, entradas_a_la_capa) for neurona in capa]
    return np.array(salidas)

# --- 5. Funciones para la Red Neuronal (con dataclasses para capas) ---

@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

def crear_red_dc(num_entradas: int, num_neuronas_capa_oculta: int, num_salidas: int) -> RedNeuronalDC:
    """Crea una RedNeuronalDC."""
    capa_oculta = crear_capa_dc(num_neuronas_capa_oculta, num_entradas)
    capa_salida = crear_capa_dc(num_salidas, num_neuronas_capa_oculta)
    return RedNeuronalDC(capa_oculta=capa_oculta, capa_salida=capa_salida)

def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray) -> np.ndarray:
    """Realiza una predicción con la RedNeuronalDC."""
    salidas_capa_oculta = propagar_adelante_capa_dc(red.capa_oculta, entradas)
    salida_final = propagar_adelante_capa_dc(red.capa_salida, salidas_capa_oculta)
    return salida_final

# --- Ejemplo de Uso con make_circles ---
if __name__ == "__main__":
    # --- Generar el dataset make_circles ---
    n_samples = 200
    noise_level = 0.05
    factor_val = 0.5

    X_circles, y_circles = make_circles(
        n_samples=n_samples,
        factor=factor_val,
        noise=noise_level,
        random_state=42 # Para reproducibilidad
    )

    print(f"Dataset make_circles generado: {n_samples} muestras, {X_circles.shape[1]} características.")
    print(f"Primeras 5 muestras de X:\n{X_circles[:5]}")
    print(f"Primeras 5 etiquetas de y:\n{y_circles[:5]}")

    # --- Definir la arquitectura de la red para make_circles ---
    # make_circles tiene 2 características de entrada
    num_caracteristicas_entrada = X_circles.shape[1]
    neuronas_en_capa_oculta = 8 # Puedes experimentar con este número
    # Para clasificación binaria, una salida (probabilidad entre 0 y 1)
    num_clases_salida = 1

    print("\n--- Creando la Red Neuronal (solo con dataclass y funciones) ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    print("\n--- Probando predicción con el dataset make_circles (RED NO ENTRENADA) ---")

    # Realizar predicciones para todo el dataset
    predicciones_raw = []
    for entrada_muestra in X_circles:
        prediccion_raw = predecir_red_dc(mi_red_dc, entrada_muestra)
        predicciones_raw.append(prediccion_raw[0]) # Solo nos interesa la primera (y única) salida

    # Convertir las salidas crudas (probabilidades) a clases binarias (0 o 1)
    predicciones_clase = np.array([1 if p >= 0.5 else 0 for p in predicciones_raw])

    # Calcular la precisión
    precision = np.mean(predicciones_clase == y_circles)
    print(f"Precisión de la red (sin entrenar) en make_circles: {precision:.4f}")
    print("Nota: La precisión será cercana al 50% ya que la red no ha sido entrenada y sus pesos son aleatorios.")

    # --- Visualización de las predicciones ---
    plt.figure(figsize=(10, 8))

    # Graficar los puntos originales con sus clases verdaderas
    # Eliminar el argumento 'label' ya que 'hue' se encarga de la leyenda
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles,
                    palette='coolwarm', s=100, alpha=0.8, legend='full')

    # Graficar los puntos con las predicciones de la red (para ver si acierta o no)
    # Eliminar el argumento 'label' ya que 'hue'/'style' se encargan de la leyenda
    sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=predicciones_clase,
                    marker='X', s=50, alpha=0.7, legend='full',
                    palette='viridis', style=predicciones_clase)

    plt.title('Dataset make_circles con Predicciones de Red no Entrenada')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.grid(True, linestyle='--', alpha=0.6)
    # Seaborn con legend='full' ya crea la leyenda. Si quieres modificar el título, hazlo aquí.
    plt.legend(title='Clase')
    plt.show()

    # Si quieres ver un ejemplo de una predicción individual:
    entrada_ejemplo_individual = np.array([0.1, 0.2])
    pred_individual = predecir_red_dc(mi_red_dc, entrada_ejemplo_individual)
    print(f"\nPredicción para entrada [0.1, 0.2]: {pred_individual[0]:.4f} (Clase: {1 if pred_individual[0] >= 0.5 else 0})")

In [ ]:
import numpy as np
from dataclasses import dataclass

# --- 1. Definiciones de Funciones (podrían ser Lambdas) ---

# Función de activación sigmoide (lambda)
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (lambda)
derivada_sigmoide_de_salida = lambda y: y * (1 - y) # 'y' es la salida de la sigmoide

# --- 2. Dataclass para la Neurona (solo datos, no métodos de instancia) ---

@dataclass
class NeuronaDC:
    pesos: np.ndarray
    sesgo: float

# --- 3. Funciones para Operaciones de la Neurona (reciben la NeuronaDC como argumento) ---

def predecir_neurona_dc(neurona: NeuronaDC, entradas: np.ndarray) -> float:
    """Calcula la salida de una NeuronaDC."""
    z = np.dot(entradas, neurona.pesos) + neurona.sesgo
    return funcion_activacion_sigmoide(z)

# --- 4. Funciones para la Capa (Colección de NeuronasDC) ---

# Una capa será una lista de NeuronaDC
def crear_capa_dc(num_neuronas: int, num_entradas_por_neurona: int) -> list[NeuronaDC]:
    """Crea una lista de NeuronasDC para una capa."""
    capa = []
    for _ in range(num_neuronas):
        pesos = np.random.rand(num_entradas_por_neurona) * 0.1 - 0.05
        sesgo = np.random.rand() * 0.1 - 0.05
        capa.append(NeuronaDC(pesos=pesos, sesgo=sesgo))
    return capa

def propagar_adelante_capa_dc(capa: list[NeuronaDC], entradas_a_la_capa: np.ndarray) -> np.ndarray:
    """Propaga las entradas a través de una capa de NeuronasDC."""
    salidas = [predecir_neurona_dc(neurona, entradas_a_la_capa) for neurona in capa]
    return np.array(salidas)

# --- 5. Funciones para la Red Neuronal (con dataclasses para capas) ---

@dataclass
class RedNeuronalDC:
    capa_oculta: list[NeuronaDC]
    capa_salida: list[NeuronaDC]

def crear_red_dc(num_entradas: int, num_neuronas_capa_oculta: int, num_salidas: int) -> RedNeuronalDC:
    """Crea una RedNeuronalDC."""
    capa_oculta = crear_capa_dc(num_neuronas_capa_oculta, num_entradas)
    capa_salida = crear_capa_dc(num_salidas, num_neuronas_capa_oculta)
    return RedNeuronalDC(capa_oculta=capa_oculta, capa_salida=capa_salida)

def predecir_red_dc(red: RedNeuronalDC, entradas: np.ndarray) -> np.ndarray:
    """Realiza una predicción con la RedNeuronalDC."""
    salidas_capa_oculta = propagar_adelante_capa_dc(red.capa_oculta, entradas)
    salida_final = propagar_adelante_capa_dc(red.capa_salida, salidas_capa_oculta)
    return salida_final

# --- Ejemplo de Uso ---
if __name__ == "__main__":
    num_caracteristicas_entrada = 2
    neuronas_en_capa_oculta = 4
    num_clases_salida = 1

    print("--- Creando la Red Neuronal (solo con dataclass y funciones) ---")
    mi_red_dc = crear_red_dc(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    print("\n--- Probando predicción ---")
    entrada_ejemplo = np.array([0.7, 0.3])
    prediccion = predecir_red_dc(mi_red_dc, entrada_ejemplo)

    print(f"Entrada de ejemplo: {entrada_ejemplo}")
    print(f"Predicción de la red: {prediccion}")
    print(f"Clase predicha (si es binaria): {1 if prediccion[0] >= 0.5 else 0}")

In [ ]:
import numpy as np

# Función de activación sigmoide como lambda
funcion_activacion_sigmoide_lambda = lambda x: 1 / (1 + np.exp(-x))

# La "predicción" como lambda, pero debe recibir todo (entradas, pesos, sesgo)
# directamente como argumentos, ya que no puede tener estado.
predecir_lambda = lambda entradas, pesos, sesgo: \
    funcion_activacion_sigmoide_lambda(np.dot(entradas, pesos) + sesgo)

# --- Ejemplo de uso ---
if __name__ == "__main__":
    num_entradas = 3

    # Simular la inicialización (generar pesos y sesgo fuera de cualquier "lambda")
    pesos_iniciales = np.random.rand(num_entradas) * 2 - 1
    sesgo_inicial = np.random.rand() * 2 - 1

    print(f"Pesos generados (como si fueran de una neurona): {pesos_iniciales}")
    print(f"Sesgo generado: {sesgo_inicial}")

    entradas_ejemplo_1 = np.array([0.5, 0.7, 0.2])

    # Para predecir, ahora tenemos que pasarle TODOS los parámetros
    salida_1 = predecir_lambda(entradas_ejemplo_1, pesos_iniciales, sesgo_inicial)
    print(f"\nEntradas de ejemplo 1: {entradas_ejemplo_1}")
    print(f"Salida de la neurona (ejemplo 1): {salida_1:.4f}")

    entradas_ejemplo_2 = np.array([-0.1, 0.9, 0.8])
    salida_2 = predecir_lambda(entradas_ejemplo_2, pesos_iniciales, sesgo_inicial)
    print(f"\nEntradas de ejemplo 2: {entradas_ejemplo_2}")
    print(f"Salida de la neurona (ejemplo 2): {salida_2:.4f}")

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns

# --- Funciones de la "Neurona" con Lambda ---
# Función de activación sigmoide
funcion_activacion_sigmoide = lambda x: 1 / (1 + np.exp(-x))

# Derivada de la función de activación sigmoide (necesaria para el backpropagation simple)
derivada_sigmoide = lambda x: x * (1 - x) # x aquí es la salida de la sigmoide

# Función de predicción de la neurona
# Recibe entradas, pesos y sesgo, y devuelve la salida
predecir_neurona = lambda entradas, pesos, sesgo: \
    funcion_activacion_sigmoide(np.dot(entradas, pesos) + sesgo)

# --- Proceso de Entrenamiento (Simulando el "aprendizaje" con Lambdas) ---

def entrenar_neurona_lambda(X_train, y_train, learning_rate=0.1, epochs=1000):
    # La "neurona" no tiene estado, así que gestionamos pesos y sesgo aquí.
    # Inicialización fuera de lambdas
    num_entradas = X_train.shape[1]
    pesos = np.random.rand(num_entradas) * 0.1 - 0.05 # Valores pequeños para un inicio estable
    sesgo = np.random.rand() * 0.1 - 0.05

    print("--- Inicio del Entrenamiento ---")
    print(f"Pesos iniciales: {pesos}")
    print(f"Sesgo inicial: {sesgo}")

    errores_epoca = []

    for epoch in range(epochs):
        error_total_epoca = 0

        for i in range(X_train.shape[0]):
            entrada = X_train[i]
            objetivo = y_train[i]

            # Paso 1: Predicción (Usando la lambda predecir_neurona)
            salida = predecir_neurona(entrada, pesos, sesgo)

            # Paso 2: Calcular el error
            error = objetivo - salida
            error_total_epoca += np.abs(error) # Suma el error absoluto para monitorear

            # Paso 3: Calcular el gradiente (Backpropagation simplificado)
            # Esto se basa en la regla delta para una neurona simple (Perceptrón simple)
            # Es la "contribución" de este error al ajuste de pesos/sesgo
            delta_salida = error * derivada_sigmoide(salida) # Error * derivada de la activación

            # Paso 4: Actualizar pesos y sesgo
            # Los pesos se ajustan proporcionalmente al error, a la tasa de aprendizaje y a la entrada.
            pesos += learning_rate * delta_salida * entrada
            # El sesgo se ajusta proporcionalmente al error y a la tasa de aprendizaje.
            sesgo += learning_rate * delta_salida

        # Monitoreo del entrenamiento
        avg_error_epoca = error_total_epoca / X_train.shape[0]
        errores_epoca.append(avg_error_epoca)
        if (epoch + 1) % (epochs // 10) == 0: # Imprime cada 10% de las épocas
            print(f"Época {epoch + 1}/{epochs}, Error promedio: {avg_error_epoca:.4f}")

    print("--- Entrenamiento Finalizado ---")
    print(f"Pesos finales: {pesos}")
    print(f"Sesgo final: {sesgo}")

    return pesos, sesgo, errores_epoca

# --- Generación del Dataset Sintético ---
# Usaremos make_classification para un problema de clasificación binaria simple
X, y = make_classification(
    n_samples=200,          # 200 puntos de datos
    n_features=2,           # 2 características para poder graficar
    n_informative=2,        # Ambas características son relevantes
    n_redundant=0,          # Sin características redundantes
    n_clusters_per_class=1, # Un solo cluster por clase
    n_classes=2,            # 2 clases (0 y 1)
    random_state=42,        # Para reproducibilidad
    flip_y=0.05             # Un 5% de ruido en las etiquetas
)

# --- Ejecutar el Entrenamiento ---
if __name__ == "__main__":
    tasa_aprendizaje = 0.5 # Qué tan grande es el paso de ajuste en cada iteración
    num_epocas = 500      # Cuántas veces se itera sobre todo el dataset

    # Entrenar la neurona, obteniendo los pesos y el sesgo "aprendidos"
    pesos_entrenados, sesgo_entrenado, historial_errores = entrenar_neurona_lambda(
        X, y, learning_rate=tasa_aprendizaje, epochs=num_epocas
    )

    # --- Evaluación del Modelo Entrenado ---
    print("\n--- Evaluación del Modelo Entrenado ---")
    predicciones = []
    for i in range(X.shape[0]):
        salida_raw = predecir_neurona(X[i], pesos_entrenados, sesgo_entrenado)
        # Convertimos la salida (probabilidad entre 0 y 1) a una clase binaria (0 o 1)
        clase_predicha = 1 if salida_raw >= 0.5 else 0
        predicciones.append(clase_predicha)

    # Calcular la precisión
    precision = np.mean(np.array(predicciones) == y)
    print(f"Precisión del modelo entrenado: {precision:.4f}")

    # --- Visualización del Entrenamiento y la Clasificación ---
    plt.figure(figsize=(14, 6))

    # Gráfico del error a lo largo de las épocas
    plt.subplot(1, 2, 1)
    plt.plot(range(num_epocas), historial_errores, color='blue')
    plt.title('Error Promedio por Época')
    plt.xlabel('Época')
    plt.ylabel('Error')
    plt.grid(True, linestyle='--', alpha=0.6)

    # Gráfico de la clasificación
plt.subplot(1, 2, 2)
# Puntos originales
# Se elimina el argumento 'label' ya que seaborn lo genera automáticamente con hue/style
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, style=y, palette='viridis', s=100, alpha=0.8, legend='full')

# Dibujar la frontera de decisión (para 2 características)
# La frontera de decisión para una neurona sigmoide es una línea
# w1*x1 + w2*x2 + b = 0 => x2 = (-w1*x1 - b) / w2
if pesos_entrenados[1] != 0: # Evitar división por cero
    x_plot = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
    y_plot = (-pesos_entrenados[0] * x_plot - sesgo_entrenado) / pesos_entrenados[1]
    # Si quieres etiquetar la línea de decisión, hazlo aquí
    plt.plot(x_plot, y_plot, color='red', linestyle='--', label='Frontera de Decisión (0.5)')

plt.title('Clasificación por la Neurona Entrenada')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
# legend='full' en scatterplot crea la leyenda para hue/style.
# Si también tienes una línea de decisión con label, plt.legend() combinará ambas.
plt.legend(title='Clase', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- 1. Definición de la Neurona (usando clase, esencial para el estado) ---
class Neurona:
    def __init__(self, num_entradas):
        self.pesos = np.random.rand(num_entradas) * 0.1 - 0.05 # Pesos pequeños
        self.sesgo = np.random.rand() * 0.1 - 0.05 # Sesgo pequeño

    def activar(self, x):
        # Función de activación sigmoide (o ReLU, tanh, etc.)
        return 1 / (1 + np.exp(-x))

    def predecir(self, entradas):
        # Suma ponderada + sesgo
        z = np.dot(entradas, self.pesos) + self.sesgo
        # Aplicar activación para obtener la salida de esta neurona
        return self.activar(z)

# --- 2. Definición de la Capa de Neuronas ---
class CapaNeuronal:
    def __init__(self, num_neuronas, num_entradas_por_neurona):
        self.neuronas = [Neurona(num_entradas_por_neurona) for _ in range(num_neuronas)]
        print(f"Capa con {num_neuronas} neuronas, cada una con {num_entradas_por_neurona} entradas.")

    def propagar_adelante(self, entradas_a_la_capa):
        # Cada neurona en la capa procesa las mismas entradas (o la salida de la capa anterior)
        # y produce su propia salida.
        # np.array([neurona.predecir(entradas_a_la_capa) for neurona in self.neuronas])
        # Esta forma es más eficiente con numpy si las entradas_a_la_capa son un solo ejemplo.

        # Para un solo ejemplo de entrada (vector):
        salidas_capa = []
        for neurona in self.neuronas:
            salidas_capa.append(neurona.predecir(entradas_a_la_capa))
        return np.array(salidas_capa)

# --- 3. Definición de la Red Neuronal Completa ---
class RedNeuronal:
    def __init__(self, num_entradas, num_neuronas_capa_oculta, num_salidas):
        # La primera capa oculta recibe las entradas directas
        self.capa_oculta = CapaNeuronal(num_neuronas_capa_oculta, num_entradas)

        # La capa de salida recibe las salidas de la capa oculta
        self.capa_salida = CapaNeuronal(num_salidas, num_neuronas_capa_oculta)

        print("\nRed Neuronal creada.")
        print(f"  Entradas: {num_entradas}")
        print(f"  Neuronas Capa Oculta: {num_neuronas_capa_oculta}")
        print(f"  Salidas: {num_salidas}")


    def predecir(self, entradas):
        # La entrada se propaga a través de la capa oculta
        salidas_capa_oculta = self.capa_oculta.propagar_adelante(entradas)

        # Las salidas de la capa oculta se convierten en las entradas de la capa de salida
        salida_final = self.capa_salida.propagar_adelante(salidas_capa_oculta)

        return salida_final

# --- Ejemplo de Uso de la Red Neuronal ---
if __name__ == "__main__":
    # Definimos la arquitectura de la red:
    num_caracteristicas_entrada = 2 # Por ejemplo, si usas el dataset make_classification con n_features=2
    neuronas_en_capa_oculta = 4    # Un número arbitrario de neuronas ocultas
    num_clases_salida = 1          # Para clasificación binaria (0 o 1)

    # Creamos la red neuronal
    mi_red = RedNeuronal(num_caracteristicas_entrada, neuronas_en_capa_oculta, num_clases_salida)

    # Ejemplo de una entrada de datos (un solo "data point")
    entrada_ejemplo = np.array([0.7, 0.3]) # Valores de ejemplo para 2 características

    # La red hace una predicción
    prediccion_red = mi_red.predecir(entrada_ejemplo)

    print(f"\nEntrada a la red: {entrada_ejemplo}")
    print(f"Predicción de la red (salida raw): {prediccion_red}")

    # Si es para clasificación binaria, puedes convertir a clase
    clase_predicha = 1 if prediccion_red[0] >= 0.5 else 0
    print(f"Clase predicha (si es binaria): {clase_predicha}")

    # Otro ejemplo
    entrada_ejemplo_2 = np.array([-0.5, 0.9])
    prediccion_red_2 = mi_red.predecir(entrada_ejemplo_2)
    print(f"\nEntrada a la red 2: {entrada_ejemplo_2}")
    print(f"Predicción de la red 2 (salida raw): {prediccion_red_2}")
    clase_predicha_2 = 1 if prediccion_red_2[0] >= 0.5 else 0
    print(f"Clase predicha 2 (si es binaria): {clase_predicha_2}")

In [ ]:
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Generar un dataset de clasificación simple
X, y = make_classification(
    n_samples=100,         # 100 puntos de datos
    n_features=2,          # 2 características (para poder graficarlo fácilmente)
    n_informative=2,       # Las 2 características son informativas (relevantes)
    n_redundant=0,         # 0 características redundantes
    n_clusters_per_class=1, # 1 cluster por clase
    n_classes=2,           # 2 clases (clasificación binaria)
    random_state=42        # Para reproducibilidad
)

print("Forma de X (características):", X.shape)
print("Forma de y (etiquetas):", y.shape)

# Graficar para visualizar
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette='viridis', s=80, alpha=0.8)
plt.title('Dataset Sintético de Clasificación (make_classification)')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Otro ejemplo: 3 clases, más características y un poco de ruido
X_multi, y_multi = make_classification(
    n_samples=300,
    n_features=4,
    n_informative=3,
    n_redundant=1,
    n_classes=3,
    n_clusters_per_class=2,
    random_state=21,
    flip_y=0.1 # Un poco de ruido (algunas etiquetas incorrectas)
)
print("\nForma de X_multi:", X_multi.shape)
print("Forma de y_multi:", y_multi.shape)

## Neurona DC: notas

In [ ]:
!pip install transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Busca el modelo en Hugging Face, por ejemplo:
# "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct" (puede que necesites una versión más pequeña o cuantizada)
# O un modelo como "HuggingFaceH4/starchat-alpha" si buscas algo muy pequeño
model_name = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct" # Revisa el tamaño y la compatibilidad con Colab free tier

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # Usa float16 para menor uso de memoria
    device_map="auto" # Para cargar el modelo en la GPU automáticamente
)

# Para generar código:
prompt = "Escribe una función en Python para calcular el factorial de un número."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
!pip install llama-cpp-python
# Luego descargar el archivo .gguf del modelo y cargarlo con LlamaCpp

In [ ]:
from llama_cpp import Llama

# Ruta al archivo del modelo .gguf que descargaste
# Asegúrate de que el nombre del archivo coincida exactamente con el que descargaste.
model_path = "codegemma-7b-it.Q2_K.gguf"

# Cargar el modelo
# n_gpu_layers: Número de capas del modelo que se cargarán en la GPU.
#               Pon -1 para cargarlas todas si tienes suficiente VRAM,
#               o un número menor si la GPU de Colab es limitada (0 si no quieres usar GPU).
# n_ctx: Tamaño del contexto (longitud máxima de la entrada y salida combinadas).
#        Aumenta si necesitas generar respuestas más largas o manejar prompts grandes.
# verbose: Muestra información de carga.

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1, # Intenta -1 para GPU, si falla, usa 0 (solo CPU)
    n_ctx=2048,      # Puedes ajustar esto
    verbose=True
)

# Definir el prompt para generar código
prompt = "Escribe una función en Python para calcular el factorial de un número."

# Generar la respuesta
output = llm(
    prompt,
    max_tokens=200, # Cantidad máxima de tokens a generar
    stop=["```", "</s>", "```python", "<|eot_id|>"], # Tokens para detener la generación
    echo=False,     # No repetir el prompt en la salida
    temperature=0.7 # Controla la creatividad (más alto = más creativo, más bajo = más determinista)
)

# Imprimir el código generado
print(output["choices"][0]["text"])

In [ ]:
from llama_cpp import Llama
import os # Importamos os para verificar si el archivo existe
import requests # Importamos requests para descargar el archivo si es necesario

# Ruta al archivo del modelo .gguf que descargaste o vas a descargar
# Asegúrate de que el nombre del archivo coincida exactamente con el que descargaste.
model_path = "codegemma-7b-it.Q2_K.gguf" # Ruta local en Colab, ej. /content/codegemma-7b-it.Q2_K.gguf

# URL de donde descargar el modelo (EJEMPLO: reemplaza con la URL real del archivo .gguf)
# Puedes encontrar estas URLs en repositorios de modelos en Hugging Face, por ejemplo.
# Busca en la sección "Files and versions" de un modelo GGUF compatible con Llama.cpp
model_url = "https://huggingface.co/ggerganov/gemma-2b-it-GGUF/resolve/main/gemma-2b-it.Q2_K.gguf" # <-- **REEMPLAZA ESTA URL**
# La URL anterior es para gemma-2b, no codegemma-7b. Necesitas encontrar la URL correcta.
# Un ejemplo de URL para CodeGemma podría ser algo como:
# model_url = "https://huggingface.co/chargoddard/codegemma-7b-it-gguf/resolve/main/codegemma-7b-it.Q2_K.gguf" # <-- **VERIFICA ESTA URL**


# Verificar si el archivo del modelo ya existe, si no, descargarlo
if not os.path.exists(model_path):
    print(f"El archivo del modelo '{model_path}' no encontrado. Descargando...")
    try:
        # Usar !wget para descargar el archivo. Asegúrate de que la URL sea correcta.
        # También puedes usar la librería `requests` para descargar si prefieres Python puro.
        # Por simplicidad en Colab, !wget es a menudo conveniente.
        # !wget {model_url} -O {model_path} # <-- Descomentar y usar si la URL es correcta y prefieres wget

        # Alternativa con requests:
        response = requests.get(model_url, stream=True)
        response.raise_for_status() # Lanza una excepción para códigos de estado de error (4xx o 5xx)
        with open(model_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Descarga completa: '{model_path}'")

    except requests.exceptions.RequestException as e:
        print(f"Error al descargar el archivo del modelo desde {model_url}: {e}")
        print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
        # Puedes agregar un 'exit()' o 'raise' aquí si no quieres continuar sin el modelo.
        raise e # Volvemos a lanzar la excepción para detener la ejecución

    except Exception as e:
         print(f"Ocurrió un error inesperado durante la descarga: {e}")
         raise e # Volvemos a lanzar la excepción


# Cargar el modelo
# n_gpu_layers: Número de capas del modelo que se cargarán en la GPU.
#               Pon -1 para cargarlas todas si tienes suficiente VRAM,
#               o un número menor si la GPU de Colab es limitada (0 si no quieres usar GPU).
# n_ctx: Tamaño del contexto (longitud máxima de la entrada y salida combinadas).
#        Aumenta si necesitas generar respuestas más largas o manejar prompts grandes.
# verbose: Muestra información de carga.

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1, # Intenta -1 para GPU, si falla, usa 0 (solo CPU)
    n_ctx=2048,      # Puedes ajustar esto
    verbose=True
)

# Definir el prompt para generar código
prompt = "Escribe una función en Python para calcular el factorial de un número."

# Generar la respuesta
output = llm(
    prompt,
    max_tokens=200, # Cantidad máxima de tokens a generar
    stop=["", "", "python", "<|eot_id|>"], # Tokens para detener la generación
    echo=False,     # No repetir el prompt en la salida
    temperature=0.7 # Controla la creatividad (más alto = más creativo, más bajo = más determinista)
)

# Imprimir el código generado
print(output["choices"][0]["text"])

In [ ]:
from llama_cpp import Llama
import os # Importamos os para verificar si el archivo existe
import requests # Mantenemos requests importado por si acaso, aunque usaremos wget para descargar
import sys # Importamos sys para salir si la descarga falla

# Ruta al archivo del modelo .gguf que descargaste o vas a descargar
# Asegúrate de que el nombre del archivo coincida exactamente con el que descargaste.
model_path = "codegemma-7b-it.Q2_K.gguf" # Ruta local en Colab, ej. /content/codegemma-7b-it.Q2_K.gguf

# URL de donde descargar el modelo (EJEMPLO: reemplaza con la URL real del archivo .gguf)
# Puedes encontrar estas URLs en repositorios de modelos en Hugging Face, por ejemplo.
# Busca en la sección "Files and versions" de un modelo GGUF compatible con Llama.cpp
# La URL original causó un 401 error. Intentemos con una URL más probable para CodeGemma.
# model_url = "https://huggingface.co/ggerganov/gemma-2b-it-GGUF/resolve/main/gemma-2b-it.Q2_K.gguf" # <-- URL original (gemma-2b, causó error)

# **VERIFICA SI ESTA URL ES LA CORRECTA Y ACTIVA**
# Busca en Hugging Face repositorios que contengan versiones GGUF de CodeGemma 7B IT.
# Un ejemplo común es el repositorio de "chargoddard".
model_url = "https://huggingface.co/chargoddard/codegemma-7b-it-gguf/resolve/main/codegemma-7b-it.Q2_K.gguf" # <-- Probable URL correcta

# Verificar si el archivo del modelo ya existe, si no, descargarlo
if not os.path.exists(model_path):
    print(f"El archivo del modelo '{model_path}' no encontrado. Descargando desde {model_url}...")
    try:
        # Usamos !wget para descargar el archivo. Es a menudo más robusto en Colab/shell.
        # -O {model_path} guarda el archivo con el nombre especificado en model_path.
        # -q para que sea más silencioso, --show-progress para ver el progreso.
        get_ipython().system(f'wget -q --show-progress {model_url} -O {model_path}')

        # Verificar si la descarga fue exitosa (comprobando si el archivo existe y no está vacío)
        if os.path.exists(model_path) and os.path.getsize(model_path) > 0:
             print(f"Descarga completa: '{model_path}' ({os.path.getsize(model_path)} bytes)")
        else:
             print(f"Error: La descarga de '{model_url}' no fue exitosa. El archivo '{model_path}' no se creó o está vacío.")
             print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
             # Salir o lanzar una excepción si la descarga falla críticamente
             sys.exit("Error de descarga. Saliendo.") # Detiene la ejecución de la celda y las siguientes
             # raise FileNotFoundError(f"Descarga fallida para {model_url}")


    except Exception as e:
         print(f"Ocurrió un error inesperado durante la descarga con wget: {e}")
         print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
         sys.exit(f"Error inesperado durante la descarga: {e}. Saliendo.") # Detiene la ejecución


# Cargar el modelo
# n_gpu_layers: Número de capas del modelo que se cargarán en la GPU.
#               Pon -1 para cargarlas todas si tienes suficiente VRAM,
#               o un número menor si la GPU de Colab es limitada (0 si no quieres usar GPU).
# n_ctx: Tamaño del contexto (longitud máxima de la entrada y salida combinadas).
#        Aumenta si necesitas generar respuestas más largas o manejar prompts grandes.
# verbose: Muestra información de carga.

print(f"\nCargando modelo Llama.cpp desde '{model_path}'...")

try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1, # Intenta -1 para GPU, si falla, usa 0 (solo CPU)
        n_ctx=2048,      # Puedes ajustar esto
        verbose=True
    )
    print("Modelo cargado exitosamente.")
except Exception as e:
    print(f"Error al cargar el modelo Llama.cpp: {e}")
    print("Asegúrate de que el archivo .gguf sea válido y compatible con la versión instalada de llama-cpp-python.")
    print("Intenta ajustar 'n_gpu_layers'. Si el error persiste, el archivo del modelo podría estar dañado o ser incorrecto.")
    sys.exit(f"Error de carga del modelo: {e}. Saliendo.")


# Definir el prompt para generar código
prompt = "Escribe una función en Python para calcular el factorial de un número."

# Generar la respuesta
from llama_cpp import Llama
import os # Importamos os para verificar si el archivo existe
import requests # Mantenemos requests importado por si acaso, aunque usaremos wget para descargar
import sys # Importamos sys para salir si la descarga falla

# Ruta al archivo del modelo .gguf que descargaste o vas a descargar
# Asegúrate de que el nombre del archivo coincida exactamente con el que descargaste.
model_path = "codegemma-7b-it.Q2_K.gguf" # Ruta local en Colab, ej. /content/codegemma-7b-it.Q2_K.gguf

# URL de donde descargar el modelo (EJEMPLO: reemplaza con la URL real del archivo .gguf)
# Puedes encontrar estas URLs en repositorios de modelos en Hugging Face, por ejemplo.
# Busca en la sección "Files and versions" de un modelo GGUF compatible con Llama.cpp
# La URL original causó un 401 error. Intentemos con una URL más probable para CodeGemma.
# model_url = "https://huggingface.co/ggerganov/gemma-2b-it-GGUF/resolve/main/gemma-2b-it.Q2_K.gguf" # <-- URL original (gemma-2b, causó error)

# **VERIFICA SI ESTA URL ES LA CORRECTA Y ACTIVA**
# Busca en Hugging Face repositorios que contengan versiones GGUF de CodeGemma 7B IT.
# Un ejemplo común es el repositorio de "chargoddard".
model_url = "https://huggingface.co/chargoddard/codegemma-7b-it-gguf/resolve/main/codegemma-7b-it.Q2_K.gguf" # <-- Probable URL correcta

# Verificar si el archivo del modelo ya existe, si no, descargarlo
if not os.path.exists(model_path):
    print(f"El archivo del modelo '{model_path}' no encontrado. Descargando desde {model_url}...")
    try:
        # Usamos !wget para descargar el archivo. Es a menudo más robusto en Colab/shell.
        # -O {model_path} guarda el archivo con el nombre especificado en model_path.
        # -q para que sea más silencioso, --show-progress para ver el progreso.
        get_ipython().system(f'wget -q --show-progress {model_url} -O {model_path}')

        # Verificar si la descarga fue exitosa (comprobando si el archivo existe y no está vacío)
        if os.path.exists(model_path) and os.path.getsize(model_path) > 0:
             print(f"Descarga completa: '{model_path}' ({os.path.getsize(model_path)} bytes)")
        else:
             print(f"Error: La descarga de '{model_url}' no fue exitosa. El archivo '{model_path}' no se creó o está vacío.")
             print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
             # Salir o lanzar una excepción si la descarga falla críticamente
             sys.exit("Error de descarga. Saliendo.") # Detiene la ejecución de la celda y las siguientes
             # raise FileNotFoundError(f"Descarga fallida para {model_url}")


    except Exception as e:
         print(f"Ocurrió un error inesperado durante la descarga con wget: {e}")
         print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
         sys.exit(f"Error inesperado durante la descarga: {e}. Saliendo.") # Detiene la ejecución


# Cargar el modelo
# n_gpu_layers: Número de capas del modelo que se cargarán en la GPU.
#               Pon -1 para cargarlas todas si tienes suficiente VRAM,
#               o un número menor si la GPU de Colab es limitada (0 si no quieres usar GPU).
# n_ctx: Tamaño del contexto (longitud máxima de la entrada y salida combinadas).
#        Aumenta si necesitas generar respuestas más largas o manejar prompts grandes.
# verbose: Muestra información de carga.

print(f"\nCargando modelo Llama.cpp desde '{model_path}'...")

try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1, # Intenta -1 para GPU, si falla, usa 0 (solo CPU)
        n_ctx=2048,      # Puedes ajustar esto
        verbose=True
    )
    print("Modelo cargado exitosamente.")
except Exception as e:
    print(f"Error al cargar el modelo Llama.cpp: {e}")
    print("Asegúrate de que el archivo .gguf sea válido y compatible con la versión instalada de llama-cpp-python.")
    print("Intenta ajustar 'n_gpu_layers'. Si el error persiste, el archivo del modelo podría estar dañado o ser incorrecto.")
    sys.exit(f"Error de carga del modelo: {e}. Saliendo.")


# Definir el prompt para generar código
prompt = "Escribe una función en Python para calcular el factorial de un número."

# Generar la respuesta
print("\nGenerando respuesta...")
try:
    output = llm(
        prompt,
        max_tokens=200, # Cantidad máxima de tokens a generar
        # Asegúrate de que estos tokens de parada sean correctos para el modelo CodeGemma IT.
        # Los tokens comunes son "<end_of_turn>", "<|eot_id|>", etc.
        # Consulta la documentación o el repositorio del modelo específico.
        stop=["<end_of_turn>", "<|eot_id|>", "</s>"], # Ejemplo de tokens de parada
        echo=False # No incluir el prompt en la salida
    )

    # Imprimir la respuesta generada
    # La respuesta generada está dentro de la clave 'choices' de la salida,
    # y el texto está dentro de la clave 'text' del primer (y usualmente único) elemento.
    generated_text = output["choices"][0]["text"]
    print("\n--- Código Generado ---")
    print(generated_text)
    print("-----------------------")

except Exception as e:
    print(f"Error al generar texto con el modelo: {e}")
    print("Verifica los parámetros de generación (max_tokens, stop tokens, etc.) y la carga del modelo.")
    sys.exit(f"Error de generación: {e}. Saliendo.")
import os # Importamos os para verificar si el archivo existe
import requests # Mantenemos requests importado por si acaso, aunque usaremos wget para descargar
import sys # Importamos sys para salir si la descarga falla

# Ruta al archivo del modelo .gguf que descargaste o vas a descargar
# Asegúrate de que el nombre del archivo coincida exactamente con el que descargaste.
model_path = "codegemma-7b-it.Q2_K.gguf" # Ruta local en Colab, ej. /content/codegemma-7b-it.Q2_K.gguf

# URL de donde descargar el modelo (EJEMPLO: reemplaza con la URL real del archivo .gguf)
# Puedes encontrar estas URLs en repositorios de modelos en Hugging Face, por ejemplo.
# Busca en la sección "Files and versions" de un modelo GGUF compatible con Llama.cpp
# La URL original causó un 401 error. Intentemos con una URL más probable para CodeGemma.
# model_url = "https://huggingface.co/ggerganov/gemma-2b-it-GGUF/resolve/main/gemma-2b-it.Q2_K.gguf" # <-- URL original (gemma-2b, causó error)

# **VERIFICA SI ESTA URL ES LA CORRECTA Y ACTIVA**
# Busca en Hugging Face repositorios que contengan versiones GGUF de CodeGemma 7B IT.
# Un ejemplo común es el repositorio de "chargoddard".
model_url = "https://huggingface.co/chargoddard/codegemma-7b-it-gguf/resolve/main/codegemma-7b-it.Q2_K.gguf" # <-- Probable URL correcta

# Verificar si el archivo del modelo ya existe, si no, descargarlo
if not os.path.exists(model_path):
    print(f"El archivo del modelo '{model_path}' no encontrado. Descargando desde {model_url}...")
    try:
        # Usamos !wget para descargar el archivo. Es a menudo más robusto en Colab/shell.
        # -O {model_path} guarda el archivo con el nombre especificado en model_path.
        # -q para que sea más silencioso, --show-progress para ver el progreso.
        get_ipython().system(f'wget -q --show-progress {model_url} -O {model_path}')

        # Verificar si la descarga fue exitosa (comprobando si el archivo existe y no está vacío)
        if os.path.exists(model_path) and os.path.getsize(model_path) > 0:
             print(f"Descarga completa: '{model_path}' ({os.path.getsize(model_path)} bytes)")
        else:
             print(f"Error: La descarga de '{model_url}' no fue exitosa. El archivo '{model_path}' no se creó o está vacío.")
             print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
             # Salir o lanzar una excepción si la descarga falla críticamente
             sys.exit("Error de descarga. Saliendo.") # Detiene la ejecución de la celda y las siguientes
             # raise FileNotFoundError(f"Descarga fallida para {model_url}")


    except Exception as e:
         print(f"Ocurrió un error inesperado durante la descarga con wget: {e}")
         print("Por favor, verifica la URL del modelo o descarga el archivo manualmente y súbelo a Colab.")
         sys.exit(f"Error inesperado durante la descarga: {e}. Saliendo.") # Detiene la ejecución


# Cargar el modelo
# n_gpu_layers: Número de capas del modelo que se cargarán en la GPU.
#               Pon -1 para cargarlas todas si tienes suficiente VRAM,
#               o un número menor si la GPU de Colab es limitada (0 si no quieres usar GPU).
# n_ctx: Tamaño del contexto (longitud máxima de la entrada y salida combinadas).
#        Aumenta si necesitas generar respuestas más largas o manejar prompts grandes.
# verbose: Muestra información de carga.

print(f"\nCargando modelo Llama.cpp desde '{model_path}'...")

try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1, # Intenta -1 para GPU, si falla, usa 0 (solo CPU)
        n_ctx=2048,      # Puedes ajustar esto
        verbose=True
    )
    print("Modelo cargado exitosamente.")
except Exception as e:
    print(f"Error al cargar el modelo Llama.cpp: {e}")
    print("Asegúrate de que el archivo .gguf sea válido y compatible con la versión instalada de llama-cpp-python.")
    print("Intenta ajustar 'n_gpu_layers'. Si el error persiste, el archivo del modelo podría estar dañado o ser incorrecto.")
    sys.exit(f"Error de carga del modelo: {e}. Saliendo.")


# Definir el prompt para generar código
prompt = "Escribe una función en Python para calcular el factorial de un número."

# Generar la respuesta
print("\nGenerando respuesta...")
try:
    output = llm(
        prompt,
        max_tokens=200, # Cantidad máxima de tokens a generar
        # Asegúrate de que estos tokens de parada sean correctos para el modelo CodeGemma IT.
        # Los tokens comunes son "<end_of_turn>", "<|eot_id|>", etc.
        # Consulta la documentación o el repositorio del modelo específico.
        stop=["<end_of_turn>", "<|eot_id|>", "</s>"], # Ejemplo de tokens de parada
        echo=False # No incluir el prompt en la salida
    )

    # Imprimir la respuesta generada
    # La respuesta generada está dentro de la clave 'choices' de la salida,
    # y el texto está dentro de la clave 'text' del primer (y usualmente único) elemento.
    generated_text = output["choices"][0]["text"]
    print("\n--- Código Generado ---")
    print(generated_text)
    print("-----------------------")

except Exception as e:
    print(f"Error al generar texto con el modelo: {e}")
    print("Verifica los parámetros de generación (max_tokens, stop tokens, etc.) y la carga del modelo.")
    sys.exit(f"Error de generación: {e}. Saliendo.")

## Neuronas flotantes cargadas desde archivos

In [ ]:
pip install transformers

In [ ]:
import pandas as pd
onda= ("/content/DJ/palabras_con_embedding_y_posicion.pkl")
df = pd.read_pickle(onda)
print(df)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.manifold import TSNE
import pandas as pd
import os # Import the os module

# --- Configuración del modelo BERT ---
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# --- Frase de prueba ---
sentence = "The house is big and has a beautiful garden with flowers."

# --- Tokenización ---
inputs = tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# Extraemos los embeddings de la última capa oculta (batch_size, seq_len, hidden_size)
embeddings = outputs.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze(0))

# --- Filtrado: removemos [CLS], [SEP] y tokens subword (##)
valid_data = [
    (token, embeddings[i].numpy())
    for i, token in enumerate(tokens)
    if token not in ["[CLS]", "[SEP]"] and not token.startswith("##")
]

words, vectors = zip(*valid_data)

# --- Reducción a 2D con t-SNE ---
# Adjust perplexity if the number of samples is small
perplexity_value = min(len(vectors) - 1, 30) # Default perplexity is 30, but needs to be < n_samples
if perplexity_value < 1: # Perplexity must be greater than 1
    print("Warning: Not enough data points to perform t-SNE with perplexity > 1. Skipping t-SNE.")
    # Handle the case where t-SNE is skipped, perhaps by assigning default positions or raising an error
    positions_2d = np.zeros((len(vectors), 2)) # Assign dummy positions if skipping
else:
    tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity_value)
    positions_2d = tsne.fit_transform(np.stack(vectors))


# --- Estructura final: palabra, x, y, vector768 ---
records = []
# Ensure that all lists for zip have the same length, especially if t-SNE was skipped
if len(words) == len(positions_2d) == len(vectors):
    for word, pos, vec in zip(words, positions_2d, vectors):
        record = {
            "word": word,
            "x": pos[0],
            "y": pos[1],
            "vector768": vec
        }
        records.append(record)
else:
    print("Error: Mismatch in list lengths after data processing. Cannot create records.")


# --- Guardar a archivo ---
df = pd.DataFrame(records)

# Define the directory path
output_dir = "/content/DJ"
# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the full file paths
pickle_path = os.path.join(output_dir, "palabras_con_embedding_y_posicion.pkl")
csv_path = os.path.join(output_dir, "palabras_con_posicion.csv")

# Save to pickle
df.to_pickle(pickle_path)  # formato binario eficiente
print(f"Saved pickle file to: {pickle_path}")

# Also save as CSV if desired (without full vector)
df_simple = df.drop(columns=["vector768"])
df_simple.to_csv(csv_path, index=False)
print(f"Saved CSV file to: {csv_path}")

pickle_path, csv_path

In [ ]:
import pandas as pd
import torch
import numpy as np

# --- Clase de neurona flotante ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768):
        self.id = ID                    # palabra (string)
        self.x, self.y = position       # posición 2D (floats)
        self.vector768 = vector768      # numpy o tensor (irreemplazable, fijo)

    def __repr__(self):
        return f"Neuron[{self.id}] @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos desde archivos ---
def load_neurons_from_files(pkl_path, csv_path):
    # Leer el .pkl con embeddings y coordenadas
    data = pd.read_pickle(pkl_path)  # debe tener columns: word, x, y, vector768
    neurons = []

    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768
        )
        neurons.append(neuron)

    return neurons

# --- Ejemplo de uso ---
pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"
csv_path = "/content/DJ/palabras_con_posicion.csv"  # opcional si ya está todo en el .pkl

neurons = load_neurons_from_files(pkl_path, csv_path)

# Verificamos
for n in neurons[:5]:
    print(n)

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- Clase de neurona flotante ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768):
        self.id = ID                    # palabra (string)
        self.x, self.y = position       # posición 2D (floats)
        self.vector768 = vector768      # numpy o tensor (irreemplazable, fijo)

    def __repr__(self):
        return f"Neuron[{self.id}] @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos desde archivos ---
def load_neurons_from_files(pkl_path): # Eliminado csv_path ya que no se usa en este contexto
    # Leer el .pkl con embeddings y coordenadas
    data = pd.read_pickle(pkl_path)  # debe tener columns: word, x, y, vector768
    neurons = []

    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768
        )
        neurons.append(neuron)

    return neurons

# --- Función para visualizar las neuronas ---
def visualize_neurons(neurons):
    if not neurons:
        print("No hay neuronas para visualizar.")
        return

    # Extraer posiciones e IDs para la visualización
    x_coords = [n.x for n in neurons]
    y_coords = [n.y for n in neurons]
    ids = [n.id for n in neurons]

    plt.figure(figsize=(12, 10)) # Ajusta el tamaño para una mejor visualización

    # Dibujar los puntos de las neuronas
    plt.scatter(x_coords, y_coords, s=100, alpha=0.7, edgecolors='w', linewidth=0.5)

    # Añadir etiquetas (ID de la palabra) a cada punto
    for i, ID in enumerate(ids):
        plt.annotate(
            ID,                      # Texto de la etiqueta
            (x_coords[i], y_coords[i]), # Coordenadas del punto
            textcoords="offset points", # Cómo se posiciona el texto
            xytext=(5,5),             # Desplazamiento desde el punto
            ha='left',                # Alineación horizontal
            va='bottom',              # Alineación vertical
            fontsize=9,               # Tamaño de la fuente
            color='dimgray'           # Color del texto
        )

    plt.title('Posición 2D de Neuronas Flotantes (Palabras)', fontsize=16)
    plt.xlabel('Coordenada X', fontsize=12)
    plt.ylabel('Coordenada Y', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6) # Cuadrícula para mejor referencia
    plt.axhline(0, color='grey', linewidth=0.5) # Eje X en 0
    plt.axvline(0, color='grey', linewidth=0.5) # Eje Y en 0
    plt.show()

# --- Ejemplo de uso ---
pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"
# csv_path ya no es necesario aquí si toda la info está en el pkl

neurons = load_neurons_from_files(pkl_path)

# Verificamos las primeras neuronas cargadas
print("Neuronas cargadas (primeras 5):")
for n in neurons[:5]:
    print(n)

print("\nVisualizando neuronas...")
visualize_neurons(neurons)

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE # Necesario si generas nuevas coordenadas

# --- Clase de neurona flotante (modificada para tipos) ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768, neuron_type='input'):
        self.id = ID
        self.x, self.y = position
        self.vector768 = vector768  # El "núcleo" o embedding de la neurona
        self.type = neuron_type     # 'input' (palabra) o 'processing'

    def __repr__(self):
        return f"Neuron[{self.id}] (Type: {self.type}) @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos de neuronas de entrada desde archivos ---
def load_input_neurons_from_files(pkl_path):
    data = pd.read_pickle(pkl_path)
    neurons = []
    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768,
            neuron_type='input'
        )
        neurons.append(neuron)
    return neurons

# --- Función para generar neuronas de procesamiento ---
def generate_processing_neurons(num_neurons, avg_vector_dim=768, space_range_x=(-50, 50), space_range_y=(-50, 50)):
    processing_neurons = []
    for i in range(num_neurons):
        # Generar un vector 768 aleatorio como "núcleo" de la neurona de procesamiento
        # Estos vectores podrían ser centros de clusters, embeddings aprendidos, etc.
        # Aquí, aleatorios para el ejemplo.
        vector768 = np.random.rand(avg_vector_dim) * 2 - 1 # Valores entre -1 y 1

        # Generar una posición 2D aleatoria para la visualización
        x_pos = np.random.uniform(space_range_x[0], space_range_x[1])
        y_pos = np.random.uniform(space_range_y[0], space_range_y[1])

        neuron = FloatingNeuron(
            ID=f"Proc_N{i+1}",
            position=(x_pos, y_pos),
            vector768=vector768,
            neuron_type='processing'
        )
        processing_neurons.append(neuron)
    return processing_neurons

# --- Simulación de Activación y Salida ---
def simulate_neuron_activation(input_word_embedding, processing_neurons, activation_threshold=0.5, scale_factor=5):
    """
    Simula la activación de las neuronas de procesamiento dado un embedding de palabra de entrada.
    """
    results = []
    input_word_embedding_tensor = torch.tensor(input_word_embedding, dtype=torch.float32)

    for proc_neuron in processing_neurons:
        # Calcular similitud (coseno o distancia euclidiana inversa)
        # Aquí usaremos similitud coseno, es más común para embeddings.
        proc_neuron_vector_tensor = torch.tensor(proc_neuron.vector768, dtype=torch.float32)

        # Función de activación lineal (producto punto)
        # Una activación más sofisticada podría usar un kernel RBF o sigmoide
        raw_activation = torch.dot(input_word_embedding_tensor, proc_neuron_vector_tensor)

        # Simulación de umbral: si la activación es baja, se considera inactiva o 0
        if raw_activation.item() < activation_threshold:
            activation_output = 0.0
        else:
            activation_output = raw_activation.item() * scale_factor # Suma ponderada

        results.append({
            'processing_neuron_id': proc_neuron.id,
            'raw_activation': raw_activation.item(),
            'output': activation_output
        })
    return results

# --- Función para visualizar TODAS las neuronas ---
def visualize_all_neurons(all_neurons):
    if not all_neurons:
        print("No hay neuronas para visualizar.")
        return

    # Preparar datos para visualización
    x_coords = [n.x for n in all_neurons]
    y_coords = [n.y for n in all_neurons]
    ids = [n.id for n in all_neurons]
    types = [n.type for n in all_neurons]

    # Mapeo de colores por tipo de neurona
    color_map = {'input': 'blue', 'processing': 'red'}
    colors = [color_map[t] for t in types]

    plt.figure(figsize=(14, 12))

    # Dibujar puntos
    plt.scatter(x_coords, y_coords, c=colors, s=150, alpha=0.7, edgecolors='w', linewidth=0.5)

    # Añadir etiquetas
    for i, ID in enumerate(ids):
        plt.annotate(
            ID,
            (x_coords[i], y_coords[i]),
            textcoords="offset points",
            xytext=(5,5),
            ha='left', va='bottom',
            fontsize=9, color='dimgray' if types[i] == 'input' else 'black'
        )

    plt.title('Posición 2D de Neuronas Flotantes (Input y Procesamiento)', fontsize=16)
    plt.xlabel('Coordenada X', fontsize=12)
    plt.ylabel('Coordenada Y', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linewidth=0.5)
    plt.axvline(0, color='grey', linewidth=0.5)

    # Leyenda
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Input (Palabra)', markerfacecolor='blue', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Procesamiento', markerfacecolor='red', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")

    plt.show()

# --- Parte principal del script ---
if __name__ == "__main__":
    pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"

    # 1. Cargar neuronas de entrada (palabras)
    input_neurons = load_input_neurons_from_files(pkl_path)
    print("Neuronas de entrada cargadas (primeras 5):")
    for n in input_neurons[:5]:
        print(n)

    # 2. Generar neuronas de procesamiento
    # Puedes ajustar el número de neuronas de procesamiento y el rango de sus posiciones
    num_processing_neurons = 5
    processing_neurons = generate_processing_neurons(num_processing_neurons,
                                                     space_range_x=(-70, 70),
                                                     space_range_y=(-70, 70))
    print("\nNeuronas de procesamiento generadas:")
    for n in processing_neurons:
        print(n)

    # 3. Combinar todas las neuronas para visualización
    all_neurons = input_neurons + processing_neurons
    print("\nVisualizando todas las neuronas...")
    visualize_all_neurons(all_neurons)

    # 4. Simular un Input y Activación
    print("\n--- Simulación de Activación ---")

    # Elegir una palabra de tu input_neurons para simular la entrada
    # Aquí seleccionamos la primera palabra como ejemplo
    if input_neurons:
        input_word_neuron = input_neurons[0]
        print(f"\nInput: '{input_word_neuron.id}' (Embedding de la palabra de entrada)")

        # Realizar la simulación
        activation_results = simulate_neuron_activation(
            input_word_neuron.vector768,
            processing_neurons,
            activation_threshold=0.5, # Umbral de activación
            scale_factor=5            # Factor de escala para la salida
        )

        print("\nResultados de Activación de Neuronas de Procesamiento:")
        for result in activation_results:
            print(f"- {result['processing_neuron_id']}: Activación Cruda={result['raw_activation']:.4f}, Salida={result['output']:.4f}")
            # Puedes agregar lógica aquí para decidir si una neurona se "activa" basado en su salida
            if result['output'] > 0:
                print(f"  -> {result['processing_neuron_id']} se ha activado!")
    else:
        print("No hay neuronas de entrada para simular.")

    # 5. Ejemplo de tu modelo: "The house"
    # Para esto, necesitarías los embeddings de "The" y "house".
    # Suponiendo que tienes "The" y "house" en tus input_neurons:
    print("\n--- Simulación de 'The house' ---")
    the_neuron = next((n for n in input_neurons if n.id == 'The'), None)
    house_neuron = next((n for n in input_neurons if n.id == 'house'), None)

    if the_neuron and house_neuron:
        print(f"\nActivación para 'The':")
        simulate_neuron_activation(the_neuron.vector768, processing_neurons) # Solo imprimo la activación

        print(f"\nActivación para 'house':")
        simulate_neuron_activation(house_neuron.vector768, processing_neurons)

        # Aquí puedes agregar tu lógica para cómo "Modelo: The house" se forma.
        # Por ejemplo, la neurona de procesamiento con mayor activación podría "decidir" la siguiente palabra.
        print("\nEl modelo debería producir: 'The house'")
        # Esto es solo un placeholder, la lógica de secuencia es más compleja.
    else:
        print("Las palabras 'The' o 'house' no se encontraron en las neuronas de entrada cargadas.")

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F # Para cosine_similarity y sigmoide

# --- Hiperparámetros ---
NUM_PROCESSING_NEURONS = 100 # Cantidad de neuronas de procesamiento
PROCESSING_NEURON_SPACE_RANGE_X = (-70, 70) # Rango X para generar neuronas de procesamiento
PROCESSING_NEURON_SPACE_RANGE_Y = (-70, 70) # Rango Y para generar neuronas de procesamiento
ACTIVATION_THRESHOLD = 0.5 # Umbral para la activación lineal (ajustar si usas cos_sim)
OUTPUT_SCALE_FACTOR = 5 # Factor de escala para la salida (considerar eliminar si usas sigmoide/tanh)
USE_COSINE_SIMILARITY = True # Usar similitud coseno en lugar de producto punto
USE_SIGMOID_ACTIVATION = False # Usar función de activación sigmoide en lugar de lineal/umbral

# --- Clase de neurona flotante (sin cambios aquí) ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768, neuron_type='input'):
        self.id = ID
        self.x, self.y = position
        self.vector768 = vector768
        self.type = neuron_type

    def __repr__(self):
        return f"Neuron[{self.id}] (Type: {self.type}) @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos de neuronas de entrada desde archivos (sin cambios) ---
def load_input_neurons_from_files(pkl_path):
    data = pd.read_pickle(pkl_path)
    neurons = []
    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768,
            neuron_type='input'
        )
        neurons.append(neuron)
    return neurons

# --- Función para generar neuronas de procesamiento (sin cambios, usa hiperparámetros) ---
def generate_processing_neurons(num_neurons, avg_vector_dim=768,
                                space_range_x=PROCESSING_NEURON_SPACE_RANGE_X,
                                space_range_y=PROCESSING_NEURON_SPACE_RANGE_Y):
    processing_neurons = []
    for i in range(num_neurons):
        vector768 = np.random.rand(avg_vector_dim) * 2 - 1 # Valores entre -1 y 1
        x_pos = np.random.uniform(space_range_x[0], space_range_x[1])
        y_pos = np.random.uniform(space_range_y[0], space_range_y[1])

        neuron = FloatingNeuron(
            ID=f"Proc_N{i+1}",
            position=(x_pos, y_pos),
            vector768=vector768,
            neuron_type='processing'
        )
        processing_neurons.append(neuron)
    return processing_neurons

# --- Solución Posible para mapear output a palabra (tu código) ---
def map_output_to_word(activation_vector, input_neurons):
    max_sim = -float('inf') # Inicializar con valor muy bajo
    best_word = None

    # Asegúrate de que el activation_vector sea un tensor para la comparación
    if isinstance(activation_vector, np.ndarray):
        activation_vector = torch.tensor(activation_vector, dtype=torch.float32)
    elif not isinstance(activation_vector, torch.Tensor):
        raise TypeError("activation_vector debe ser numpy array o torch tensor")

    # Si el activation_vector es solo un escalar (output de una neurona), no tiene sentido compararlo así.
    # Esta función asume que activation_vector es un embedding de 768D (o similar)
    # que es el "resultado" de la red y quieres saber a qué palabra se parece más.
    # Si la salida es solo un escalar, necesitas un "decodificador" diferente.
    if activation_vector.dim() == 0: # Si es un escalar
        print("Advertencia: activation_vector es un escalar. map_output_to_word necesita un vector de embeddings.")
        return None

    for neuron in input_neurons:
        if neuron.vector768 is None:
            continue # Saltar si el vector de la neurona de entrada es None

        neuron_vec_tensor = torch.tensor(neuron.vector768, dtype=torch.float32)

        # Usar similitud coseno para la comparación
        sim = F.cosine_similarity(
            activation_vector.unsqueeze(0),
            neuron_vec_tensor.unsqueeze(0)
        )
        sim_val = sim.item()

        if sim_val > max_sim:
            max_sim = sim_val
            best_word = neuron.id

    return best_word, max_sim

# --- Función para visualizar TODAS las neuronas (sin cambios) ---
def visualize_all_neurons(all_neurons):
    if not all_neurons:
        print("No hay neuronas para visualizar.")
        return

    x_coords = [n.x for n in all_neurons]
    y_coords = [n.y for n in all_neurons]
    ids = [n.id for n in all_neurons]
    types = [n.type for n in all_neurons]

    color_map = {'input': 'blue', 'processing': 'red'}
    colors = [color_map[t] for t in types]

    plt.figure(figsize=(14, 12))
    plt.scatter(x_coords, y_coords, c=colors, s=150, alpha=0.7, edgecolors='w', linewidth=0.5)

    for i, ID in enumerate(ids):
        plt.annotate(
            ID,
            (x_coords[i], y_coords[i]),
            textcoords="offset points",
            xytext=(5,5),
            ha='left', va='bottom',
            fontsize=9, color='dimgray' if types[i] == 'input' else 'black'
        )

    plt.title('Posición 2D de Neuronas Flotantes (Input y Procesamiento)', fontsize=16)
    plt.xlabel('Coordenada X', fontsize=12)
    plt.ylabel('Coordenada Y', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linewidth=0.5)
    plt.axvline(0, color='grey', linewidth=0.5)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Input (Palabra)', markerfacecolor='blue', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Procesamiento', markerfacecolor='red', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")
    plt.show()

# --- Simulación de Activación y Salida (Modificada) ---
def simulate_neuron_activation(input_word_embedding, processing_neurons):
    """
    Simula la activación de las neuronas de procesamiento dado un embedding de palabra de entrada.
    Aplica las soluciones para Problemas 3 y 4.
    """
    results = []
    input_word_embedding_tensor = torch.tensor(input_word_embedding, dtype=torch.float32).unsqueeze(0) # Añadir dimensión batch

    for proc_neuron in processing_neurons:
        proc_neuron_vector_tensor = torch.tensor(proc_neuron.vector768, dtype=torch.float32).unsqueeze(0) # Añadir dimensión batch

        # --- Solución para Problema 3: Usar Similitud Coseno ---
        if USE_COSINE_SIMILARITY:
            raw_activation = F.cosine_similarity(input_word_embedding_tensor, proc_neuron_vector_tensor).item()
        else:
            # Producto punto tradicional (menos robusto)
            raw_activation = torch.dot(input_word_embedding_tensor.squeeze(0), proc_neuron_vector_tensor.squeeze(0)).item()

        # --- Simulación de Sesgo y Función de Activación ---
        # Puedes añadir un sesgo aquí si lo deseas, e.g., raw_activation + bias

        # --- Solución para Problema 4: Controlar la escala de activación ---
        # Si USE_SIGMOID_ACTIVATION es True, la salida estará entre 0 y 1.
        # Si es False, se aplica el umbral y el factor de escala opcional.
        if USE_SIGMOID_ACTIVATION:
            activation_output = torch.sigmoid(torch.tensor(raw_activation)).item()
        else:
            if raw_activation < ACTIVATION_THRESHOLD:
                activation_output = 0.0
            else:
                activation_output = raw_activation * OUTPUT_SCALE_FACTOR # Solo si no usas sigmoide

        results.append({
            'processing_neuron_id': proc_neuron.id,
            'raw_activation_score': raw_activation, # Puntuación de similitud/producto punto
            'output_activation': activation_output # Salida final después de función de activación
        })
    return results

# --- Parte principal del script ---
if __name__ == "__main__":
    pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"

    # 1. Cargar neuronas de entrada (palabras)
    input_neurons = load_input_neurons_from_files(pkl_path)
    print("Neuronas de entrada cargadas (primeras 5):")
    for n in input_neurons[:5]:
        print(n)

    # 2. Generar neuronas de procesamiento usando hiperparámetros
    processing_neurons = generate_processing_neurons(
        num_neurons=NUM_PROCESSING_NEURONS,
        space_range_x=PROCESSING_NEURON_SPACE_RANGE_X,
        space_range_y=PROCESSING_NEURON_SPACE_RANGE_Y
    )
    print(f"\nNeuronas de procesamiento generadas ({NUM_PROCESSING_NEURONS} en total):")
    for n in processing_neurons:
        print(n)

    # 3. Combinar todas las neuronas para visualización
    all_neurons = input_neurons + processing_neurons
    print("\nVisualizando todas las neuronas...")
    visualize_all_neurons(all_neurons)

    # 4. Simular un Input y Activación (con mejoras)
    print("\n--- Simulación de Activación de Neuronas de Procesamiento ---")

    # Elegir una palabra de tu input_neurons para simular la entrada
    if input_neurons:
        # Intenta encontrar una palabra de ejemplo si existe
        word_to_simulate = next((n for n in input_neurons if n.id == 'casa'), None)
        if word_to_simulate is None and len(input_neurons) > 0:
            word_to_simulate = input_neurons[0] # Usa la primera si 'casa' no existe

        if word_to_simulate:
            print(f"\nInput a la red de procesamiento: '{word_to_simulate.id}' (Embedding de la palabra de entrada)")

            activation_results = simulate_neuron_activation(
                word_to_simulate.vector768,
                processing_neurons
            )

            print("\nResultados de Activación de Neuronas de Procesamiento:")
            # Solo muestra las que tienen una salida > 0 para ver activaciones
            active_results = [r for r in activation_results if r['output_activation'] > 0]
            if active_results:
                for result in active_results:
                    print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
            else:
                print("Ninguna neurona de procesamiento se activó por encima del umbral.")

            # --- Conectar a la siguiente capa: Decodificación ---
            # Si una neurona de procesamiento genera un "embedding" de salida,
            # podríamos intentar mapearlo a la palabra más cercana.
            # Aquí, vamos a sumar/promediar las salidas de las neuronas activadas
            # para generar un "vector de respuesta" y mapearlo.

            # Puedes decidir cómo se "combina" la salida de varias neuronas de procesamiento
            # para formar el input a la siguiente etapa. Aquí un simple promedio ponderado.

            combined_output_vector = np.zeros(768)
            total_output_weight = 0

            for i, result in enumerate(activation_results):
                if result['output_activation'] > 0: # Si la neurona se activó
                    # Usamos el vector de la neurona de procesamiento como parte de la "respuesta"
                    # ponderado por su nivel de activación
                    combined_output_vector += processing_neurons[i].vector768 * result['output_activation']
                    total_output_weight += result['output_activation']

            if total_output_weight > 0:
                combined_output_vector /= total_output_weight # Normalizar
                print(f"\nVector de salida combinado de las neuronas de procesamiento.")

                # --- Solución para Problema 6: Mapear la salida a una palabra ---
                predicted_word, similarity = map_output_to_word(combined_output_vector, input_neurons)
                print(f"Predicción de la siguiente palabra (basada en neuronas de procesamiento): '{predicted_word}' (Similitud: {similarity:.4f})")
            else:
                print("\nNo se pudo generar un vector de salida combinado (ninguna neurona de procesamiento se activó).")
        else:
            print("No se encontró la palabra para simular o no hay neuronas de entrada.")

    # --- Simulación de 'The house' (Mejorada para composición) ---
    print("\n--- Simulación de Composición: 'The house' ---")
    the_neuron = next((n for n in input_neurons if n.id.lower() == 'the'), None)
    house_neuron = next((n for n in input_neurons if n.id.lower() == 'house'), None)

    if the_neuron and house_neuron:
        # --- Solución para Problema 6: Composición de Embeddings ---
        # Simple promedio de embeddings. Puedes experimentar con otras combinaciones (suma, concatenación, etc.)
        # Asegúrate de que los embeddings sean np.array para la suma/promedio.
        combo_embedding = (the_neuron.vector768 + house_neuron.vector768) / 2.0

        print(f"Embedding combinado para 'The house' (promedio).")

        # Ahora, simula cómo las neuronas de procesamiento reaccionan a este embedding combinado
        activation_results_combo = simulate_neuron_activation(combo_embedding, processing_neurons)

        print("\nResultados de Activación para el Embedding Combinado:")
        active_results_combo = [r for r in activation_results_combo if r['output_activation'] > 0]
        if active_results_combo:
            for result in active_results_combo:
                print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
        else:
            print("Ninguna neurona de procesamiento se activó para el embedding combinado.")

        # Intentar predecir la "siguiente" palabra o un concepto relacionado
        combined_output_vector_combo = np.zeros(768)
        total_output_weight_combo = 0

        for i, result in enumerate(activation_results_combo):
            if result['output_activation'] > 0:
                combined_output_vector_combo += processing_neurons[i].vector768 * result['output_activation']
                total_output_weight_combo += result['output_activation']

        if total_output_weight_combo > 0:
            combined_output_vector_combo /= total_output_weight_combo
            predicted_word_combo, similarity_combo = map_output_to_word(combined_output_vector_combo, input_neurons)
            print(f"\nPredicción de la siguiente palabra/concepto para 'The house': '{predicted_word_combo}' (Similitud: {similarity_combo:.4f})")
        else:
            print("\nNo se pudo generar un vector de salida combinado para 'The house'.")

    else:
        print("Las palabras 'The' o 'house' no se encontraron en las neuronas de entrada cargadas.")

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

# --- Hiperparámetros (algunos se vuelven redundantes o cambian su significado) ---
# Ya no necesitamos definir NUM_PROCESSING_NEURONS directamente, se calcula
# Ya no necesitamos rangos aleatorios para las posiciones de procesamiento
ACTIVATION_THRESHOLD = 0.5
OUTPUT_SCALE_FACTOR = 5
USE_COSINE_SIMILARITY = True
USE_SIGMOID_ACTIVATION = False

# --- Clase de neurona flotante (sin cambios) ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768, neuron_type='input'):
        self.id = ID
        self.x, self.y = position
        self.vector768 = vector768
        self.type = neuron_type

    def __repr__(self):
        return f"Neuron[{self.id}] (Type: {self.type}) @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos de neuronas de entrada desde archivos (sin cambios) ---
def load_input_neurons_from_files(pkl_path):
    data = pd.read_pickle(pkl_path)
    neurons = []
    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768,
            neuron_type='input'
        )
        neurons.append(neuron)
    return neurons

# --- MODIFICADO: Función para generar neuronas de procesamiento con la nueva lógica ---
def generate_processing_neurons_v2(input_neurons, avg_vector_dim=768):
    processing_neurons = []

    for input_neuron in input_neurons:
        # Vector base para la neurona de procesamiento.
        # Aquí, aleatorio. En un modelo entrenado, sería algo aprendido.
        # Para el ejemplo, podríamos incluso usar una variación del vector de la neurona de entrada
        # para que tengan alguna relación semántica desde el inicio.
        base_proc_vector = np.random.rand(avg_vector_dim) * 2 - 1

        # --- Primera neurona de procesamiento conectada a esta neurona de entrada ---
        # ID: basado en la neurona de entrada y su multiplicador de posición
        proc_id_1 = f"Proc_{input_neuron.id}_pos2"
        # Posición: X*2, Y*2
        proc_pos_1 = (input_neuron.x * 2, input_neuron.y * 2)

        neuron_1 = FloatingNeuron(
            ID=proc_id_1,
            position=proc_pos_1,
            vector768=base_proc_vector, # Este vector será el "peso" de esta neurona de procesamiento
            neuron_type='processing'
        )
        processing_neurons.append(neuron_1)

        # --- Segunda neurona de procesamiento conectada a esta neurona de entrada ---
        proc_id_2 = f"Proc_{input_neuron.id}_neg2"
        # Posición: X*-2, Y*-2
        proc_pos_2 = (input_neuron.x * -2, input_neuron.y * -2)

        neuron_2 = FloatingNeuron(
            ID=proc_id_2,
            position=proc_pos_2,
            vector768=np.random.rand(avg_vector_dim) * 2 - 1, # Otro vector aleatorio
            neuron_type='processing'
        )
        processing_neurons.append(neuron_2)

    return processing_neurons

# --- map_output_to_word (sin cambios) ---
def map_output_to_word(activation_vector, input_neurons):
    max_sim = -float('inf')
    best_word = None

    if isinstance(activation_vector, np.ndarray):
        activation_vector = torch.tensor(activation_vector, dtype=torch.float32)
    elif not isinstance(activation_vector, torch.Tensor):
        raise TypeError("activation_vector debe ser numpy array o torch tensor")

    if activation_vector.dim() == 0:
        print("Advertencia: activation_vector es un escalar. map_output_to_word necesita un vector de embeddings.")
        return None, None # Devuelve None para ambos

    for neuron in input_neurons:
        if neuron.vector768 is None:
            continue

        neuron_vec_tensor = torch.tensor(neuron.vector768, dtype=torch.float32)

        sim = F.cosine_similarity(
            activation_vector.unsqueeze(0),
            neuron_vec_tensor.unsqueeze(0)
        )
        sim_val = sim.item()

        if sim_val > max_sim:
            max_sim = sim_val
            best_word = neuron.id

    return best_word, max_sim

# --- visualize_all_neurons (sin cambios, ya maneja diferentes tipos) ---
def visualize_all_neurons(all_neurons):
    if not all_neurons:
        print("No hay neuronas para visualizar.")
        return

    x_coords = [n.x for n in all_neurons]
    y_coords = [n.y for n in all_neurons]
    ids = [n.id for n in all_neurons]
    types = [n.type for n in all_neurons]

    color_map = {'input': 'blue', 'processing': 'red'}
    colors = [color_map[t] for t in types]

    plt.figure(figsize=(14, 12))
    plt.scatter(x_coords, y_coords, c=colors, s=150, alpha=0.7, edgecolors='w', linewidth=0.5)

    for i, ID in enumerate(ids):
        plt.annotate(
            ID,
            (x_coords[i], y_coords[i]),
            textcoords="offset points",
            xytext=(5,5),
            ha='left', va='bottom',
            fontsize=9, color='dimgray' if types[i] == 'input' else 'black'
        )

    plt.title('Posición 2D de Neuronas Flotantes (Input y Procesamiento)', fontsize=16)
    plt.xlabel('Coordenada X', fontsize=12)
    plt.ylabel('Coordenada Y', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linewidth=0.5)
    plt.axvline(0, color='grey', linewidth=0.5)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Input (Palabra)', markerfacecolor='blue', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Procesamiento', markerfacecolor='red', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")
    plt.show()

# --- Simulación de Activación y Salida (sin cambios, ya usa hiperparámetros) ---
def simulate_neuron_activation(input_word_embedding, processing_neurons):
    results = []
    input_word_embedding_tensor = torch.tensor(input_word_embedding, dtype=torch.float32).unsqueeze(0)

    for proc_neuron in processing_neurons:
        proc_neuron_vector_tensor = torch.tensor(proc_neuron.vector768, dtype=torch.float32).unsqueeze(0)

        if USE_COSINE_SIMILARITY:
            raw_activation = F.cosine_similarity(input_word_embedding_tensor, proc_neuron_vector_tensor).item()
        else:
            raw_activation = torch.dot(input_word_embedding_tensor.squeeze(0), proc_neuron_vector_tensor.squeeze(0)).item()

        if USE_SIGMOID_ACTIVATION:
            activation_output = torch.sigmoid(torch.tensor(raw_activation)).item()
        else:
            if raw_activation < ACTIVATION_THRESHOLD:
                activation_output = 0.0
            else:
                activation_output = raw_activation * OUTPUT_SCALE_FACTOR

        results.append({
            'processing_neuron_id': proc_neuron.id,
            'raw_activation_score': raw_activation,
            'output_activation': activation_output
        })
    return results

# --- Parte principal del script ---
if __name__ == "__main__":
    pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"

    # 1. Cargar neuronas de entrada (palabras)
    input_neurons = load_input_neurons_from_files(pkl_path)
    print("Neuronas de entrada cargadas (primeras 5):")
    for n in input_neurons[:5]:
        print(n)

    # 2. Generar neuronas de procesamiento CON LA NUEVA LÓGICA
    processing_neurons = generate_processing_neurons_v2(input_neurons)

    print(f"\nNeuronas de procesamiento generadas ({len(processing_neurons)} en total, el doble de las de entrada):")
    for n in processing_neurons:
        print(n)

    # 3. Combinar todas las neuronas para visualización
    all_neurons = input_neurons + processing_neurons
    print("\nVisualizando todas las neuronas...")
    visualize_all_neurons(all_neurons)

    # 4. Simular un Input y Activación (con mejoras)
    print("\n--- Simulación de Activación de Neuronas de Procesamiento ---")

    if input_neurons:
        # Tomar una palabra de ejemplo para simular la entrada
        word_to_simulate = next((n for n in input_neurons if n.id.lower() == 'casa'), None)
        if word_to_simulate is None and len(input_neurons) > 0:
            word_to_simulate = input_neurons[0] # Usa la primera si 'casa' no existe

        if word_to_simulate:
            print(f"\nInput a la red de procesamiento: '{word_to_simulate.id}' (Embedding de la palabra de entrada)")

            activation_results = simulate_neuron_activation(
                word_to_simulate.vector768,
                processing_neurons
            )

            print("\nResultados de Activación de Neuronas de Procesamiento:")
            active_results = [r for r in activation_results if r['output_activation'] > 0]
            if active_results:
                for result in active_results:
                    print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
            else:
                print("Ninguna neurona de procesamiento se activó por encima del umbral.")

            combined_output_vector = np.zeros(768)
            total_output_weight = 0

            for i, result in enumerate(activation_results):
                if result['output_activation'] > 0:
                    combined_output_vector += processing_neurons[i].vector768 * result['output_activation']
                    total_output_weight += result['output_activation']

            if total_output_weight > 0:
                combined_output_vector /= total_output_weight
                print(f"\nVector de salida combinado de las neuronas de procesamiento.")

                predicted_word, similarity = map_output_to_word(combined_output_vector, input_neurons)
                print(f"Predicción de la siguiente palabra (basada en neuronas de procesamiento): '{predicted_word}' (Similitud: {similarity:.4f})")
            else:
                print("\nNo se pudo generar un vector de salida combinado (ninguna neurona de procesamiento se activó).")
        else:
            print("No se encontró la palabra para simular o no hay neuronas de entrada.")

    # 5. Simulación de 'The house' (Mejorada para composición)
    print("\n--- Simulación de Composición: 'The house' ---")
    the_neuron = next((n for n in input_neurons if n.id.lower() == 'the'), None)
    house_neuron = next((n for n in input_neurons if n.id.lower() == 'house'), None)

    if the_neuron and house_neuron:
        combo_embedding = (the_neuron.vector768 + house_neuron.vector768) / 2.0

        print(f"Embedding combinado para 'The house' (promedio).")

        activation_results_combo = simulate_neuron_activation(combo_embedding, processing_neurons)

        print("\nResultados de Activación para el Embedding Combinado:")
        active_results_combo = [r for r in activation_results_combo if r['output_activation'] > 0]
        if active_results_combo:
            for result in active_results_combo:
                print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
        else:
            print("Ninguna neurona de procesamiento se activó para el embedding combinado.")

        combined_output_vector_combo = np.zeros(768)
        total_output_weight_combo = 0

        for i, result in enumerate(activation_results_combo):
            if result['output_activation'] > 0:
                combined_output_vector_combo += processing_neurons[i].vector768 * result['output_activation']
                total_output_weight_combo += result['output_activation']

        if total_output_weight_combo > 0:
            combined_output_vector_combo /= total_output_weight_combo
            predicted_word_combo, similarity_combo = map_output_to_word(combined_output_vector_combo, input_neurons)
            print(f"\nPredicción de la siguiente palabra/concepto para 'The house': '{predicted_word_combo}' (Similitud: {similarity_combo:.4f})")
        else:
            print("\nNo se pudo generar un vector de salida combinado para 'The house'.")

    else:
        print("Las palabras 'The' o 'house' no se encontraron en las neuronas de entrada cargadas.")

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

# 1. Cargar el tokenizador y el modelo BERT pre-entrenado
# Usamos 'bert-base-uncased' para este ejemplo.
# BertForMaskedLM es ideal porque nos permite predecir palabras enmascaradas.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.eval() # Poner el modelo en modo de evaluación

# 2. Definir tu texto de entrada
text_input = "Hola [MASK]." # '[MASK]' es el token especial para predecir la siguiente palabra

# 3. Tokenizar el texto de entrada
# 'return_tensors="pt"' asegura que el output sean tensores de PyTorch.
# 'return_attention_mask=True' nos da explícitamente la máscara de atención.
encoded_input = tokenizer(text_input, return_tensors='pt', return_attention_mask=True)

input_ids = encoded_input['input_ids']
attention_mask = encoded_input['attention_mask']

print(f"Texto original: {text_input}")
print(f"Tokens de entrada (IDs): {input_ids}")
print(f"Máscara de atención: {attention_mask}") # Verás 1s para tokens reales y 0s para padding (si lo hubiera)

# 4. Obtener el índice del token [MASK]
# Esto es importante para saber qué salida del modelo nos interesa para la predicción.
mask_token_index = torch.where(input_ids == tokenizer.mask_token_id)[1]
print(f"Índice del token [MASK]: {mask_token_index.item()}")

# 5. Realizar la inferencia con el modelo
# 'with torch.no_grad()' desactiva el cálculo de gradientes, lo cual es bueno para inferencia.
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)
    predictions = outputs.logits # Los logits son las puntuaciones brutas de predicción

# 6. Extraer las probabilidades de la siguiente palabra (para el token [MASK])
# Softmax convierte los logits en probabilidades.
mask_logits = predictions[0, mask_token_index.item(), :]
probabilities = torch.softmax(mask_logits, dim=-1)

# 7. Mostrar las palabras más probables
# Top 5 palabras más probables
top_5_probabilities, top_5_indices = torch.topk(probabilities, 5)

print("\n--- Predicción de la siguiente palabra ('[MASK]'): ---")
for i, prob in enumerate(top_5_probabilities):
    word = tokenizer.decode([top_5_indices[i]])
    print(f"Palabra: '{word}', Probabilidad: {prob.item():.4f}")

print("\n--- Máscara de Atención en detalle ---")
# La máscara de atención indica qué tokens deben ser atendidos (1) y cuáles no (0, por ejemplo, el padding).
print(f"Tokens (decodificados): {tokenizer.convert_ids_to_tokens(input_ids[0].tolist())}")
print(f"Máscara de Atención:    {attention_mask.tolist()}")

In [ ]:
def generate_output_neurons(sequence_words, base_neurons, avg_vector_dim=768):
    output_neurons = []

    for i, word in enumerate(sequence_words):
        # Buscar neurona base con el mismo ID, o usar la primera si no existe
        base = next((n for n in base_neurons if n.id == word), base_neurons[0])

        # Crear una neurona de salida basada en esa posición
        out_id = f"Out_{word}"
        out_pos = (base.x * 3, base.y * 3)  # Más alejada para que no se superponga

        out_vector = np.random.rand(avg_vector_dim) * 2 - 1  # Vector aleatorio

        neuron = FloatingNeuron(
            ID=out_id,
            position=out_pos,
            vector768=out_vector,
            neuron_type='output'
        )
        output_neurons.append(neuron)

    return output_neurons

color_map = {'input': 'blue', 'processing': 'red', 'output': 'orange'}
def generate_output_neurons(sequence_words, base_neurons, avg_vector_dim=768):
    output_neurons = []

    for i, word in enumerate(sequence_words):
        # Buscar neurona base con el mismo ID, o usar la primera si no existe
        base = next((n for n in base_neurons if n.id == word), base_neurons[0])

        # Crear una neurona de salida basada en esa posición
        out_id = f"Out_{word}"
        out_pos = (base.x * 3, base.y * 3)  # Más alejada para que no se superponga

        out_vector = np.random.rand(avg_vector_dim) * 2 - 1  # Vector aleatorio

        neuron = FloatingNeuron(
            ID=out_id,
            position=out_pos,
            vector768=out_vector,
            neuron_type='output'
        )
        output_neurons.append(neuron)

    return output_neurons

color_map = {'input': 'blue', 'processing': 'red', 'output': 'orange'}
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Input (Palabra)', markerfacecolor='blue', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Procesamiento', markerfacecolor='red', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Salida (Secuencia)', markerfacecolor='orange', markersize=10)
]

# 2b. Generar neuronas de salida para una secuencia de palabras deseada
target_sequence = ['the', 'big']  # Puedes cambiar esta lista
output_neurons = generate_output_neurons(target_sequence, input_neurons)

print(f"\nNeuronas de salida generadas ({len(output_neurons)}):")
for n in output_neurons:
    print(n)
all_neurons = input_neurons + processing_neurons + output_neurons

# 2b. Generar neuronas de salida para una secuencia de palabras deseada
# Remove the unexpected indentation from this line and subsequent lines
target_sequence = ['the', 'big']  # Puedes cambiar esta lista
output_neurons = generate_output_neurons(target_sequence, input_neurons)

print(f"\nNeuronas de salida generadas ({len(output_neurons)}):")
for n in output_neurons:
    print(n)
all_neurons = input_neurons + processing_neurons + output_neurons

In [ ]:
Huy
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

# --- Hiperparámetros (algunos se vuelven redundantes o cambian su significado) ---
# Ya no necesitamos definir NUM_PROCESSING_NEURONS directamente, se calcula
# Ya no necesitamos rangos aleatorios para las posiciones de procesamiento
ACTIVATION_THRESHOLD = 0.5
OUTPUT_SCALE_FACTOR = 5
USE_COSINE_SIMILARITY = True
USE_SIGMOID_ACTIVATION = False

# --- Clase de neurona flotante (sin cambios) ---
class FloatingNeuron:
    def __init__(self, ID, position, vector768, neuron_type='input'):
        self.id = ID
        self.x, self.y = position
        self.vector768 = vector768
        self.type = neuron_type

    def __repr__(self):
        return f"Neuron[{self.id}] (Type: {self.type}) @ ({self.x:.2f}, {self.y:.2f})"

# --- Cargar datos de neuronas de entrada desde archivos (sin cambios) ---
def load_input_neurons_from_files(pkl_path):
    data = pd.read_pickle(pkl_path)
    neurons = []
    for row in data.itertuples():
        neuron = FloatingNeuron(
            ID=row.word,
            position=(row.x, row.y),
            vector768=row.vector768,
            neuron_type='input'
        )
        neurons.append(neuron)
    return neurons

# --- MODIFICADO: Función para generar neuronas de procesamiento con la nueva lógica ---
def generate_processing_neurons_v2(input_neurons, avg_vector_dim=768):
    processing_neurons = []

    for input_neuron in input_neurons:
        # Vector base para la neurona de procesamiento.
        # Aquí, aleatorio. En un modelo entrenado, sería algo aprendido.
        # Para el ejemplo, podríamos incluso usar una variación del vector de la neurona de entrada
        # para que tengan alguna relación semántica desde el inicio.
        base_proc_vector = np.random.rand(avg_vector_dim) * 2 - 1

        # --- Primera neurona de procesamiento conectada a esta neurona de entrada ---
        # ID: basado en la neurona de entrada y su multiplicador de posición
        proc_id_1 = f"Proc_{input_neuron.id}_pos2"
        # Posición: X*2, Y*2
        proc_pos_1 = (input_neuron.x * 2, input_neuron.y * 2)

        neuron_1 = FloatingNeuron(
            ID=proc_id_1,
            position=proc_pos_1,
            vector768=base_proc_vector, # Este vector será el "peso" de esta neurona de procesamiento
            neuron_type='processing'
        )
        processing_neurons.append(neuron_1)

        # --- Segunda neurona de procesamiento conectada a esta neurona de entrada ---
        proc_id_2 = f"Proc_{input_neuron.id}_neg2"
        # Posición: X*-2, Y*-2
        proc_pos_2 = (input_neuron.x * -2, input_neuron.y * -2)

        neuron_2 = FloatingNeuron(
            ID=proc_id_2,
            position=proc_pos_2,
            vector768=np.random.rand(avg_vector_dim) * 2 - 1, # Otro vector aleatorio
            neuron_type='processing'
        )
        processing_neurons.append(neuron_2)

    return processing_neurons

# --- map_output_to_word (sin cambios) ---
def map_output_to_word(activation_vector, input_neurons):
    max_sim = -float('inf')
    best_word = None

    if isinstance(activation_vector, np.ndarray):
        activation_vector = torch.tensor(activation_vector, dtype=torch.float32)
    elif not isinstance(activation_vector, torch.Tensor):
        raise TypeError("activation_vector debe ser numpy array o torch tensor")

    if activation_vector.dim() == 0:
        print("Advertencia: activation_vector es un escalar. map_output_to_word necesita un vector de embeddings.")
        return None, None # Devuelve None para ambos

    for neuron in input_neurons:
        if neuron.vector768 is None:
            continue

        neuron_vec_tensor = torch.tensor(neuron.vector768, dtype=torch.float32)

        sim = F.cosine_similarity(
            activation_vector.unsqueeze(0),
            neuron_vec_tensor.unsqueeze(0)
        )
        sim_val = sim.item()

        if sim_val > max_sim:
            max_sim = sim_val
            best_word = neuron.id

    return best_word, max_sim

# --- visualize_all_neurons (sin cambios, ya maneja diferentes tipos) ---
def visualize_all_neurons(all_neurons):
    if not all_neurons:
        print("No hay neuronas para visualizar.")
        return

    x_coords = [n.x for n in all_neurons]
    y_coords = [n.y for n in all_neurons]
    ids = [n.id for n in all_neurons]
    types = [n.type for n in all_neurons]

    color_map = {'input': 'blue', 'processing': 'red'}
    colors = [color_map[t] for t in types]

    plt.figure(figsize=(14, 12))
    plt.scatter(x_coords, y_coords, c=colors, s=150, alpha=0.7, edgecolors='w', linewidth=0.5)

    for i, ID in enumerate(ids):
        plt.annotate(
            ID,
            (x_coords[i], y_coords[i]),
            textcoords="offset points",
            xytext=(5,5),
            ha='left', va='bottom',
            fontsize=9, color='dimgray' if types[i] == 'input' else 'black'
        )

    plt.title('Posición 2D de Neuronas Flotantes (Input y Procesamiento)', fontsize=16)
    plt.xlabel('Coordenada X', fontsize=12)
    plt.ylabel('Coordenada Y', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linewidth=0.5)
    plt.axvline(0, color='grey', linewidth=0.5)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Input (Palabra)', markerfacecolor='blue', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Procesamiento', markerfacecolor='red', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")
    plt.show()

# --- Simulación de Activación y Salida (sin cambios, ya usa hiperparámetros) ---
def simulate_neuron_activation(input_word_embedding, processing_neurons):
    results = []
    input_word_embedding_tensor = torch.tensor(input_word_embedding, dtype=torch.float32).unsqueeze(0)

    for proc_neuron in processing_neurons:
        proc_neuron_vector_tensor = torch.tensor(proc_neuron.vector768, dtype=torch.float32).unsqueeze(0)

        if USE_COSINE_SIMILARITY:
            raw_activation = F.cosine_similarity(input_word_embedding_tensor, proc_neuron_vector_tensor).item()
        else:
            raw_activation = torch.dot(input_word_embedding_tensor.squeeze(0), proc_neuron_vector_tensor.squeeze(0)).item()

        if USE_SIGMOID_ACTIVATION:
            activation_output = torch.sigmoid(torch.tensor(raw_activation)).item()
        else:
            if raw_activation < ACTIVATION_THRESHOLD:
                activation_output = 0.0
            else:
                activation_output = raw_activation * OUTPUT_SCALE_FACTOR

        results.append({
            'processing_neuron_id': proc_neuron.id,
            'raw_activation_score': raw_activation,
            'output_activation': activation_output
        })
    return results

# --- Parte principal del script ---
if __name__ == "__main__":
    pkl_path = "/content/DJ/palabras_con_embedding_y_posicion.pkl"

    # 1. Cargar neuronas de entrada (palabras)
    input_neurons = load_input_neurons_from_files(pkl_path)
    print("Neuronas de entrada cargadas (primeras 5):")
    for n in input_neurons[:5]:
        print(n)

    # 2. Generar neuronas de procesamiento CON LA NUEVA LÓGICA
    processing_neurons = generate_processing_neurons_v2(input_neurons)

    print(f"\nNeuronas de procesamiento generadas ({len(processing_neurons)} en total, el doble de las de entrada):")
    for n in processing_neurons:
        print(n)

    # 3. Combinar todas las neuronas para visualización
    all_neurons = input_neurons + processing_neurons
    print("\nVisualizando todas las neuronas...")
    visualize_all_neurons(all_neurons)

    # 4. Simular un Input y Activación (con mejoras)
    print("\n--- Simulación de Activación de Neuronas de Procesamiento ---")

    if input_neurons:
        # Tomar una palabra de ejemplo para simular la entrada
        word_to_simulate = next((n for n in input_neurons if n.id.lower() == 'casa'), None)
        if word_to_simulate is None and len(input_neurons) > 0:
            word_to_simulate = input_neurons[0] # Usa la primera si 'casa' no existe

        if word_to_simulate:
            print(f"\nInput a la red de procesamiento: '{word_to_simulate.id}' (Embedding de la palabra de entrada)")

            activation_results = simulate_neuron_activation(
                word_to_simulate.vector768,
                processing_neurons
            )

            print("\nResultados de Activación de Neuronas de Procesamiento:")
            active_results = [r for r in activation_results if r['output_activation'] > 0]
            if active_results:
                for result in active_results:
                    print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
            else:
                print("Ninguna neurona de procesamiento se activó por encima del umbral.")

            combined_output_vector = np.zeros(768)
            total_output_weight = 0

            for i, result in enumerate(activation_results):
                if result['output_activation'] > 0:
                    combined_output_vector += processing_neurons[i].vector768 * result['output_activation']
                    total_output_weight += result['output_activation']

            if total_output_weight > 0:
                combined_output_vector /= total_output_weight
                print(f"\nVector de salida combinado de las neuronas de procesamiento.")

                predicted_word, similarity = map_output_to_word(combined_output_vector, input_neurons)
                print(f"Predicción de la siguiente palabra (basada en neuronas de procesamiento): '{predicted_word}' (Similitud: {similarity:.4f})")
            else:
                print("\nNo se pudo generar un vector de salida combinado (ninguna neurona de procesamiento se activó).")
        else:
            print("No se encontró la palabra para simular o no hay neuronas de entrada.")

    # 5. Simulación de 'The house' (Mejorada para composición)
    print("\n--- Simulación de Composición: 'The house' ---")
    the_neuron = next((n for n in input_neurons if n.id.lower() == 'the'), None)
    house_neuron = next((n for n in input_neurons if n.id.lower() == 'house'), None)

    if the_neuron and house_neuron:
        combo_embedding = (the_neuron.vector768 + house_neuron.vector768) / 2.0

        print(f"Embedding combinado para 'The house' (promedio).")

        activation_results_combo = simulate_neuron_activation(combo_embedding, processing_neurons)

        print("\nResultados de Activación para el Embedding Combinado:")
        active_results_combo = [r for r in activation_results_combo if r['output_activation'] > 0]
        if active_results_combo:
            for result in active_results_combo:
                print(f"- {result['processing_neuron_id']}: Similitud/Activación Cruda={result['raw_activation_score']:.4f}, Salida={result['output_activation']:.4f}")
        else:
            print("Ninguna neurona de procesamiento se activó para el embedding combinado.")

        combined_output_vector_combo = np.zeros(768)
        total_output_weight_combo = 0

        for i, result in enumerate(activation_results_combo):
            if result['output_activation'] > 0:
                combined_output_vector_combo += processing_neurons[i].vector768 * result['output_activation']
                total_output_weight_combo += result['output_activation']

        if total_output_weight_combo > 0:
            combined_output_vector_combo /= total_output_weight_combo
            predicted_word_combo, similarity_combo = map_output_to_word(combined_output_vector_combo, input_neurons)
            print(f"\nPredicción de la siguiente palabra/concepto para 'The house': '{predicted_word_combo}' (Similitud: {similarity_combo:.4f})")
        else:
            print("\nNo se pudo generar un vector de salida combinado para 'The house'.")

    else:
        print("Las palabras 'The' o 'house' no se encontraron en las neuronas de entrada cargadas.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx

# Diccionario semántico: Neuronas de entrada → Neuronas abstractas
semantic_map = {
    "rey": "realeza",
    "reina": "realeza",
    "emperador": "realeza",
    "soldado": "militar",
    "general": "militar",
    "profesor": "educación",
    "estudiante": "educación"
}

# Lista única de entradas y categorías
input_neurons = list(semantic_map.keys())
processing_neurons = list(set(semantic_map.values()))

# Índices
input_idx = {name: i for i, name in enumerate(input_neurons)}
processing_idx = {name: i for i, name in enumerate(processing_neurons)}

# Dimensiones
input_size = len(input_neurons)
processing_size = len(processing_neurons)

# Matriz de conexiones semánticas
connection_matrix = torch.zeros(processing_size, input_size)
for inp, proc in semantic_map.items():
    i = processing_idx[proc]
    j = input_idx[inp]
    connection_matrix[i, j] = 1.0  # Conexión semántica (peso = 1)

class SemanticNetwork(nn.Module):
    def __init__(self, connection_matrix):
        super().__init__()
        self.connection_matrix = nn.Parameter(connection_matrix.clone(), requires_grad=False)

    def forward(self, x):
        # x: [batch, input_size]
        # Multiplicación lineal semántica: activa neuronas abstractas si sus entradas conectadas se activan
        y = torch.matmul(x, self.connection_matrix.T)
        return torch.sigmoid(y)  # activación de neuronas abstractas

# Red
net = SemanticNetwork(connection_matrix)

# Entrada: activamos solo "rey" y "emperador"
input_vector = torch.zeros((1, input_size))
input_vector[0, input_idx["rey"]] = 1.0
input_vector[0, input_idx["emperador"]] = 1.0

# Forward
output = net(input_vector)

# Mostrar activaciones
print("Activaciones de entrada:")
for name, idx in input_idx.items():
    print(f"{name}: {input_vector[0, idx].item():.2f}")

print("\nActivaciones de neuronas de procesamiento (semánticas):")
for name, idx in processing_idx.items():
    print(f"{name}: {output[0, idx].item():.2f}")

# Visualización con NetworkX
def draw_network(input_vector, output_vector):
    G = nx.DiGraph()
    pos = {}
    labels = {}

    # Neuronas de entrada
    for i, name in enumerate(input_neurons):
        G.add_node(name, layer="input")
        pos[name] = (0, -i)
        labels[name] = f"{name} ({input_vector[0, i].item():.0f})"

    # Neuronas de procesamiento
    for i, name in enumerate(processing_neurons):
        G.add_node(name, layer="processing")
        pos[name] = (2, -i)
        labels[name] = f"{name} ({output_vector[0, i].item():.2f})"

    # Conexiones semánticas
    for inp, proc in semantic_map.items():
        G.add_edge(inp, proc)

    # Dibujo
    plt.figure(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, labels=labels,
            node_color='skyblue', edge_color='gray',
            node_size=3000, font_size=10, font_weight='bold')
    plt.title("Red Semántica de Neuronas")
    plt.show()

draw_network(input_vector, output)

In [ ]:
# Árbol fonético-semántico neuronal básico (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Palabras y sus clases semánticas
palabras = {
    "abuelo": "familia",
    "abismo": "profundidad",
    "ebanista": "oficio",
    "ebriedad": "estado",
    "rey": "realeza",
    "reina": "realeza",
    "gato": "animal",
    "perro": "animal",
}

# 2. Tokenización por letras
def tokenizar(palabra):
    return list(palabra)

# 3. Mapeo de letras a IDs
letras = sorted(set("".join(palabras.keys())))
letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}

# 4. Mapeo de clases
clases = sorted(set(palabras.values()))
clase2idx = {c: i for i, c in enumerate(clases)}
idx2clase = {i: c for c, i in clase2idx.items()}

# 5. Modelos
embedding_dim = 16
hidden_dim = 32
num_clases = len(clases)

embedding = nn.Embedding(len(letra2idx), embedding_dim)

class NodoJerarquico(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.cell = nn.GRUCell(embedding_dim, hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, letras):
        h = torch.zeros(1, self.hidden_dim)
        for letra in letras:
            emb = embedding(torch.tensor([letra2idx[letra]]))
            h = self.cell(emb, h)
        return h

class ClasificadorSemantico(nn.Module):
    def __init__(self, hidden_dim, num_clases):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_clases)
        )

    def forward(self, h):
        return self.mlp(h)

# Instancia de modelos
nodo = NodoJerarquico(embedding_dim, hidden_dim)
clasificador = ClasificadorSemantico(hidden_dim, num_clases)

# Optimización
params = list(embedding.parameters()) + list(nodo.parameters()) + list(clasificador.parameters())
optimizer = torch.optim.Adam(params, lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# 6. Entrenamiento
for epoch in range(100):
    total_loss = 0
    for palabra, clase in palabras.items():
        letras = tokenizar(palabra)
        h = nodo(letras)
        salida = clasificador(h)
        y = torch.tensor([clase2idx[clase]])
        loss = loss_fn(salida, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f}")

# 7. Pruebas
def clasificar_palabra(palabra):
    letras = tokenizar(palabra)
    h = nodo(letras)
    salida = clasificador(h)
    pred = torch.argmax(salida).item()
    return idx2clase[pred]

print("\n--- Pruebas ---")
for palabra in palabras.keys():
    pred = clasificar_palabra(palabra)
    print(f"{palabra:<10} → {pred}")

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# 1. Letras del abecedario
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

# 2. Embedding aprendido por letra
letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# 3. Grafo: nodos = letras
G = nx.DiGraph()
for letra in letras:
    G.add_node(letra, tipo="letra")

# 4. Visualización simple
def visualizar_letras(grafo):
    pos = nx.circular_layout(grafo)
    plt.figure(figsize=(8, 8))
    nx.draw(grafo, pos, with_labels=True, node_color='lightblue', node_size=1000, font_size=12, font_weight='bold', edge_color='gray')
    plt.title("Capa base: letras como nodos neuronales")
    plt.show()

visualizar_letras(G)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# ==== Nivel 0: Letras ====
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# ==== Grafo ====
G = nx.DiGraph()

# Agregar nodos de letras (nivel 0)
for letra in letras:
    G.add_node(letra, tipo="letra")

# ==== Nivel 1: Pares consonante-vocal o vocal-consonante ====

# Clasificación simple para esta demo
vocales = set('aeiou')
consonantes = [l for l in letras if l not in vocales]

# Construir sílabas tipo CV (consonante + vocal)
silabas_cv = [c + v for c in consonantes for v in vocales]
silabas_vc = [v + c for v in vocales for c in consonantes]
silabas = silabas_cv + silabas_vc

# Agregar nodos de sílabas (nivel 1)
for silaba in silabas:
    G.add_node(silaba, tipo="silaba")
    # Conectar con sus letras base
    G.add_edge(silaba[0], silaba)  # de letra 1 a sílaba
    G.add_edge(silaba[1], silaba)  # de letra 2 a sílaba

# ==== Visualización ====

def visualizar_arbol_fonologico(grafo):
    # Posición: letras en círculo abajo, sílabas arriba
    pos = {}
    radius_letras = 5
    radius_silabas = 10
    # Letras en círculo
    for i, l in enumerate(letras):
        angle = 2 * 3.1415 * i / len(letras)
        pos[l] = (radius_letras * torch.cos(torch.tensor(angle)).item(),
                  radius_letras * torch.sin(torch.tensor(angle)).item())
    # Sílabas: distribuidas en círculo más grande
    for i, s in enumerate(silabas):
        angle = 2 * 3.1415 * i / len(silabas)
        pos[s] = (radius_silabas * torch.cos(torch.tensor(angle)).item(),
                  radius_silabas * torch.sin(torch.tensor(angle)).item())

    plt.figure(figsize=(12, 12))
    node_colors = ['lightblue' if G.nodes[n]['tipo'] == 'letra' else 'lightgreen' for n in G.nodes]
    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            node_size=1000, font_size=10, font_weight='bold', edge_color='gray', arrowsize=10)
    plt.title("Árbol neuronal: Letras (nivel 0) → Sílabas CV/VC (nivel 1)")
    plt.show()

visualizar_arbol_fonologico(G)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# ==== Nivel 0: Letras ====
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# ==== Grafo ====
G = nx.DiGraph()

# Agregar nodos de letras (nivel 0)
for letra in letras:
    G.add_node(letra, tipo="letra")

# ==== Nivel 1: Pares consonante-vocal o vocal-consonante ====
vocales = set('aeiou')
consonantes = [l for l in letras if l not in vocales]

silabas_cv = [c + v for c in consonantes for v in vocales]
silabas_vc = [v + c for v in vocales for c in consonantes]
silabas_validas = set(silabas_cv + silabas_vc) # Usar un set para búsqueda rápida

# Agregar nodos de sílabas (nivel 1)
for silaba in silabas_validas:
    G.add_node(silaba, tipo="silaba")
    # Conectar con sus letras base
    if len(silaba) == 2: # Asegurar que tiene dos letras para la conexión
        G.add_edge(silaba[0], silaba)
        G.add_edge(silaba[1], silaba)

# ==== Nivel 2: Palabras ====
# Define un vocabulario de ejemplo
palabras_ejemplo = {
    "casa": ["ca", "sa"],
    "mesa": ["me", "sa"],
    "sol": ["sol"], # Considera sílabas de una sola letra/múltiples
    "luna": ["lu", "na"],
    "rio": ["ri", "o"], # 'o' como sílaba vocal
    "pan": ["pan"]
}

# Agrega nodos de palabras y sus conexiones
for palabra, silabas_palabra in palabras_ejemplo.items():
    G.add_node(palabra, tipo="palabra")
    for silaba_compuesta in silabas_palabra:
        # Asegúrate de que la sílaba exista en tu conjunto de sílabas válidas
        if silaba_compuesta in silabas_validas:
            G.add_edge(silaba_compuesta, palabra) # Conectar sílaba a palabra
        else:
            # Manejar sílabas no "CV" o "VC" (ej. "sol", "pan")
            # Para esta demo, las tratamos como sílabas directas si son "palabras" de una sílaba
            # En un sistema real, aquí iría una lógica de silabificación más robusta
            # Por simplicidad, si la "sílaba" no está en silabas_validas pero es parte de la palabra,
            # podríamos conectarla directamente desde sus letras o simplemente añadirla si no existe.
            # Aquí, solo conectaremos si ya está en silabas_validas para mantener el ejemplo simple.
            pass


# ==== Visualización (actualizada) ====
def visualizar_arbol_fonologico_expandido(grafo):
    pos = {}
    radius_letras = 5
    radius_silabas = 10
    radius_palabras = 15

    # Nivel 0: Letras (círculo interior)
    current_letras = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'letra']
    for i, l in enumerate(current_letras):
        angle = 2 * 3.1415 * i / len(current_letras)
        pos[l] = (radius_letras * torch.cos(torch.tensor(angle)).item(),
                  radius_letras * torch.sin(torch.tensor(angle)).item() - radius_letras) # Ajustar posición Y

    # Nivel 1: Sílabas (círculo intermedio)
    current_silabas = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'silaba']
    for i, s in enumerate(current_silabas):
        angle = 2 * 3.1415 * i / len(current_silabas)
        pos[s] = (radius_silabas * torch.cos(torch.tensor(angle)).item(),
                  radius_silabas * torch.sin(torch.tensor(angle)).item())

    # Nivel 2: Palabras (círculo exterior)
    current_palabras = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'palabra']
    for i, p in enumerate(current_palabras):
        angle = 2 * 3.1415 * i / len(current_palabras)
        pos[p] = (radius_palabras * torch.cos(torch.tensor(angle)).item(),
                  radius_palabras * torch.sin(torch.tensor(angle)).item() + radius_letras) # Ajustar posición Y

    plt.figure(figsize=(15, 15))
    node_colors = []
    for n in G.nodes:
        if G.nodes[n]['tipo'] == 'letra':
            node_colors.append('lightblue')
        elif G.nodes[n]['tipo'] == 'silaba':
            node_colors.append('lightgreen')
        elif G.nodes[n]['tipo'] == 'palabra':
            node_colors.append('lightcoral')

    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            node_size=1500, font_size=9, font_weight='bold', edge_color='gray', arrowsize=10)
    plt.title("Árbol neuronal: Letras (nivel 0) → Sílabas (nivel 1) → Palabras (nivel 2)")
    plt.show()

visualizar_arbol_fonologico_expandido(G)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# ==== Nivel 0: Letras ====
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# ==== Grafo ====
G = nx.DiGraph()

# Agregar nodos de letras (nivel 0)
for letra in letras:
    G.add_node(letra, tipo="letra")

# ==== Nivel 1: Pares consonante-vocal o vocal-consonante ====
vocales = set('aeiou')
consonantes = [l for l in letras if l not in vocales]

silabas_cv = [c + v for c in consonantes for v in vocales]
silabas_vc = [v + c for v in vocales for c in consonantes]
silabas_validas = set(silabas_cv + silabas_vc) # Usar un set para búsqueda rápida

# Agregar nodos de sílabas (nivel 1)
for silaba in silabas_validas:
    G.add_node(silaba, tipo="silaba")
    # Conectar letras a sílabas (Ej. l -> lu, u -> lu)
    if len(silaba) == 2:
        G.add_edge(silaba[0], silaba)
        G.add_edge(silaba[1], silaba)

# ==== Nivel 2: Palabras ====
# Define un vocabulario de ejemplo con su silabificación
palabras_ejemplo = {
    "casa": ["ca", "sa"],
    "mesa": ["me", "sa"],
    "sol": ["sol"], # Nota: "sol" no es una sílaba CV/VC, la añadimos para el ejemplo
    "luna": ["lu", "na"],
    "rio": ["ri", "o"],
    "pan": ["pan"] # Nota: "pan" tampoco es CV/VC
}

# Agrega nodos de palabras y sus conexiones
for palabra, silabas_de_palabra in palabras_ejemplo.items():
    G.add_node(palabra, tipo="palabra")
    for silaba_compuesta in silabas_de_palabra:
        # Si la sílaba no existe en silabas_validas (ej. "sol", "pan"), la creamos como nodo de sílaba
        if silaba_compuesta not in silabas_validas:
            G.add_node(silaba_compuesta, tipo="silaba_especial") # Distinguirla si quieres
            # Conectar las letras a estas "sílabas especiales"
            for letra_s in silaba_compuesta:
                if letra_s in G: # Asegurarse de que la letra exista como nodo
                    G.add_edge(letra_s, silaba_compuesta)

        # Conectar la sílaba a la palabra (Ej. lu -> luna, na -> luna)
        G.add_edge(silaba_compuesta, palabra)

# ==== Visualización (actualizada) ====
def visualizar_arbol_fonologico_expandido(grafo):
    pos = {}

    # Define posiciones en capas verticales
    y_letras = 0
    y_silabas = 10
    y_palabras = 20

    # Agrupar nodos por tipo
    nodos_letras = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'letra']
    nodos_silabas = [n for n, data in grafo.nodes(data=True) if data['tipo'].startswith('silaba')] # Incluye 'silaba' y 'silaba_especial'
    nodos_palabras = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'palabra']

    # Distribución horizontal para cada capa
    x_offset_letras = -len(nodos_letras) * 0.5
    for i, l in enumerate(nodos_letras):
        pos[l] = (x_offset_letras + i * 1.5, y_letras)

    x_offset_silabas = -len(nodos_silabas) * 0.5
    for i, s in enumerate(nodos_silabas):
        pos[s] = (x_offset_silabas + i * 2, y_silabas)

    x_offset_palabras = -len(nodos_palabras) * 0.5
    for i, p in enumerate(nodos_palabras):
        pos[p] = (x_offset_palabras + i * 3, y_palabras)

    plt.figure(figsize=(18, 12)) # Aumentar tamaño para mejor visualización

    node_colors = []
    for n in G.nodes:
        if G.nodes[n]['tipo'] == 'letra':
            node_colors.append('lightblue')
        elif G.nodes[n]['tipo'].startswith('silaba'): # Colorea todas las sílabas igual
            node_colors.append('lightgreen')
        elif G.nodes[n]['tipo'] == 'palabra':
            node_colors.append('lightcoral')

    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            node_size=1500, font_size=10, font_weight='bold', edge_color='gray', arrowsize=10)
    plt.title("Árbol neuronal: Letras (nivel 0) → Sílabas (nivel 1) → Palabras (nivel 2) con conexiones bidireccionales")
    plt.show()

visualizar_arbol_fonologico_expandido(G)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# ==== Nivel 0: Letras ====
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# ==== Grafo ====
G = nx.DiGraph()

# Agregar nodos de letras (nivel 0)
for letra in letras:
    G.add_node(letra, tipo="letra")

# ==== Nivel 1: Sílabas ====
vocales = set('aeiouáéíóúü') # Incluir vocales con tilde y diéresis para mayor realismo
consonantes = [l for l in letras if l not in vocales]

silabas_validas = set() # Usaremos un set para almacenar sílabas únicas que construiremos
# Puedes añadir más reglas de silabificación si quieres. Aquí un ejemplo simple:
# Sílabas tipo CV (consonante + vocal)
for c in consonantes:
    for v in vocales:
        silaba = c + v
        silabas_validas.add(silaba)
# Sílabas tipo VC (vocal + consonante)
for v in vocales:
    for c in consonantes:
        silaba = v + c
        silabas_validas.add(silaba)
# Sílabas de una sola vocal
for v in vocales:
    silabas_validas.add(v)

# Agregar nodos de sílabas (nivel 1)
for silaba in silabas_validas:
    G.add_node(silaba, tipo="silaba")
    # Conectar sílaba con sus letras base
    for char in silaba:
        if char in letras: # Asegurarse de que el carácter es una letra del abecedario
            G.add_edge(char, silaba)

# ==== Nivel 2: Palabras ====
# Para un modelo más robusto, se usaría una biblioteca de silabificación real para español.
# Aquí, para el ejemplo, descomponemos manualmente.
palabras_descompuestas = {
    "luna": ["lu", "na"],
    "sol": ["sol"], # 'sol' se considera una sílaba única
    "casa": ["ca", "sa"],
    "mesa": ["me", "sa"],
    "rio": ["ri", "o"], # "o" como sílaba vocal
    "perro": ["pe", "rro"], # Ejemplo de doble consonante
    "arbol": ["ar", "bol"],
    "cielo": ["cie", "lo"]
}

# Agrega nodos de palabras y sus conexiones
for palabra, silabas_palabra in palabras_descompuestas.items():
    G.add_node(palabra, tipo="palabra")
    for silaba_componente in silabas_palabra:
        # Asegúrate de que la sílaba componente exista en tus silabas_validas
        # Si no existe (ej. "sol" o "rro" que no son CV/VC simples), puedes decidir cómo manejarlas:
        # 1. Añadirlas a silabas_validas si son sílabas genuinas pero no "CV/VC"
        # 2. Conectar las letras directamente a la palabra si la sílaba no es "estándar" (menos recomendable para esta jerarquía)
        if silaba_componente not in silabas_validas:
             silabas_validas.add(silaba_componente) # Añadir sílaba "no estándar" para que tenga nodo
             G.add_node(silaba_componente, tipo="silaba")
             for char in silaba_componente:
                 if char in letras:
                     G.add_edge(char, silaba_componente)

        G.add_edge(silaba_componente, palabra) # Conectar sílaba a palabra

# ==== Visualización ====

def visualizar_arbol_jerarquico(grafo):
    pos = {}
    # Podemos intentar organizar en capas, o con diseño spring para ver las conexiones
    # Para visualizar capas jerárquicas:
    levels = {
        "letra": 0,
        "silaba": 1,
        "palabra": 2
    }

    # Crear un diseño spring_layout para ver las relaciones, o un "layered" si hay un paquete
    # Para este ejemplo, un diseño spring puede mostrar mejor la interconexión.
    # Alternativamente, si quieres algo más estructurado como capas:

    # Intento de layout en capas (esto puede ser complicado para redes muy grandes)
    # nodes_by_level = {level: [] for level in levels.values()}
    # for n, data in grafo.nodes(data=True):
    #     nodes_by_level[levels[data['tipo']]].append(n)

    # # Posiciones para cada nivel (ejemplo simple, necesitaría más ajuste para estética)
    # for level, nodes in nodes_by_level.items():
    #     if nodes:
    #         y_pos = level * 10 # Separación vertical entre niveles
    #         x_step = 20 / len(nodes)
    #         for i, node in enumerate(nodes):
    #             pos[node] = (i * x_step - (len(nodes) * x_step / 2), y_pos)

    # Optamos por un diseño spring_layout para mejor visualización de todas las conexiones
    pos = nx.spring_layout(grafo, k=0.8, iterations=50) # k ajusta la distancia, iterations la calidad

    plt.figure(figsize=(16, 16)) # Aumentar tamaño para mejor visibilidad

    node_colors = []
    for n in G.nodes:
        if G.nodes[n]['tipo'] == 'letra':
            node_colors.append('lightblue')
        elif G.nodes[n]['tipo'] == 'silaba':
            node_colors.append('lightgreen')
        elif G.nodes[n]['tipo'] == 'palabra':
            node_colors.append('lightcoral')

    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            node_size=1800, font_size=9, font_weight='bold', edge_color='gray', arrows=True, arrowsize=15)
    plt.title("Árbol neuronal jerárquico: Letras → Sílabas → Palabras", size=18)
    plt.show()

visualizar_arbol_jerarquico(G)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

# ==== Nivel 0: Letras ====
letras = list("abcdefghijklmnñopqrstuvwxyz")
embedding_dim = 8

letra2idx = {l: i for i, l in enumerate(letras)}
idx2letra = {i: l for l, i in letra2idx.items()}
embedding = nn.Embedding(len(letras), embedding_dim)

# ==== Grafo ====
G = nx.DiGraph()

# Agregar nodos de letras (nivel 0)
for letra in letras:
    G.add_node(letra, tipo="letra")

# ==== Nivel 1: Sílabas (simplificado para esta demo) ====
vocales = set('aeiou')
consonantes = [l for l in letras if l not in vocales]

# Generamos sílabas "válidas" básicas (CV, VC) como base
silabas_base_cv = [c + v for c in consonantes for v in vocales]
silabas_base_vc = [v + c for v in vocales for c in consonantes]
silabas_validas_generadas = set(silabas_base_cv + silabas_base_vc)

# --- Nivel 2: Palabras y sus conexiones ---
# Definir palabras y sus sílabas. Aquí es crucial que las sílabas estén bien definidas.
# Para esta demo, haremos una silabificación manual simple.
# En un proyecto real, usarías una librería de silabificación para mayor precisión.
palabras_a_procesar = {
    "casa": ["ca", "sa"],
    "mesa": ["me", "sa"],
    "sol": ["sol"], # Considera casos de sílabas no CV/VC estándar
    "luna": ["lu", "na"],
    "rio": ["ri", "o"], # 'o' como sílaba vocal
    "pan": ["pan"],
    "amor": ["a", "mor"] # Ejemplo de vocal como sílaba inicial
}

# Diccionario para almacenar todas las sílabas únicas encontradas en las palabras
todas_las_silabas_encontradas = set()

for palabra, silabas_de_palabra in palabras_a_procesar.items():
    G.add_node(palabra, tipo="palabra") # Agregamos el nodo de la palabra

    for silaba in silabas_de_palabra:
        todas_las_silabas_encontradas.add(silaba) # Añadimos la sílaba al conjunto
        G.add_node(silaba, tipo="silaba") # Aseguramos que el nodo de la sílaba exista

        # Conectar palabra a sílaba (ej. "luna" -> "lu", "luna" -> "na")
        G.add_edge(silaba, palabra) # La flecha va de la parte más pequeña a la más grande

        # Conectar sílaba a sus letras (ej. "lu" -> "l", "lu" -> "u")
        for letra in silaba:
            if letra in G.nodes and G.nodes[letra]['tipo'] == 'letra': # Asegurar que la letra existe como nodo
                G.add_edge(letra, silaba) # La flecha va de la letra a la sílaba
            # Si la letra no es una letra base, no la conectamos.
            # Esto maneja el caso de sílabas como "sol" donde 's', 'o', 'l' ya son letras base.
            # Y 'sol' no tiene un nodo de sílaba 's', 'o', 'l' de forma individual para conectarlos.

# Re-agregar nodos de sílabas para asegurar que se crean todos los necesarios
# y no solo las que se derivan de silabas_validas_generadas inicialmente.
for silaba in todas_las_silabas_encontradas:
     G.add_node(silaba, tipo="silaba")


# ==== Visualización (actualizada para 3 niveles) ====
def visualizar_arbol_fonologico_detallado(grafo):
    pos = {}
    # Nivel 0: Letras (abajo, círculo pequeño)
    letras_nodes = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'letra']
    radius_letras = 5
    for i, l in enumerate(letras_nodes):
        # Convertir i y len(letras_nodes) a tensors antes de la división
        angle = 2 * torch.pi * torch.tensor(i, dtype=torch.float32) / torch.tensor(len(letras_nodes), dtype=torch.float32)
        pos[l] = (radius_letras * torch.cos(angle).item(),
                  radius_letras * torch.sin(angle).item() - 10) # Desplazar abajo

    # Nivel 1: Sílabas (medio, círculo mediano)
    silabas_nodes = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'silaba']
    radius_silabas = 12
    for i, s in enumerate(silabas_nodes):
        # Convertir i y len(silabas_nodes) a tensors
        angle = 2 * torch.pi * torch.tensor(i, dtype=torch.float32) / torch.tensor(len(silabas_nodes), dtype=torch.float32)
        pos[s] = (radius_silabas * torch.cos(angle).item(),
                  radius_silabas * torch.sin(angle).item())

    # Nivel 2: Palabras (arriba, círculo grande)
    palabras_nodes = [n for n, data in grafo.nodes(data=True) if data['tipo'] == 'palabra']
    radius_palabras = 20
    for i, p in enumerate(palabras_nodes):
        # Convertir i y len(palabras_nodes) a tensors
        angle = 2 * torch.pi * torch.tensor(i, dtype=torch.float32) / torch.tensor(len(palabras_nodes), dtype=torch.float32)
        pos[p] = (radius_palabras * torch.cos(angle).item(),
                  radius_palabras * torch.sin(angle).item() + 10) # Desplazar arriba

    plt.figure(figsize=(18, 18))
    node_colors = []
    for n in G.nodes:
        if G.nodes[n]['tipo'] == 'letra':
            node_colors.append('lightblue')
        elif G.nodes[n]['tipo'] == 'silaba':
            node_colors.append('lightgreen')
        elif G.nodes[n]['tipo'] == 'palabra':
            node_colors.append('lightcoral')

    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            node_size=1800, font_size=9, font_weight='bold', edge_color='gray', arrows=True, arrowsize=15)
    plt.title("Árbol neuronal jerárquico: Letras → Sílabas → Palabras", size=18)
    plt.show()

visualizar_arbol_fonologico_detallado(G)

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

text = "La casa es grande"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

# Obtén el embedding de la palabra "casa"
# (puedes usar la posición del token o aplicar atención para extraerla)
embedding_casa = outputs.last_hidden_state[0][2]  # índice aproximado

In [ ]:
pip install transformers

In [ ]:
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch

# Cargar tokenizer y modelo base
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')

# Congelar BERT
for param in bert.parameters():
    param.requires_grad = False

# Clasificador binario simple sobre la salida [CLS]
class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Sequential(
            nn.Linear(bert.config.hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    # Modificado para aceptar token_type_ids
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # Pasa token_type_ids al modelo BERT subyacente
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        return self.classifier(cls_output)

# Instanciar modelo
model = BERTClassifier(bert)

# Evaluador: frase coherente o no
def evaluar_con_BERT(oracion):
    # tokenizer returns a BatchEncoding object which is a dictionary-like structure.
    # We pass this dictionary directly to the model using **inputs.
    inputs = tokenizer(oracion, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        salida = model(**inputs) # **inputs unpacks the dictionary into keyword arguments
    return "sí" if salida.item() > 0.5 else "no"

# Ejemplos de variantes (puedes generar más automáticamente con tu GNN)
oraciones = [
    "mi mamá me mima mucho",
    "mi má me muero má",
    "mam mem",
    "c o m á c h o",
    "co cu ma mi",
    "mi mi mamá má"
]

# Evaluar
for oracion in oraciones:
    print(oracion, "→", evaluar_con_BERT(oracion))

In [ ]:
pip install torch_geometric

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import random

# -------------------------------
# CARGAR BERT + TOKENIZER
# -------------------------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')

for param in bert.parameters():
    param.requires_grad = False

class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Sequential(
            nn.Linear(bert.config.hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

bert_model = BERTClassifier(bert)

def evaluar_con_BERT(oracion):
    inputs = tokenizer(oracion, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        salida = bert_model(**inputs)
    return "sí" if salida.item() > 0.5 else "no"

# -------------------------------
# GNN: RED NEURONAL PARA PROPONER CAMBIOS
# -------------------------------
class CorreccionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, in_channels)
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, in_channels)
        self.decoder = nn.Linear(in_channels, vocab_size)

    def forward(self, x, edge_index):
        x = self.emb(x)
        x = torch.relu(self.gcn1(x, edge_index))
        x = self.gcn2(x, edge_index)
        return self.decoder(x)

# -------------------------------
# EJEMPLO: Frase tokenizada en palabras como nodos
# -------------------------------
vocab = {
    "mi": 0, "mamá": 1, "me": 2, "mima": 3, "mucho": 4,
    "muero": 5, "mam": 6, "mem": 7, "má": 8, "co": 9,
    "cu": 10, "ma": 11, "mi.": 12, "<unk>": 13
}
inv_vocab = {v: k for k, v in vocab.items()}

# Convertir lista de palabras a nodos con IDs
def frase_a_grafo(palabras):
    node_ids = torch.tensor([vocab.get(p, vocab["<unk>"]) for p in palabras], dtype=torch.long)
    # Cada nodo se conecta con el siguiente (secuencia lineal)
    edge_index = torch.tensor([[i, i+1] for i in range(len(palabras)-1)], dtype=torch.long).t()
    edge_index = torch.cat([edge_index, edge_index[[1,0]]], dim=1)  # bidireccional
    return Data(x=node_ids, edge_index=edge_index)

# -------------------------------
# LOOP: GNN propone, BERT evalúa
# -------------------------------
gnn = CorreccionGNN(in_channels=16, hidden_channels=32, vocab_size=len(vocab))

def nodos_a_frase(pred_logits):
    pred_ids = pred_logits.argmax(dim=1)
    return ' '.join(inv_vocab[i.item()] for i in pred_ids)

def optimizar_frase(palabras, intentos=5):
    data = frase_a_grafo(palabras)
    mejor_frase = palabras
    for intento in range(intentos):
        out = gnn(data.x, data.edge_index)
        nueva_frase = nodos_a_frase(out)
        respuesta = evaluar_con_BERT(nueva_frase)
        print(f"[{intento}] {nueva_frase} → {respuesta}")
        if respuesta == "sí":
            return nueva_frase
    return mejor_frase

# -------------------------------
# TEST
# -------------------------------
oracion_inicial = ["mi", "má", "me", "muero", "má"]
print("Corrigiendo:", " ".join(oracion_inicial))
resultado = optimizar_frase(oracion_inicial)
print("Resultado final:", resultado)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import random

# Tokenizador y modelo BERT congelado
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
for param in bert.parameters():
    param.requires_grad = False

class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Sequential(
            nn.Linear(bert.config.hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

bert_model = BERTClassifier(bert)

def evaluar_con_BERT(oracion):
    inputs = tokenizer(oracion, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        salida = bert_model(**inputs)
    return salida.item()

# Vocabulario simple de letras
letras = list("abcdefghijklmnopqrstuvwxyzáéíóúñ ")
vocab = {l: i for i, l in enumerate(letras)}
inv_vocab = {i: l for l, i in vocab.items()}

# GNN
class CorreccionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, in_channels)
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, in_channels)
        self.decoder = nn.Linear(in_channels, vocab_size)

    def forward(self, x, edge_index):
        x = self.emb(x)
        x = torch.relu(self.gcn1(x, edge_index))
        x = self.gcn2(x, edge_index)
        return self.decoder(x)

gnn = CorreccionGNN(in_channels=16, hidden_channels=32, vocab_size=len(vocab))

# Armar grafo letra por letra
def frase_a_grafo(frase):
    chars = list(frase.lower())
    node_ids = torch.tensor([vocab.get(c, vocab[' ']) for c in chars], dtype=torch.long)
    edge_index = torch.tensor([[i, i+1] for i in range(len(chars)-1)], dtype=torch.long).t()
    edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)
    return Data(x=node_ids, edge_index=edge_index), chars

def nodos_a_frase(pred_logits):
    pred_ids = pred_logits.argmax(dim=1)
    return ''.join(inv_vocab[i.item()] for i in pred_ids)

# Entrenamiento RL simple (REINFORCE)
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.001)

def entrenar_rl(frase_original, epochs=9000):
    for epoch in range(epochs):
        data, original_chars = frase_a_grafo(frase_original)
        logits = gnn(data.x, data.edge_index)
        nueva_frase = nodos_a_frase(logits)
        reward = evaluar_con_BERT(nueva_frase)

        log_probs = F.log_softmax(logits, dim=1)
        selected_log_probs = log_probs[range(len(data.x)), data.x]  # baseline: original
        loss = -reward * selected_log_probs.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"[{epoch}] {nueva_frase} → reward={reward:.4f}")

# Ejecutar entrenamiento con una frase inicial corrupta
frase_corrupta = "Ahora cuando visualices el grafo, verás cómo las letras se agrupan para formar sílabas y cómo estas sílabas, a su vez, se unen para formar las palabras. Cada palabra tendrá caminos que la conectan con las sílabas que la componen, y estas sílabas a su vez con las letras que las forman."
entrenar_rl(frase_corrupta, epochs=100000)

In [ ]:
!pip install python-Levenshtein

In [ ]:
# Añadir esto en una celda separada ANTES del código que importa Levenshtein
#!pip install python-Levenshtein

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import random

# Instalar la librería Levenshtein si no está presente
# Se recomienda ejecutar esto en una celda separada antes de este código
# Si ya la ejecutaste, puedes comentar la línea.
# !pip install python-Levenshtein
# Re-importar Levenshtein después de la instalación
import Levenshtein


# Tokenizador y modelo BERT congelado
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
for param in bert.parameters():
    param.requires_grad = False

class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Sequential(
            nn.Linear(bert.config.hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

bert_model = BERTClassifier(bert)

def evaluar_con_BERT(oracion):
    inputs = tokenizer(oracion, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        salida = bert_model(**inputs)
    return salida.item()

# Vocabulario simple de letras
letras = list("abcdefghijklmnopqrstuvwxyzáéíóúñ ")
vocab = {l: i for i, l in enumerate(letras)}
inv_vocab = {i: l for l, i in vocab.items()}

# GNN
class CorreccionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, vocab_size, num_layers=3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, in_channels)

        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))

        self.decoder = nn.Linear(hidden_channels, vocab_size) # Decodificador final

    def forward(self, x, edge_index):
        x = self.emb(x)
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1: # Aplicar ReLU en todas menos la última capa GCN
                x = torch.relu(x)
        return self.decoder(x) # La salida no pasa por ReLU para que el softmax sea más libre


def calcular_recompensa_avanzada(frase_generada, frase_objetivo, bert_score_generada):
    """
    Calcula una recompensa combinada:
    - Alta si la frase generada se parece a la frase objetivo (baja distancia de Levenshtein).
    - Modulada por el score de BERT (qué tan "bien" suena la frase).
    """
    # Normalizar distancia de Levenshtein (0 si son iguales, 1 si son muy diferentes)
    max_len = max(len(frase_generada), len(frase_objetivo))
    if max_len == 0:
        levenshtein_norm = 0.0
    else:
        levenshtein_norm = Levenshtein.distance(frase_generada, frase_objetivo) / max_len

    # Queremos minimizar Levenshtein, así que la recompensa es (1 - Levenshtein_norm)
    # y queremos maximizar bert_score_generada.
    # Una recompensa simple podría ser: (1 - levenshtein_norm) * bert_score_generada
    # O para RL, podemos usar el negativo de una función de coste:

    # Recompensa más orientada a RL:
    # Penaliza si es muy diferente o si BERT lo considera "malo"
    # Recompensa = (1 - levenshtein_norm) - (1 - bert_score_generada) * alpha
    # Donde alpha es un peso para la penalización de BERT (ej: 0.5 o 1.0)

    # Para el entrenamiento de REINFORCE, el reward debe ser alto para buenas acciones.
    # Podemos usar una combinación lineal:
    # Un buen balance sería: (1 - levenshtein_norm) + bert_score_generada
    # Donde 1 es perfecto Levenshtein y 1 es perfecto BERT.

    reward = (1 - levenshtein_norm) + bert_score_generada

    return reward

# Re-instanciar GNN con más capas
gnn = CorreccionGNN(in_channels=16, hidden_channels=32, vocab_size=len(vocab), num_layers=2) # 2 capas GCN
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.0005) # Reducir learning rate

# Armar grafo letra por letra
def frase_a_grafo(frase):
    chars = list(frase.lower())
    node_ids = torch.tensor([vocab.get(c, vocab[' ']) for c in chars], dtype=torch.long)
    edge_index = torch.tensor([[i, i+1] for i in range(len(chars)-1)], dtype=torch.long).t()
    edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1) if edge_index.numel() > 0 else torch.empty(2, 0, dtype=torch.long)
    return Data(x=node_ids, edge_index=edge_index), chars

def nodos_a_frase(pred_logits):
    pred_ids = pred_logits.argmax(dim=1)
    return ''.join(inv_vocab[i.item()] for i in pred_ids)

def introducir_errores(texto, num_errores=1):
    """Introduce un número aleatorio de errores (sustitución, inserción, eliminación, transposición)."""
    texto_list = list(texto)
    for _ in range(random.randint(1, num_errores)): # Introduce entre 1 y num_errores

        if not texto_list: # Evitar errores en texto vacío
            continue

        error_tipo = random.choice(['sustituir', 'insertar', 'eliminar', 'transponer'])

        if error_tipo == 'sustituir':
            idx = random.randint(0, len(texto_list) - 1)
            texto_list[idx] = random.choice(letras) # Reemplazar con una letra aleatoria
        elif error_tipo == 'insertar':
            idx = random.randint(0, len(texto_list))
            texto_list.insert(idx, random.choice(letras))
        elif error_tipo == 'eliminar' and len(texto_list) > 0: # Asegurar que hay elementos para eliminar
            idx = random.randint(0, len(texto_list) - 1)
            texto_list.pop(idx)
        elif error_tipo == 'transponer' and len(texto_list) >= 2: # Asegurar al menos 2 elementos para transponer
            idx1 = random.randint(0, len(texto_list) - 2)
            idx2 = idx1 + 1
            texto_list[idx1], texto_list[idx2] = texto_list[idx2], texto_list[idx1]

    return "".join(texto_list)


def entrenar_rl_avanzado(frase_original_limpia, epochs=10000, num_generations=5):
    # Generamos una versión corrupta para el entrenamiento
    frase_corrupta_inicial = introducir_errores(frase_original_limpia, num_errores=3)
    print(f"Frase original para el entrenamiento: '{frase_original_limpia}'")
    print(f"Frase corrupta inicial: '{frase_corrupta_inicial}'")

    for epoch in range(epochs):
        # Cada época, trabajamos con una versión potencialmente corrupta de la frase
        # Nota: Aquí se re-genera el grafo de la frase corrupta inicial en cada época.
        # Si la GNN debe aprender a corregir una *única* frase corrupta, esto está bien.
        # Si debe aprender a corregir *cualquier* frase corrupta, necesitarías
        # generar una nueva frase corrupta (quizás introduciendo errores de nuevo)
        # o usar un batch de frases corruptas.
        current_frase_data, _ = frase_a_grafo(frase_corrupta_inicial)

        # Asegurarse de que el grafo no esté vacío
        if current_frase_data.num_nodes == 0:
            print(f"Epoch {epoch}: Grafo vacío, saltando entrenamiento para esta época.")
            continue

        # Generar múltiples candidatos para encontrar el mejor camino
        candidate_log_probs = []
        candidate_phrases = []
        candidate_rewards = []

        for _ in range(num_generations):
            logits = gnn(current_frase_data.x, current_frase_data.edge_index)

            # Para entrenamiento, queremos muestrear de la distribución de logits
            # En lugar de usar argmax, usamos Gumbel-Softmax o muestreo multinomial
            # Aquí, para simplicidad, aún usamos argmax para la frase generada,
            # pero la pérdida se calcula sobre las log_probs.

            # Muestreo estocástico para las acciones del GNN
            probs = F.softmax(logits, dim=1)
            # Manejar el caso donde probs podría tener NaN o ser irregular
            if torch.isnan(probs).any():
                 print(f"Epoch {epoch}, Generation {_}: NaN detectado en probabilidades, saltando esta generación.")
                 continue

            # Asegurar que las probabilidades sean válidas para Categorical
            probs = torch.clamp(proba, min=1e-8) # Clamp para evitar log(0) si hay 0s exactos
            probs = probs / probs.sum(dim=1, keepdim=True) # Re-normalizar después del clamp

            try:
                m = torch.distributions.Categorical(probs)
                sampled_ids = m.sample() # Muestrear un ID de caracter para cada posición
            except ValueError as e:
                 print(f"Epoch {epoch}, Generation {_}: Error en muestreo de Categorical: {e}. Probabilidades: {probs.min().item(), probs.max().item()}, Suma: {probs.sum(dim=1).min().item(), probs.sum(dim=1).max().item()}. Saltando esta generación.")
                 continue


            generated_frase_list = [inv_vocab.get(i.item(), ' ') for i in sampled_ids] # Usar .get para manejar posibles IDs fuera de vocab
            generated_frase = "".join(generated_frase_list)

            # Calcular recompensa
            bert_score = evaluar_con_BERT(generated_frase)
            reward = calcular_recompensa_avanzada(generated_frase, frase_original_limpia, bert_score)

            candidate_log_probs.append(m.log_prob(sampled_ids).mean()) # Usar el promedio de log_probs para un batch
            candidate_phrases.append(generated_frase)
            candidate_rewards.append(reward)

        # Si no se generaron candidatos válidos, saltar esta época
        if not candidate_rewards:
             print(f"Epoch {epoch}: No se generaron candidatos válidos. Saltando esta época.")
             continue


        # Seleccionar la mejor frase generada de las N muestras para actualizar el modelo
        # Convertir a tensor antes de argmax
        candidate_rewards_tensor = torch.tensor(candidate_rewards)
        best_reward_idx = torch.argmax(candidate_rewards_tensor)
        best_reward = candidate_rewards_tensor[best_reward_idx]
        best_log_prob = candidate_log_probs[best_reward_idx] # Ya es un tensor con 1 elemento
        best_generated_phrase = candidate_phrases[best_reward_idx]

        # Cálculo de la pérdida REINFORCE
        # Ajustamos el baseline para reducir la varianza (promedio de recompensas)
        baseline = torch.mean(candidate_rewards_tensor) # Simple baseline

        # pérdida REINFORCE = - (recompensa - baseline) * log_prob(acción)
        loss = - (best_reward - baseline) * best_log_prob

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0: # Imprimir cada 100 épocas para no saturar
            print(f"--- Época {epoch} ---")
            print(f"Frase Generada (mejor): '{best_generated_phrase}'")
            print(f"Recompensa (mejor): {best_reward:.4f}")
            print(f"Baseline: {baseline:.4f}")
            print(f"Pérdida: {loss.item():.4f}")
            print("-" * 20)

        # Opcional: Si la recompensa alcanza un umbral alto, detener el entrenamiento
        if best_reward > 1.9: # Ejemplo: si la similitud Levenshtein es casi 1 y BERT score es casi 1
             print(f"Umbral de recompensa {1.9} alcanzado. Deteniendo el entrenamiento.")
             break


# Frase original "limpia" para entrenamiento
frase_limpia_ejemplo = "el perro salta y ladra felizmente en el parque"
entrenar_rl_avanzado(frase_limpia_ejemplo, epochs=50000, num_generations=5)

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- 1. Definición del Espacio ---
espacio_dimension = 100 # Un cubo de 100x100x100 unidades

# --- 2. Crear Múltiples Neuronas con Coordenadas 3D ---
num_neuronas = 50
neuronas = {}
for i in range(num_neuronas):
    x = np.random.uniform(0, espacio_dimension)
    y = np.random.uniform(0, espacio_dimension)
    z = np.random.uniform(0, espacio_dimension)
    neuronas[f'N_{i}'] = {'pos': np.array([x, y, z]), 'tipo': random.choice(['excitatoria', 'inhibitoria'])}

# --- 3. Mecanismo de Conectividad (Regla simple: por distancia) ---
G = nx.DiGraph() # Grafo dirigido para las conexiones sinápticas
distancia_max_conexion = 20 # Distancia máxima para que dos neuronas se conecten
prob_conexion_base = 0.1 # Probabilidad base para conectar

for id1, n1_data in neuronas.items():
    G.add_node(id1, pos=n1_data['pos'], tipo=n1_data['tipo'])
    for id2, n2_data in neuronas.items():
        if id1 == id2:
            continue

        dist = np.linalg.norm(n1_data['pos'] - n2_data['pos'])

        if dist < distancia_max_conexion:
            # La probabilidad de conexión disminuye con la distancia
            prob_conexion = prob_conexion_base * (1 - (dist / distancia_max_conexion))
            if random.random() < prob_conexion:
                # Añadir conexión (sinapsis)
                peso = np.random.uniform(0.1, 1.0) # Fuerza sináptica
                G.add_edge(id1, id2, weight=peso)

# --- 4. Simulación de la Actividad (Modelo muy simple) ---
# Aquí solo visualizaremos la red, la simulación de actividad sería el siguiente paso
# Puedes añadir lógica de activación y propagación de pulsos (spikes)

# --- 5. Visualización de las Neuronas en 3D ---
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Dibujar neuronas
node_xyz = np.array([neuronas[node]['pos'] for node in G.nodes()])
ax.scatter(node_xyz[:, 0], node_xyz[:, 1], node_xyz[:, 2],
           c=['red' if G.nodes[node]['tipo'] == 'excitatoria' else 'blue' for node in G.nodes()],
           s=50, label='Neuronas')

# Dibujar conexiones (sinapsis)
for edge in G.edges():
    n1_pos = neuronas[edge[0]]['pos']
    n2_pos = neuronas[edge[1]]['pos']
    ax.plot([n1_pos[0], n2_pos[0]],
            [n1_pos[1], n2_pos[1]],
            [n1_pos[2], n2_pos[2]],
            color='gray', alpha=0.3)

ax.set_title("Simulación simple de neuronas en 3D")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_xlim(0, espacio_dimension)
ax.set_ylim(0, espacio_dimension)
ax.set_zlim(0, espacio_dimension)
plt.legend()
plt.show()

# neuros en un arreglo k means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from matplotlib.colors import ListedColormap

# --- 1. Generar un dataset sintético ---
# Vamos a crear un dataset con 2 características (para que sea fácil de visualizar)
# y 2 clases. No será linealmente separable para desafiar un poco a los modelos.
X, y = make_classification(n_samples=500, n_features=2, n_informative=2,
                           n_redundant=0, n_clusters_per_class=1,
                           random_state=42, class_sep=1.5)

# Dividir el dataset en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Forma de X_train:", X_train.shape)
print("Forma de y_train:", y_train.shape)
print("Forma de X_test:", X_test.shape)
print("Forma de y_test:", y_test.shape)

# Visualizar el dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.8)
plt.title('Dataset Sintético para Clasificación')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.colorbar(label='Clase')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- 2. Implementar y entrenar los modelos ---

# Modelo 1: Regresión Logística
log_reg_model = LogisticRegression(random_state=42)
log_reg_model.fit(X_train, y_train)
y_pred_log_reg = log_reg_model.predict(X_test)

# Modelo 2: K-Nearest Neighbors (elegiremos k=5 para este ejemplo)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

# --- 3. Visualizar las fronteras de decisión ---

def plot_decision_boundary(X, y, model, title):
    h = .02  # Tamaño del paso en la malla
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Predecir la clase para cada punto en la malla
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Crear mapa de colores
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF']) # Para los puntos reales

    plt.figure(figsize=(8, 6))
    plt.pcolormesh(xx, yy, Z, cmap=cmap_light, shading='auto') # Fronteras de decisión

    # Graficar los puntos de entrenamiento/prueba
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold,
                edgecolor='k', s=20)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.title(title)
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.show()

plot_decision_boundary(X_train, y_train, log_reg_model, 'Frontera de Decisión: Regresión Logística')
plot_decision_boundary(X_train, y_train, knn_model, 'Frontera de Decisión: K-Nearest Neighbors (k=5)')

# --- 4. Comparar las métricas usando el conjunto de prueba y validación cruzada ---

print("\n--- Métricas en el Conjunto de Prueba (X_test) ---")

print("\nRegresión Logística:")
print(classification_report(y_test, y_pred_log_reg))
print(f"F1-Score: {f1_score(y_test, y_pred_log_reg):.2f}")
print(f"Precisión: {precision_score(y_test, y_pred_log_reg):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_log_reg):.2f}")

print("\nK-Nearest Neighbors (k=5):")
print(classification_report(y_test, y_pred_knn))
print(f"F1-Score: {f1_score(y_test, y_pred_knn):.2f}")
print(f"Precisión: {precision_score(y_test, y_pred_knn):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_knn):.2f}")

print("\n--- Evaluación con Validación Cruzada (5-Fold) ---")

# Para una evaluación más robusta, usamos cross_val_score
# Scoring con 'f1' para problemas de clasificación binaria es común.
# Puedes cambiar 'f1' por 'accuracy', 'precision', 'recall' si lo necesitas.

log_reg_cv_scores = cross_val_score(log_reg_model, X, y, cv=5, scoring='f1')
print(f"\nRegresión Logística (F1-Score CV): {log_reg_cv_scores.mean():.2f} (+/- {log_reg_cv_scores.std() * 2:.2f})")

knn_cv_scores = cross_val_score(knn_model, X, y, cv=5, scoring='f1')
print(f"K-Nearest Neighbors (F1-Score CV): {knn_cv_scores.mean():.2f} (+/- {knn_cv_scores.std() * 2:.2f})")

##arreglo knn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KNearestNeighborsNeuron(nn.Module):
    """
    Una implementación conceptual de una "neurona con arreglo KNN".
    Esta neurona mantiene una memoria de 'm' patrones y su activación
    se basa en la similitud con los K vecinos más cercanos de esa memoria.
    """
    def __init__(self, input_dim, memory_size=5, k_neighbors=3, learning_rate=0.01):
        super().__init__()
        self.input_dim = input_dim
        self.memory_size = memory_size  # 'm' en nuestra discusión: cuántos patrones memoriza la neurona
        self.k_neighbors = k_neighbors  # 'K' en KNN: cuántos vecinos usa para la activación
        self.learning_rate = learning_rate

        # La "memoria" de la neurona: 'm' patrones, cada uno de 'input_dim' dimensiones
        # Inicializamos los patrones aleatoriamente
        self.register_buffer('patterns', torch.randn(memory_size, input_dim))

        # Un contador simple para el aprendizaje de "reemplazo" (opcional)
        self.register_buffer('pattern_usage_counts', torch.zeros(memory_size))


    def forward(self, x):
        """
        Calcula la activación de la neurona para una entrada 'x'.
        x: Un tensor de entrada de una sola muestra [input_dim]
           o un batch de muestras [batch_size, input_dim]
        """
        if x.dim() == 1: # Si es una sola muestra, expandir para que sea un batch de 1
            x = x.unsqueeze(0)

        # Calcular distancias euclidianas de la entrada a cada patrón en la memoria
        # x: [batch_size, input_dim]
        # self.patterns: [memory_size, input_dim]
        # Distances: [batch_size, memory_size]

        # Expandir las dimensiones para la resta y la norma
        expanded_x = x.unsqueeze(1)          # [batch_size, 1, input_dim]
        expanded_patterns = self.patterns.unsqueeze(0) # [1, memory_size, input_dim]

        # Calcular la distancia euclidiana al cuadrado (más eficiente que la raíz cuadrada)
        # torch.cdist también se puede usar, pero la forma manual es clara aquí.
        distances_sq = torch.sum((expanded_x - expanded_patterns)**2, dim=2) # [batch_size, memory_size]
        distances = torch.sqrt(distances_sq) # [batch_size, memory_size]

        # Encontrar los K vecinos más cercanos (menores distancias)
        # topk retorna (valores, índices)
        # Queremos los k_neighbors más pequeños, por eso invertimos las distancias con -
        # o simplemente usamos torch.topk(..., largest=False)

        # Asegurarse de que k_neighbors no sea mayor que memory_size
        actual_k = min(self.k_neighbors, self.memory_size)

        # Tomar los k vecinos más cercanos
        # Si k_neighbors es 0, simplemente devolver 0.0 o manejar de otra forma
        if actual_k == 0:
            return torch.zeros(x.size(0), device=x.device)

        k_nearest_distances, _ = torch.topk(distances, k=actual_k, largest=False, dim=1) # [batch_size, actual_k]

        # Calcular la activación: Inverso del promedio de las distancias (o suma inversa)
        # Una menor distancia debería significar una mayor activación
        # Podemos usar 1 / (1 + distancia) o alguna función de kernel como exp(-distancia^2)

        # Usaremos el promedio de las inversas de las distancias para una "activación" simple.
        # Evitar división por cero si la distancia es 0
        activations = 1.0 / (1.0 + k_nearest_distances) # [batch_size, actual_k]

        # Sumar o promediar las activaciones de los k vecinos
        neuron_output = torch.mean(activations, dim=1) # [batch_size]

        return neuron_output

    def learn_from_input(self, x_batch, threshold_distance=0.5):
        """
        Una función simplificada de "aprendizaje" para actualizar los patrones internos.
        No es backpropagation, sino un ajuste al estilo competitivo/prototipo.

        Para cada muestra en el batch:
        - Encuentra el patrón más cercano en la memoria.
        - Si la muestra es suficientemente diferente del patrón más cercano (o no lo activa lo suficiente),
          podríamos añadirla como un nuevo patrón o reemplazar el menos utilizado.
        - Si es muy similar, podríamos ajustar ligeramente el patrón más cercano hacia la muestra.

        Esta es una aproximación y puede ser más compleja en un modelo real.
        """
        # Asegurar que x_batch es 2D
        if x_batch.dim() == 1:
            x_batch = x_batch.unsqueeze(0)

        for i in range(x_batch.size(0)):
            x = x_batch[i]

            # Calcular distancias a todos los patrones
            distances = torch.sqrt(torch.sum((x - self.patterns)**2, dim=1)) # [memory_size]

            # Encontrar el patrón más cercano
            min_dist, closest_pattern_idx = torch.min(distances, dim=0)

            # Estrategia de aprendizaje simple:
            if min_dist > threshold_distance:
                # Si la entrada es "nueva" o lo suficientemente diferente del patrón más cercano,
                # reemplazar el patrón menos utilizado o el más antiguo en la memoria.
                least_used_idx = torch.argmin(self.pattern_usage_counts)
                self.patterns[least_used_idx] = x.clone().detach() # Reemplazar con la nueva muestra
                self.pattern_usage_counts[least_used_idx] = 1 # Reiniciar su conteo
            else:
                # Si la entrada es similar a un patrón existente, ajustar ese patrón ligeramente.
                # Esto es como el aprendizaje de Kohonen (SOM)
                self.patterns[closest_pattern_idx] = (1 - self.learning_rate) * self.patterns[closest_pattern_idx] + \
                                                      self.learning_rate * x.clone().detach()
                self.pattern_usage_counts[closest_pattern_idx] += 1 # Incrementar el uso


# --- Ejemplo de Uso ---
input_dimension = 10  # Dimensión de las entradas a esta neurona
memory_capacity = 5   # Número de patrones que la neurona puede "recordar"
k_to_use = 3          # Usar los 3 patrones más cercanos para la activación

# Crear una neurona con arreglo KNN
neuron = KNearestNeighborsNeuron(input_dim=input_dimension,
                                  memory_size=memory_capacity,
                                  k_neighbors=k_to_use)

print(f"Neurona creada con {memory_capacity} patrones en memoria y K={k_to_use} vecinos.")
print("Patrones iniciales de la neurona:\n", neuron.patterns)

# --- Simular el proceso de aprendizaje ---
print("\n--- Simulación de Aprendizaje ---")
# Generar algunas "experiencias" (datos de entrada)
num_learning_steps = 20
for step in range(num_learning_steps):
    # Crear una entrada aleatoria o un patrón específico
    if step % 5 == 0: # Simular algunas entradas "nuevas" o muy diferentes
        new_input = torch.randn(input_dimension) * 2 # Más varianza para ver el reemplazo
    else: # Entradas más cercanas a un patrón base para ver el ajuste
        new_input = torch.randn(input_dimension) * 0.1 + torch.tensor([1.0]*input_dimension) # clustering alrededor de [1,1,...,1]

    # La neurona "aprende" de esta entrada
    neuron.learn_from_input(new_input)
    # print(f"Paso {step+1}: Entrada procesada. Patrones actualizados.")


print("\nPatrones finales de la neurona después del aprendizaje:\n", neuron.patterns)
print("Conteo de uso de patrones:\n", neuron.pattern_usage_counts)

# --- Simular la inferencia (activación) ---
print("\n--- Simulación de Inferencia (Activación) ---")
test_input_1 = torch.tensor([1.0]*input_dimension) + torch.randn(input_dimension) * 0.05 # Muy similar a un patrón aprendido
test_input_2 = torch.randn(input_dimension) * 3       # Muy diferente (ruido)
test_input_3 = torch.tensor([-1.0]*input_dimension) + torch.randn(input_dimension) * 0.05 # Nuevo patrón

activation_1 = neuron(test_input_1)
activation_2 = neuron(test_input_2)
activation_3 = neuron(test_input_3)

print(f"Activación para Input 1 (similar a patrón aprendido): {activation_1.item():.4f}")
print(f"Activación para Input 2 (ruido/muy diferente): {activation_2.item():.4f}")
print(f"Activación para Input 3 (nuevo patrón distinto): {activation_3.item():.4f}")

# Simular un batch de entrada
batch_inputs = torch.randn(4, input_dimension)
batch_activations = neuron(batch_inputs)
print(f"\nActivaciones para un batch de entradas:\n {batch_activations}")

## arreglo árbol de decisiones

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class IDTNeuron(nn.Module):
    """
    Una implementación conceptual de una "Neurona de Árbol de Decisión Integrado (IDT-Neuron)".
    Cada neurona encapsula un pequeño DecisionTreeClassifier.
    """
    def __init__(self, input_dim, output_classes=2, max_depth=3, random_state=42):
        super().__init__()
        self.input_dim = input_dim
        self.output_classes = output_classes # Por ejemplo, si el árbol interno clasifica en 2 clases (relevante/no relevante)
        self.max_depth = max_depth
        self.random_state = random_state

        # Cada neurona tiene su propio Decision Tree Classifier
        # NOTA: En un escenario real, este árbol se entrenaría localmente.
        # Aquí, lo inicializamos, y la función 'learn_locally' lo entrenará.
        self.decision_tree = DecisionTreeClassifier(max_depth=self.max_depth,
                                                    random_state=self.random_state)

        # Una pequeña "memoria" local para simular el dataset de entrenamiento interno del árbol
        # Esto es para la simulación del aprendizaje local.
        self.local_X_memory = []
        self.local_y_memory = []
        self.memory_capacity = 50 # Cuántas muestras puede "recordar" para entrenar su árbol

    def forward(self, x):
        """
        Calcula la activación de la neurona pasando la entrada a su árbol de decisión interno.
        x: Un tensor de entrada de una sola muestra [input_dim]
           o un batch de muestras [batch_size, input_dim]
        """
        if x.dim() == 1: # Si es una sola muestra, expandir para que sea un batch de 1
            x_np = x.detach().cpu().numpy().reshape(1, -1)
        else:
            x_np = x.detach().cpu().numpy()

        # Si el árbol aún no ha sido entrenado (no tiene clases_ activas), no puede predecir.
        # Esto ocurre antes de la primera llamada a learn_locally.
        if not hasattr(self.decision_tree, 'classes_') or len(self.local_X_memory) < 2:
            # Devuelve una activación neutral o un indicador de "no entrenado"
            return torch.zeros(x.size(0), device=x.device, dtype=torch.float32)

        # Usamos predict_proba para obtener una activación "suave" (probabilidad)
        # o predict para una activación "dura" (0 o 1)
        # Aquí elegimos la probabilidad de la clase positiva (índice 1) como activación.

        # [batch_size, output_classes]
        probabilities = self.decision_tree.predict_proba(x_np)

        # Asumiendo que la clase positiva es el índice 1
        # Si output_classes > 2, necesitarías decidir cómo combinar las probabilidades.
        activation = torch.tensor(probabilities[:, 1], dtype=torch.float32, device=x.device)

        return activation

    def learn_locally(self, X_local_samples, y_local_labels):
        """
        Simula el aprendizaje local de la neurona, entrenando su árbol de decisión interno.
        X_local_samples: Un batch de características de entrada para que esta neurona aprenda.
                         [num_samples, input_dim]
        y_local_labels: Etiquetas asociadas a estas muestras (el "objetivo" local de esta neurona).
                        [num_samples]
        """
        if not isinstance(X_local_samples, np.ndarray):
            X_local_samples = X_local_samples.detach().cpu().numpy()
        if not isinstance(y_local_labels, np.ndarray):
            y_local_labels = y_local_labels.detach().cpu().numpy()

        # Añadir las nuevas muestras a la memoria local de la neurona
        for x_samp, y_samp in zip(X_local_samples, y_local_labels):
            self.local_X_memory.append(x_samp)
            self.local_y_memory.append(y_samp)

            # Mantener la memoria dentro de la capacidad definida
            if len(self.local_X_memory) > self.memory_capacity:
                self.local_X_memory.pop(0) # Eliminar la muestra más antigua
                self.local_y_memory.pop(0)

        # Solo entrenar si hay suficientes muestras en la memoria
        if len(self.local_X_memory) >= 2: # Se necesitan al menos 2 muestras para entrenar un DT
            current_X = np.array(self.local_X_memory)
            current_y = np.array(self.local_y_memory)

            # Entrenar el árbol de decisión interno con la memoria acumulada
            self.decision_tree.fit(current_X, current_y)

            # Opcional: imprimir la precisión local del árbol
            # y_pred_local = self.decision_tree.predict(current_X)
            # local_acc = accuracy_score(current_y, y_pred_local)
            # print(f"  Neurona entrenada localmente. Precisión interna: {local_acc:.2f}")

    def print_internal_tree(self, feature_names=None):
        """Muestra la estructura del árbol de decisión interno de la neurona."""
        if hasattr(self.decision_tree, 'tree_') and self.decision_tree.tree_.node_count > 1:
            print(f"\n--- Estructura del Árbol de Decisión Interno de IDT-Neuron ---")
            r = export_text(self.decision_tree, feature_names=feature_names)
            print(r)
        else:
            print("\nEl árbol de decisión interno aún no ha sido entrenado o es trivial.")


# --- Ejemplo de Uso ---

# 1. Crear una neurona IDT
input_dimension = 5 # La neurona espera 5 características de entrada
neuron = IDTNeuron(input_dim=input_dimension, output_classes=2, max_depth=2)
print(f"Neurona IDT creada. Profundidad máxima del árbol interno: {neuron.max_depth}")

# 2. Simular el Aprendizaje Local para la Neurona
print("\n--- Simulación de Aprendizaje Local ---")

# Generar algunos datos de ejemplo para que la neurona aprenda localmente
# X_data: características que llegarían a esta neurona desde la capa anterior
# y_labels: la "señal de supervisión" local para esta neurona (ej. si esta entrada debe activar la neurona o no)
np.random.seed(0)
num_samples = 40
X_learning_data = np.random.rand(num_samples, input_dimension) * 10
# Crear etiquetas simples: si la primera característica es alta, la etiqueta es 1, de lo contrario 0.
y_learning_labels = (X_learning_data[:, 0] > 5).astype(int)

# Simular "batches" de aprendizaje local
for i in range(0, num_samples, 10): # Aprender en grupos de 10 muestras
    batch_X = X_learning_data[i : i+10]
    batch_y = y_learning_labels[i : i+10]
    print(f"Aprendiendo con muestras {i} a {i+len(batch_X)-1}")
    neuron.learn_locally(batch_X, batch_y)

# Mostrar el árbol interno después del aprendizaje
neuron.print_internal_tree(feature_names=[f'Feature_{i+1}' for i in range(input_dimension)])


# 3. Simular la Inferencia (Activación)
print("\n--- Simulación de Inferencia (Activación) ---")

# Crear nuevas entradas para probar la neurona
test_input_1 = torch.tensor([8.0, 2.0, 3.0, 1.0, 5.0], dtype=torch.float32) # Debería activar fuerte (Feature_1 > 5)
test_input_2 = torch.tensor([2.0, 7.0, 1.0, 9.0, 4.0], dtype=torch.float32) # Debería activar débil (Feature_1 <= 5)
test_input_3 = torch.tensor([6.0, 0.5, 0.1, 9.5, 2.0], dtype=torch.float32) # Debería activar fuerte

activation_1 = neuron(test_input_1)
activation_2 = neuron(test_input_2)
activation_3 = neuron(test_input_3)

print(f"Activación para Input 1 (Feature_1=8.0): {activation_1.item():.4f}")
print(f"Activación para Input 2 (Feature_1=2.0): {activation_2.item():.4f}")
print(f"Activación para Input 3 (Feature_1=6.0): {activation_3.item():.4f}")

# Simular un batch de entrada
batch_test_inputs = torch.tensor([
    [7.5, 1.1, 2.3, 4.5, 6.7],
    [3.1, 8.2, 5.4, 0.9, 1.2],
    [9.0, 0.1, 0.2, 0.3, 0.4]
], dtype=torch.float32)
batch_activations = neuron(batch_test_inputs)
print(f"\nActivaciones para un batch de entradas:\n {batch_activations}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from matplotlib.colors import ListedColormap
import warnings

# Suprimir advertencias para una salida más limpia
warnings.filterwarnings("ignore")

# --- 1. Generar un dataset sintético para clasificación ---
# Creamos un dataset con 2 características para que sea fácil de visualizar
# y 2 clases. Intentaremos que no sea linealmente separable para desafiar a los modelos.
X, y = make_classification(n_samples=500, n_features=2, n_informative=2,
                           n_redundant=0, n_clusters_per_class=1,
                           random_state=42, class_sep=1.5) # class_sep controla la separación entre clases

# Dividir el dataset en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Forma de X_train:", X_train.shape)
print("Forma de y_train:", y_train.shape)
print("Forma de X_test:", X_test.shape)
print("Forma de y_test:", y_test.shape)

# Visualizar el dataset
plt.figure(figsize=(9, 7))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.8, edgecolors='k')
plt.title('Dataset Sintético para Clasificación')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.colorbar(label='Clase')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- 2. Definir los clasificadores a comparar ---
classifiers = {
    "Regresión Logística": LogisticRegression(random_state=42),
    "K-Nearest Neighbors (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine (Lineal)": SVC(kernel='linear', random_state=42, probability=True), # probability=True para cross_val_score con F1
    "Árbol de Decisión (Max Depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Naive Bayes (Gaussiano)": GaussianNB()
}

# --- 3. Bucle para entrenar, visualizar y evaluar cada clasificador ---

# Preparar la visualización de fronteras de decisión
# Crear una malla para dibujar las fronteras
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF']) # Colores para las regiones de decisión
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])   # Colores para los puntos de datos reales

results = {} # Para almacenar los resultados de la validación cruzada

print("\n--- Entrenando y Evaluando Clasificadores ---")

for name, classifier in classifiers.items():
    print(f"\n***** Modelo: {name} *****")

    # Entrenar el modelo
    classifier.fit(X_train, y_train)

    # Predecir en el conjunto de prueba
    y_pred = classifier.predict(X_test)

    # --- Visualizar fronteras de decisión ---
    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(9, 7))
    plt.pcolormesh(xx, yy, Z, cmap=cmap_light, shading='auto') # Fronteras de decisión

    # Graficar los puntos de entrenamiento
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cmap_bold,
                edgecolor='k', s=20, label='Datos de Entrenamiento')
    # Graficar los puntos de prueba (opcional, para ver cómo caen)
    # plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cmap_bold,
    #             edgecolor='k', s=20, marker='x', alpha=0.6, label='Datos de Prueba')

    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.title(f'Frontera de Decisión: {name}')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend()
    plt.show()

    # --- Imprimir métricas en el conjunto de prueba ---
    print("\nMétricas en el Conjunto de Prueba:")
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(f"F1-Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.2f}") # Weighted F1 para clases potencialmente desequilibradas

    # --- Realizar Validación Cruzada para una evaluación robusta ---
    # Usaremos 5-fold cross-validation y F1-score como métrica principal
    # para problemas de clasificación binaria.
    cv_scores = cross_val_score(classifier, X, y, cv=5, scoring='f1_weighted')
    results[name] = cv_scores.mean()
    print(f"\nF1-Score (Weighted) con 5-Fold Cross-Validation: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")
    print("-" * 50)

# --- Resumen Final de Rendimiento ---
print("\n--- Resumen de F1-Score (Weighted) en Cross-Validation ---")
for name, score in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f"{name}: {score:.2f}")

##bayesiano

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs # Para generar datos para la neurona interna
from sklearn.model_selection import train_test_split

class NBNNeuron(nn.Module):
    """
    Una implementación conceptual de una "Neurona con Clasificador Bayesiano Ingenuo Integrado".
    Cada neurona encapsula su propio Clasificador Bayesiano Ingenuo.
    """
    def __init__(self, input_dim, output_classes=2):
        super().__init__()
        self.input_dim = input_dim
        self.output_classes = output_classes # Número de "clases" internas que el NB de la neurona aprenderá
                                            # Por ejemplo, 2 para 'patrón A' vs 'patrón B'

        # Cada neurona tiene su propio Clasificador Bayesiano Gaussiano
        self.naive_bayes_classifier = GaussianNB()

        # Una pequeña "memoria" local para simular el dataset de entrenamiento interno del NB
        self.local_X_memory = []
        self.local_y_memory = []
        self.memory_capacity = 100 # Cuántas muestras puede "recordar" para entrenar su NB

    def forward(self, x):
        """
        Calcula la activación de la neurona pasando la entrada a su clasificador Naive Bayes interno.
        x: Un tensor de entrada de una sola muestra [input_dim]
           o un batch de muestras [batch_size, input_dim]
        """
        if x.dim() == 1: # Si es una sola muestra, expandir para que sea un batch de 1
            x_np = x.detach().cpu().numpy().reshape(1, -1)
        else:
            x_np = x.detach().cpu().numpy()

        # Verificar si el clasificador interno ha sido entrenado
        if not hasattr(self.naive_bayes_classifier, 'classes_') or len(self.local_X_memory) < 2:
            # Si no ha sido entrenado, devuelve una activación neutral
            return torch.zeros(x.size(0), device=x.device, dtype=torch.float32)

        # Usamos predict_proba para obtener una activación "suave" (probabilidad)
        # La activación de la neurona será la probabilidad de la clase positiva (índice 1).
        # [batch_size, output_classes]
        probabilities = self.naive_bayes_classifier.predict_proba(x_np)

        # Asumiendo que la clase positiva es el índice 1
        # Si output_classes > 2, podrías elegir la probabilidad de la clase más alta, o la suma de ciertas probabilidades.
        activation = torch.tensor(probabilities[:, 1], dtype=torch.float32, device=x.device)

        return activation

    def learn_locally(self, X_local_samples, y_local_labels):
        """
        Simula el aprendizaje local de la neurona, entrenando/actualizando su clasificador Naive Bayes interno.
        X_local_samples: Un batch de características de entrada para que esta neurona aprenda.
                         [num_samples, input_dim]
        y_local_labels: Etiquetas asociadas a estas muestras (el "objetivo" local de esta neurona).
                        [num_samples]
        """
        if not isinstance(X_local_samples, np.ndarray):
            X_local_samples = X_local_samples.detach().cpu().numpy()
        if not isinstance(y_local_labels, np.ndarray):
            y_local_labels = y_local_labels.detach().cpu().numpy()

        # Añadir las nuevas muestras a la memoria local de la neurona
        for x_samp, y_samp in zip(X_local_samples, y_local_labels):
            self.local_X_memory.append(x_samp)
            self.local_y_memory.append(y_samp)

            # Mantener la memoria dentro de la capacidad definida
            if len(self.local_X_memory) > self.memory_capacity:
                self.local_X_memory.pop(0) # Eliminar la muestra más antigua
                self.local_y_memory.pop(0)

        # Solo entrenar si hay suficientes muestras y al menos dos clases en la memoria
        if len(self.local_X_memory) >= 2 and len(np.unique(self.local_y_memory)) >= 2:
            current_X = np.array(self.local_X_memory)
            current_y = np.array(self.local_y_memory)

            # Entrenar el clasificador Naive Bayes interno con la memoria acumulada
            # Nota: GaussianNB puede ser entrenado de forma incremental con partial_fit
            # pero para esta simulación simple, reentrenamos con todo lo acumulado.
            self.naive_bayes_classifier.fit(current_X, current_y)

            # Opcional: imprimir la precisión local del NB
            # y_pred_local = self.naive_bayes_classifier.predict(current_X)
            # local_acc = accuracy_score(current_y, y_pred_local)
            # print(f"  Neurona entrenada localmente. Precisión interna: {local_acc:.2f}")


# --- 1. Simulación de la Neurona NBN ---
print("--- Simulación de la Neurona NBN ---")
input_dimension = 5 # La neurona espera 5 características de entrada
neuron = NBNNeuron(input_dim=input_dimension, output_classes=2)
print(f"Neurona NBN creada. Espera {input_dimension} características de entrada.")

# Generar datos para el aprendizaje local de la neurona
# Creamos dos "blobs" de datos, simulando dos tipos de patrones que la neurona podría aprender.
X_local_data, y_local_labels = make_blobs(n_samples=150, n_features=input_dimension,
                                          centers=2, cluster_std=1.0, random_state=42)

# Simular el aprendizaje local de la neurona en batches
print("\n--- Simulación de Aprendizaje Local para NBN-Neuron ---")
for i in range(0, len(X_local_data), 20): # Aprender en grupos de 20 muestras
    batch_X = X_local_data[i : i+20]
    batch_y = y_local_labels[i : i+20]
    print(f"Neurona aprendiendo con muestras {i} a {i+len(batch_X)-1}")
    neuron.learn_locally(batch_X, batch_y)

# 2. Simular la Inferencia (Activación) de la Neurona NBN
print("\n--- Simulación de Inferencia (Activación) de NBN-Neuron ---")
test_input_1 = torch.tensor([0.5, 0.5, 0.5, 0.5, 0.5], dtype=torch.float32) # Cercano al centro del primer blob
test_input_2 = torch.tensor([3.0, 3.0, 3.0, 3.0, 3.0], dtype=torch.float32) # Cercano al centro del segundo blob
test_input_3 = torch.tensor([10.0, 10.0, 10.0, 10.0, 10.0], dtype=torch.float32) # Muy diferente

activation_1 = neuron(test_input_1)
activation_2 = neuron(test_input_2)
activation_3 = neuron(test_input_3)

print(f"Activación para Input 1 (cercano al centro del blob 0): {activation_1.item():.4f}")
print(f"Activación para Input 2 (cercano al centro del blob 1): {activation_2.item():.4f}")
print(f"Activación para Input 3 (muy diferente): {activation_3.item():.4f}")

# Simular un batch de entrada
batch_test_inputs = torch.tensor([
    [0.1, 0.2, 0.3, 0.4, 0.5],
    [2.8, 3.1, 2.9, 3.0, 3.2],
    [5.0, 5.0, 5.0, 5.0, 5.0] # Un punto intermedio
], dtype=torch.float32)
batch_activations = neuron(batch_test_inputs)
print(f"\nActivaciones para un batch de entradas:\n {batch_activations}")


# --- Comparativa Justa: NBN-Neuron (como componente) vs. GaussianNB (como clasificador completo) ---
print("\n--- Comparativa Justa: NBN-Neuron vs. GaussianNB Completo ---")

# Vamos a usar el mismo dataset con el que "entrenamos" la neurona NBN.
# X_local_data y y_local_labels son los datos que la neurona "vio" y almacenó.

# Entrenar un GaussianNB estándar con el mismo dataset completo
nb_full_model = GaussianNB()
nb_full_model.fit(X_local_data, y_local_labels)

print("\nModelo GaussianNB completo entrenado con todo el dataset.")

# --- 2. Inferencia y Predicción del Modelo Completo ---
print("\n--- Inferencia con el Modelo GaussianNB Completo ---")

# Usar las mismas entradas de prueba que usamos para la NBN-Neuron
test_inputs_np = torch.stack([test_input_1, test_input_2, test_input_3]).numpy()

# Predecir probabilidades con el modelo completo
probabilities_full_model = nb_full_model.predict_proba(test_inputs_np)

print(f"Probabilidad de Clase 1 (modelo completo) para Input 1: {probabilities_full_model[0, 1]:.4f}")
print(f"Probabilidad de Clase 1 (modelo completo) para Input 2: {probabilities_full_model[1, 1]:.4f}")
print(f"Probabilidad de Clase 1 (modelo completo) para Input 3: {probabilities_full_model[2, 1]:.4f}")

# Predecir clases con el modelo completo
predictions_full_model = nb_full_model.predict(test_inputs_np)
print(f"Predicción de Clase (modelo completo) para Input 1: {predictions_full_model[0]}")
print(f"Predicción de Clase (modelo completo) para Input 2: {predictions_full_model[1]}")
print(f"Predicción de Clase (modelo completo) para Input 3: {predictions_full_model[2]}")

# Comparar con las activaciones de la NBN-Neuron
print("\n--- Comparación Directa de Activaciones ---")
print(f"NBN-Neuron Activation (Input 1): {activation_1.item():.4f} | GaussianNB Full Model Probability (Input 1, Class 1): {probabilities_full_model[0, 1]:.4f}")
print(f"NBN-Neuron Activation (Input 2): {activation_2.item():.4f} | GaussianNB Full Model Probability (Input 2, Class 1): {probabilities_full_model[1, 1]:.4f}")
print(f"NBN-Neuron Activation (Input 3): {activation_3.item():.4f} | GaussianNB Full Model Probability (Input 3, Class 1): {probabilities_full_model[2, 1]:.4f}")

#neuronas con algoritmos Genéticos

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- Dataset Sintético ---
X_data = torch.randn(100, 4)  # 100 muestras, 4 características
y_data = (X_data[:, 0]**2 + X_data[:, 1]**2 + X_data[:, 2] + 0.5 * torch.randn(100)).unsqueeze(1)

X_np = X_data.numpy()
y_np = y_data.numpy()
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()
X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()

# --- Clase Neurona Personalizada ---
class Neurona(nn.Module):
    _next_id = 0
    def __init__(self, input_size):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1))
        self.ajustes = 1 # Contador de ajustes por optimizador
        self.generation = 2 # Generación en la que fue creada/evolucionada
        self.id = Neurona._next_id # Asignar un ID único a cada neurona
        Neurona._next_id += 1

    def forward(self, x):
        # x es (batch_size, input_size)
        # self.w es (input_size)
        # Realiza el producto punto para cada fila (muestra) en el lote.
        # (batch_size, input_size) * (input_size) -> (batch_size, 1) después de sum
        # El .sum(dim=1, keepdim=True) asegura que la salida sea (batch_size, 1)
        return (x * self.w).sum(dim=1, keepdim=True) + self.b

    def mutate_from(self, p1, p2=None, sigma=0.01, generation_num=0):
        with torch.no_grad():
            if p2:
                self.w.copy_(0.1 * p1.w + 0.1 * p2.w + sigma * torch.randn_like(self.w))
                self.b.copy_(0.1 * p1.b + 0.1 * p2.b + sigma * torch.randn_like(self.b))
            else: # Mutación de un solo padre (p1)
                self.w.copy_(p1.w + sigma * torch.randn_like(self.w))
                self.b.copy_(p1.b + sigma * torch.randn_like(self.b))
        self.ajustes = 0 # Resetear ajustes al ser mutada/clonada
        self.generation = generation_num # Registrar la generación de origen
        self.id = Neurona._next_id # Asignar un nuevo ID único
        Neurona._next_id += 1

# --- Capa Genética ---
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])
        self.initial_num_neurons = num_neuronas # Guardar el número inicial

    def forward(self, x):
        # x es (batch_size, input_size)
        # Cada neurona 'n' procesa el lote 'x' y produce una salida de forma (batch_size, 1)
        # Creamos una lista de estas salidas: [ (batch_size, 1), (batch_size, 1), ... ]
        neuron_outputs = [n(x) for n in self.neuronas]

        # Concatenamos estas salidas a lo largo de la dimensión de las características (dim=1)
        # Esto resulta en un tensor de forma (batch_size, num_neuronas)
        return torch.cat(neuron_outputs, dim=1)

    def registrar_ajustes(self):
        for n in self.neuronas:
            # Comprobar si el gradiente existe y si tiene algún valor no cero
            # Esto indica que la neurona contribuyó a la pérdida y fue ajustada
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1
            # Opcional: también podrías registrar ajustes por el sesgo si lo consideras
            # if n.b.grad is not None and torch.any(n.b.grad != 0):
            #     n.ajustes += 1

    def evolucionar(self, generation_num, pruning_threshold=10):
        print(f"\n[Evolución - Generación {generation_num}]")

        # Asegurarse de tener al menos 2 neuronas para selección de padres
        if len(self.neuronas) < 2:
            print("Insuficientes neuronas para evolución. Saltando.")
            return

        # Filtrar las neuronas que no han sido inicializadas correctamente
        valid_neuronas = [n for n in self.neuronas if hasattr(n, 'ajustes')]

        if not valid_neuronas:
            print("No hay neuronas válidas para evolucionar.")
            return

        # Asegurarse de que tenemos al menos 2 neuronas válidas para la selección
        if len(valid_neuronas) < 2:
            print("Insuficientes neuronas válidas para seleccionar padres. Saltando evolución.")
            return

        # Ordenar por ajustes (menor es mejor) para encontrar a los "mejores" padres
        mejores = sorted(valid_neuronas, key=lambda n: n.ajustes)[:2]

        nuevas_neuronas = []
        podadas_count = 0

        for i, n in enumerate(self.neuronas):
            if n.ajustes > pruning_threshold: # Criterio de poda
                # REEMPLAZO: crear una nueva neurona mutada de los mejores padres
                nuevo = Neurona(n.w.size(0)) # Mantener el mismo input_size
                nuevo.mutate_from(mejores[0], mejores[1], generation_num=generation_num)
                nuevas_neuronas.append(nuevo)
                podadas_count += 1
                # print(f"  - Neurona ID: {n.id} (Ajustes: {n.ajustes}) podada.") # Desactivado para menos verbosidad
            else:
                # Si no se poda, mantener la neurona existente
                nuevas_neuronas.append(n)

        # Resetear ajustes para todas las neuronas (las nuevas y las que se mantuvieron)
        # Esto es crucial para que el criterio de poda se base en los ajustes desde la última evolución
        for n in nuevas_neuronas:
            n.ajustes = 0

        # Actualizar la lista de neuronas de la capa
        self.neuronas = nn.ModuleList(nuevas_neuronas)
        print(f"  => {podadas_count} neuronas reemplazadas.")


# --- Red Completa con Evolución (Tu Modelo Genético) ---
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.oculta = Capa(input_size, hidden_size)
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.oculta(x)
        x = F.relu(x)
        return self.salida(x)

    def registrar_ajustes(self):
        self.oculta.registrar_ajustes()

    def evolucionar(self, gen, pruning_threshold=10):
        self.oculta.evolucionar(gen, pruning_threshold)

# --- Modelo Estándar (sin neuronas genéticas para comparación) ---
class RedEstandar(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return self.fc2(x)

# --- Configuración de Parámetros Globales ---
input_dim = 4
hidden_dim = 100
output_dim = 1
EPOCHS = 3000
LR = 0.01
EVOLUTION_FREQ = 500 # Cada cuántas épocas se activa la evolución
PRUNING_THRESHOLD = 5 # Umbral de ajustes para podar una neurona

# --- Entrenamiento del Modelo Genético ---
print("\n" + "="*40)
print("--- Entrenando Red Genética ---")
print("="*40)
modelo_genetico = RedGenetica(input_dim, hidden_dim, output_dim)
opt_genetico = torch.optim.SGD(modelo_genetico.parameters(), lr=LR)
criterion = nn.MSELoss()

losses_genetico = []
for epoch in range(EPOCHS):
    modelo_genetico.train()
    opt_genetico.zero_grad()

    out = modelo_genetico(X_train)
    loss = criterion(out, y_train)
    loss.backward()

    modelo_genetico.registrar_ajustes() # Registra los ajustes antes de optimizar
    opt_genetico.step()

    losses_genetico.append(loss.item())

    if (epoch + 1) % EVOLUTION_FREQ == 0:
        modelo_genetico.evolucionar((epoch + 1) // EVOLUTION_FREQ, pruning_threshold=PRUNING_THRESHOLD)

# --- Evaluación del Modelo Genético ---
modelo_genetico.eval()
with torch.no_grad():
    preds_genetico = modelo_genetico(X_test)
    test_loss_genetico = criterion(preds_genetico, y_test).item()
print(f"\n[Resultados Modelo Genético] Pérdida en test: {test_loss_genetico:.4f}")

# --- Entrenamiento del Modelo Estándar ---
print("\n" + "="*40)
print("--- Entrenando Red Estándar ---")
print("="*40)
modelo_estandar = RedEstandar(input_dim, hidden_dim, output_dim)
opt_estandar = torch.optim.SGD(modelo_estandar.parameters(), lr=LR)

losses_estandar = []
for epoch in range(EPOCHS):
    modelo_estandar.train()
    opt_estandar.zero_grad()

    out = modelo_estandar(X_train)
    loss = criterion(out, y_train)
    loss.backward()
    opt_estandar.step()

    losses_estandar.append(loss.item())

# --- Evaluación del Modelo Estándar ---
modelo_estandar.eval()
with torch.no_grad():
    preds_estandar = modelo_estandar(X_test)
    test_loss_estandar = criterion(preds_estandar, y_test).item()
print(f"\n[Resultados Modelo Estándar] Pérdida en test: {test_loss_estandar:.4f}")

# --- Visualización Comparativa ---
plt.figure(figsize=(12, 6))
plt.plot(losses_genetico, label="Genético: Loss de Entrenamiento", color='blue')
plt.plot(losses_estandar, label="Estándar: Loss de Entrenamiento", color='red', linestyle='--')
for epoch_evo in range(EVOLUTION_FREQ, EPOCHS + 1, EVOLUTION_FREQ):
    plt.axvline(x=epoch_evo, color='green', linestyle=':', linewidth=0.8, label='Evolución/Poda' if epoch_evo == EVOLUTION_FREQ else "")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Comparación de Pérdidas: Modelo Genético vs. Modelo Estándar")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print("\n--- Comparación Completa ---")
print(f"Pérdida en Test - Modelo Genético: {test_loss_genetico:.4f}")
print(f"Pérdida en Test - Modelo Estándar: {test_loss_estandar:.4f}")

In [ ]:


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_classification
from sklearn.cluster import KMeans

# Import the required library for plotting
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np # Import numpy

# Define the NeuralBiomesModel class here again or ensure it's defined in a previous cell and executed
# Assuming the NeuralBiomesModel class is defined in a previous cell and is available in the environment

# --- Data Preparation ---
# Generate synthetic classification data
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, n_classes=2, random_state=42)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long) # Use torch.long for classification targets
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# --- Model Initialization ---
# Define model parameters (adjust these based on your data and architecture)
input_dim = X_train_tensor.shape[1]
shared_features_dim = 64
memory_hidden_dim = 32
processing_output_dim = 16
output_dim = len(torch.unique(y_train_tensor)) # Number of classes

model = NeuralBiomesModel(input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss() # Use CrossEntropyLoss for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training Loop ---
losses = [] # Initialize the losses list
neuron_adjustments_history = [] # Initialize the history for plotting

num_epochs = 200 # Reduced for a quicker example

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    optimizer.zero_grad() # Zero the gradients

    # Forward pass
    outputs = model(X_train_tensor)

    # Calculate loss
    loss = criterion(outputs, y_train_tensor)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    # Record the loss
    losses.append(loss.item())

    # --- Placeholder for neuron adjustment tracking ---
    # This section is a placeholder to match the plotting code.
    # You need to implement the actual logic to track neuron adjustments
    # within your model's forward/backward pass or through custom hooks.
    # For now, we'll add some dummy data to prevent errors in the plotting section.
    dummy_adjustments = {}
    if epoch == 99:
       # Simulate some adjustments before evolution
       for i in range(processing_output_dim): # Assuming this layer is where pruning happens
           dummy_adjustments[i] = np.random.randint(5, 15) # Dummy adjustment counts
    elif epoch == 101:
         # Simulate adjustments after reset/evolution
         for i in range(processing_output_dim + 5): # Simulate adding 5 new neurons
             dummy_adjustments[i] = np.random.randint(0, 5)
    else:
         # Simulate ongoing adjustments
         if neuron_adjustments_history:
             last_adjustments = neuron_adjustments_history[-1]
             dummy_adjustments = last_adjustments.copy()
             for nid in dummy_adjustments:
                  dummy_adjustments[nid] += np.random.randint(0, 3)


    neuron_adjustments_history.append(dummy_adjustments)

    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# --- Evaluation (Optional) ---
model.eval() # Set model to evaluation mode
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs.data, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f'Accuracy on test set: {accuracy:.4f}')


# Visualización de la Pérdida
plt.figure(figsize=(12, 6))
plt.plot(losses, label='Pérdida de Entrenamiento (MSE)')
plt.title('Evolución de la Pérdida del Modelo')
plt.xlabel('Época')
plt.ylabel('Pérdida')
# The axvline at 100 is based on the original prompt's comment, indicating an "evolution" event
# Adjust this if your planned evolution happens at a different epoch
plt.axvline(x=100, color='r', linestyle='--', label='Evolución Activada')
plt.legend()
plt.grid(True)
plt.show()

# Visualización de Ajustes por Neurona (antes y después de la evolución)
plt.figure(figsize=(14, 7))
# Graficar ajustes de las neuronas en la época 99 (antes de la evolución)
# Ensure the epoch index exists in the history
if 99 < len(neuron_adjustments_history):
    adjustments_before_evo = neuron_adjustments_history[99]
    neuron_ids_before = list(adjustments_before_evo.keys())
    adj_counts_before = [adjustments_before_evo[n_id] for n_id in neuron_ids_before]

    plt.subplot(1, 2, 1)
    plt.bar([f"N{nid}" for nid in neuron_ids_before], adj_counts_before, color='skyblue')
    plt.title(f'Ajustes por Neurona Oculta (Época 99 - Antes de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()


# Graficar ajustes de las neuronas en la época 101 (después de la evolución, recién reseteados)
# o en una época posterior para ver la tendencia de las nuevas/existentes
# Ensure the epoch index exists in the history
if 101 < len(neuron_adjustments_history):
    adjustments_after_evo = neuron_adjustments_history[101] # O elige una época más tardía, ej. 150
    neuron_ids_after = list(adjustments_after_evo.keys())
    adj_counts_after = [adjustments_after_evo[n_id] for n_id in neuron_ids_after]

    plt.subplot(1, 2, 2)
    plt.bar([f"N{nid}" for nid in neuron_ids_after], adj_counts_after, color='lightgreen')
    plt.title(f'Ajustes por Neurona Oculta (Época 101 - Después de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- Dataset Sintético (igual al de tu modelo genético) ---
X_data = torch.randn(100, 4)
y_data = (X_data[:, 0]**2 + X_data[:, 1]**2 + X_data[:, 2] + 0.5 * torch.randn(100)).unsqueeze(1)

X_np = X_data.numpy()
y_np = y_data.numpy()
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()
X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()

# --- Clase Neurona Personalizada ---
class Neurona(nn.Module):
    _next_id = 0
    def __init__(self, input_size):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1))
        self.ajustes = 0
        self.generation = 0
        self.id = Neurona._next_id
        Neurona._next_id += 1

    def forward(self, x):
        # x puede ser (batch_size, input_size)
        # self.w es (input_size)
        # Usamos broadcasting para multiplicar x por w y luego sumamos las columnas
        return (x * self.w).sum(dim=1, keepdim=True) + self.b # keepdim para que la salida sea (batch_size, 1)

    def mutate_from(self, p1, p2=None, sigma=0.01, generation_num=0):
        with torch.no_grad():
            if p2:
                self.w.copy_(0.5 * p1.w + 0.5 * p2.w + sigma * torch.randn_like(self.w))
                self.b.copy_(0.5 * p1.b + 0.5 * p2.b + sigma * torch.randn_like(self.b))
            else:
                self.w.copy_(p1.w + sigma * torch.randn_like(self.w))
                self.b.copy_(p1.b + sigma * torch.randn_like(self.b))
        self.ajustes = 0
        self.generation = generation_num
        self.id = Neurona._next_id
        Neurona._next_id += 1

# --- Capa Genética ---
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])

    def forward(self, x):
        # x es (batch_size, input_size)
        # Cada neurona 'n' produce una salida (batch_size, 1)
        # Apilamos estas salidas para obtener (batch_size, num_neuronas)
        neuron_outputs = [n(x) for n in self.neuronas]
        return torch.cat(neuron_outputs, dim=1) # Concatenar a lo largo de la dimensión de features

    def registrar_ajustes(self):
        for n in self.neuronas:
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1
            if n.b.grad is not None and torch.any(n.b.grad != 0): # También considerar el bias para los ajustes
                n.ajustes += 1


    def evolucionar(self, generation_num):
        # Asegurarse de que el ID de la neurona se reinicia para las nuevas
        # NO reiniciar _next_id a 0 aquí, queremos IDs globales únicos en el tiempo.
        # Neurona._next_id = 0 # Eliminado para mantener IDs únicos globales

        print(f"\n[Evolución - Generación {generation_num}]")
        if len(self.neuronas) < 2:
            print("Insuficientes neuronas para evolución.")
            return

        valid_neuronas = [n for n in self.neuronas if hasattr(n, 'ajustes')]
        if len(valid_neuronas) < 2:
            print("Insuficientes neuronas válidas para seleccionar padres. Saltando evolución.")
            return

        # Ordenar por ajustes (menor es mejor)
        mejores = sorted(valid_neuronas, key=lambda n: n.ajustes)[:2]
        nuevas_neuronas = []
        podadas = 0

        for i, n in enumerate(self.neuronas):
            if n.ajustes > 10: # Criterio de poda: más de 10 ajustes
                nuevo = Neurona(n.w.size(0))
                nuevo.mutate_from(mejores[0], mejores[1], generation_num=generation_num)
                nuevas_neuronas.append(nuevo)
                podadas += 1
                # print(f"  - Neurona {n.id} podada y reemplazada.") # Desactivado para menos verbosidad
            else:
                nuevas_neuronas.append(n)

        # Resetear ajustes para todas las neuronas después de la evolución
        for n in nuevas_neuronas:
            n.ajustes = 0

        self.neuronas = nn.ModuleList(nuevas_neuronas)
        print(f"  => {podadas} neuronas reemplazadas.")

# --- Red Completa con Evolución (Tu Modelo Genético) ---
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.oculta = Capa(input_size, hidden_size)
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.oculta(x)
        x = F.relu(x)
        return self.salida(x)

    def registrar_ajustes(self):
        self.oculta.registrar_ajustes()

    def evolucionar(self, gen):
        self.oculta.evolucionar(gen)

# --- Modelo Estándar (sin neuronas genéticas) ---
class RedEstandar(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return self.fc2(x)

# --- Configuración de Parámetros ---
input_dim = 4
hidden_dim = 10
output_dim = 1
EPOCHS = 300
LR = 0.01

# --- Entrenamiento del Modelo Genético ---
print("\n" + "="*40)
print("--- Entrenando Red Genética ---")
print("="*40)
modelo_genetico = RedGenetica(input_dim, hidden_dim, output_dim)
opt_genetico = torch.optim.SGD(modelo_genetico.parameters(), lr=LR)
criterion = nn.MSELoss()

losses_genetico = []
for epoch in range(EPOCHS):
    modelo_genetico.train()
    opt_genetico.zero_grad()

    out = modelo_genetico(X_train)
    loss = criterion(out, y_train)
    loss.backward()

    modelo_genetico.registrar_ajustes() # Registra los ajustes antes de optimizar
    opt_genetico.step()

    losses_genetico.append(loss.item())

    if (epoch + 1) % 50 == 0:
        modelo_genetico.evolucionar((epoch + 1) // 50)

# --- Evaluación del Modelo Genético ---
modelo_genetico.eval()
with torch.no_grad():
    preds_genetico = modelo_genetico(X_test)
    test_loss_genetico = criterion(preds_genetico, y_test).item()
print(f"\n[Resultados Modelo Genético] Pérdida en test: {test_loss_genetico:.4f}")

# --- Entrenamiento del Modelo Estándar ---
print("\n" + "="*40)
print("--- Entrenando Red Estándar ---")
print("="*40)
modelo_estandar = RedEstandar(input_dim, hidden_dim, output_dim)
opt_estandar = torch.optim.SGD(modelo_estandar.parameters(), lr=LR)

losses_estandar = []
for epoch in range(EPOCHS):
    modelo_estandar.train()
    opt_estandar.zero_grad()

    out = modelo_estandar(X_train)
    loss = criterion(out, y_train)
    loss.backward()
    opt_estandar.step()

    losses_estandar.append(loss.item())

# --- Evaluación del Modelo Estándar ---
modelo_estandar.eval()
with torch.no_grad():
    preds_estandar = modelo_estandar(X_test)
    test_loss_estandar = criterion(preds_estandar, y_test).item()
print(f"\n[Resultados Modelo Estándar] Pérdida en test: {test_loss_estandar:.4f}")

# --- Visualización Comparativa ---
plt.figure(figsize=(12, 6))
plt.plot(losses_genetico, label="Genético: Loss de Entrenamiento", color='blue')
plt.plot(losses_estandar, label="Estándar: Loss de Entrenamiento", color='red', linestyle='--')
for epoch_evo in range(50, EPOCHS + 1, 50):
    plt.axvline(x=epoch_evo, color='green', linestyle=':', linewidth=0.8, label='Evolución/Poda' if epoch_evo == 50 else "")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Comparación de Pérdidas: Modelo Genético vs. Modelo Estándar")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print("\n--- Comparación Completa ---")
print(f"Pérdida en Test - Modelo Genético: {test_loss_genetico:.4f}")
print(f"Pérdida en Test - Modelo Estándar: {test_loss_estandar:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- Dataset Sintético ---
X_data = torch.randn(100, 4)  # 100 muestras, 4 características
y_data = (X_data[:, 0]**2 + X_data[:, 1]**2 + X_data[:, 2] + 0.5 * torch.randn(100)).unsqueeze(1)

X_np = X_data.numpy()
y_np = y_data.numpy()
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()
X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()

# --- Clase Neurona Personalizada ---
class Neurona(nn.Module):
    _next_id = 0
    def __init__(self, input_size):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1))
        self.ajustes = 0
        self.generation = 0
        self.id = Neurona._next_id
        Neurona._next_id += 1

    def forward(self, x):
        return torch.dot(self.w, x) + self.b

    def mutate_from(self, p1, p2=None, sigma=0.01, generation_num=0):
        with torch.no_grad():
            if p2:
                self.w.copy_(0.5 * p1.w + 0.5 * p2.w + sigma * torch.randn_like(self.w))
                self.b.copy_(0.5 * p1.b + 0.5 * p2.b + sigma * torch.randn_like(self.b))
            else:
                self.w.copy_(p1.w + sigma * torch.randn_like(self.w))
                self.b.copy_(p1.b + sigma * torch.randn_like(self.b))
        self.ajustes = 0
        self.generation = generation_num
        self.id = Neurona._next_id
        Neurona._next_id += 1

# --- Capa Genética ---
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])

    def forward(self, x):
        return torch.stack([n(sample) for n in self.neuronas for sample in x]).view(x.size(0), len(self.neuronas))

    def registrar_ajustes(self):
        for n in self.neuronas:
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1

    def evolucionar(self, generation_num):
        print(f"\n[Evolución - Generación {generation_num}]")
        if len(self.neuronas) < 2:
            print("Insuficientes neuronas para evolución.")
            return

        mejores = sorted(self.neuronas, key=lambda n: n.ajustes)[:2]
        nuevas_neuronas = []
        podadas = 0

        for i, n in enumerate(self.neuronas):
            if n.ajustes > 10:
                nuevo = Neurona(n.w.size(0))
                nuevo.mutate_from(mejores[0], mejores[1], generation_num=generation_num)
                nuevas_neuronas.append(nuevo)
                podadas += 1
                print(f"  - Neurona {n.id} podada y reemplazada.")
            else:
                nuevas_neuronas.append(n)

        self.neuronas = nn.ModuleList(nuevas_neuronas)
        print(f"  => {podadas} neuronas reemplazadas.")

# --- Red Completa con Evolución ---
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.oculta = Capa(input_size, hidden_size)
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.oculta(x)
        x = F.relu(x)
        return self.salida(x)

    def registrar_ajustes(self):
        self.oculta.registrar_ajustes()

    def evolucionar(self, gen):
        self.oculta.evolucionar(gen)

# --- Entrenamiento ---
input_dim = 4
hidden_dim = 10
output_dim = 1

modelo = RedGenetica(input_dim, hidden_dim, output_dim)
opt = torch.optim.SGD(modelo.parameters(), lr=0.01)
criterion = nn.MSELoss()

losses = []
podas_historial = []

print("--- Entrenando Red Genética ---")
for epoch in range(300):
    modelo.train()
    opt.zero_grad()

    out = modelo(X_train)
    loss = criterion(out, y_train)
    loss.backward()

    modelo.registrar_ajustes()
    opt.step()
    losses.append(loss.item())

    # Evolucionar cada 50 épocas
    if (epoch + 1) % 50 == 0:
        modelo.evolucionar((epoch + 1) // 50)

# --- Evaluación ---
modelo.eval()
with torch.no_grad():
    preds = modelo(X_test)
    test_loss = criterion(preds, y_test).item()
print(f"\nPérdida en test: {test_loss:.4f}")

# --- Visualización ---
plt.figure(figsize=(10, 4))
plt.plot(losses, label="Loss de entrenamiento")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Pérdida durante el entrenamiento")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Fijar semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- Dataset Sintético ---
# Para un problema de regresión o clasificación más interesante
# Generamos datos 2D para una mejor visualización si es necesario.
# Pero dado que tu input_size es 4, usaremos un X de 4 dimensiones.
X_data = torch.randn(100, 4) # 100 muestras, 4 features
# La salida depende de la suma de los cuadrados de los primeros dos features
y_data = (X_data[:, 0]**2 + X_data[:, 1]**2 + X_data[:, 2] + 0.5 * torch.randn(100)).unsqueeze(1) # Salida de 1 dimensión

# Convertir a numpy para usar train_test_split de sklearn
from sklearn.model_selection import train_test_split
X_np = X_data.numpy()
y_np = y_data.numpy()

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train = torch.from_numpy(X_train_np).float()
y_train = torch.from_numpy(y_train_np).float()
X_test = torch.from_numpy(X_test_np).float()
y_test = torch.from_numpy(y_test_np).float()


# --- Clase de Neurona con seguimiento de ajustes ---
class Neurona(nn.Module):
    _next_id = 0 # Contador global para IDs únicos de neuronas
    def __init__(self, input_size):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1))
        self.ajustes = 0 # Contador de ajustes por optimizador
        self.generation = 0 # Generación en la que fue creada/evolucionada
        self.id = Neurona._next_id # Asignar un ID único a cada neurona
        Neurona._next_id += 1


    def forward(self, x):
        # Asegurarse de que x sea compatible con el producto punto
        # Si x es un lote, se esperaría un mm.Linear. Esto es para una sola muestra.
        return torch.dot(self.w, x) + self.b

    def mutate_from(self, parent1, parent2=None, sigma=0.01, generation_num=0):
        with torch.no_grad():
            if parent2:
                # Cruce y mutación: promedio de padres + ruido
                self.w.copy_(0.5 * parent1.w + 0.5 * parent2.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(0.5 * parent1.b + 0.5 * parent2.b + torch.randn_like(self.b) * sigma)
            else:
                # Solo mutación de un padre
                self.w.copy_(parent1.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(parent1.b + torch.randn_like(self.b) * sigma)
        self.ajustes = 0 # Resetear ajustes al ser mutada/clonada
        self.generation = generation_num # Registrar la generación de origen
        self.id = Neurona._next_id # Asignar un nuevo ID si se desea un seguimiento de "nueva entidad"
        Neurona._next_id += 1 # Incrementar el contador global de IDs


# --- Capa de neuronas personalizadas ---
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])
        self.initial_num_neurons = num_neuronas # Guardar el número inicial

    def forward(self, x):
        # x puede ser un lote (batch_size, input_size)
        # Cada neurona procesa cada muestra del lote
        # El resultado será (batch_size, num_neuronas)
        return torch.stack([n(sample) for n in self.neuronas for sample in x]).view(x.size(0), len(self.neuronas))


    def registrar_ajustes(self):
        for n in self.neuronas:
            # Comprobar si el gradiente existe y si tiene algún valor no cero
            # Esto indica que la neurona contribuyó a la pérdida y fue ajustada
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1
            if n.b.grad is not None and torch.any(n.b.grad != 0):
                # Opcional: registrar ajustes también por el sesgo
                pass

    def evolucionar(self, generation_num):
        print(f"\n--- Evolución de neuronas en Capa (Generación {generation_num}) ---")

        # Ordenar por ajustes (menor es mejor)
        # Asegurarse de tener al menos 2 neuronas para selección
        if len(self.neuronas) < 2:
            print("¡Advertencia! Menos de 2 neuronas para la evolución. Saltando.")
            return

        # Filtrar las neuronas que no han sido inicializadas correctamente
        # o que tienen problemas en su 'ajustes' contador
        valid_neuronas = [n for n in self.neuronas if hasattr(n, 'ajustes')]

        if not valid_neuronas:
            print("No hay neuronas válidas para evolucionar.")
            return

        # Asegurarse de que tenemos al menos 2 neuronas válidas para la selección
        if len(valid_neuronas) < 2:
            print("Insuficientes neuronas válidas para seleccionar padres. Saltando evolución.")
            return

        mejores = sorted(valid_neuronas, key=lambda n: n.ajustes)[:2]

        # Listas para auditoría
        nuevas_neuronas = []
        podadas_info = []

        for i, n in enumerate(self.neuronas):
            if n.ajustes > 10: # Criterio de poda: más de 10 ajustes
                # Poda: registrar la neurona actual
                podadas_info.append(f"Neurona ID: {n.id} (Ajustes: {n.ajustes}) podada de la posición {i}.")

                # Reemplazo: crear una nueva neurona mutada de los mejores
                nuevo = Neurona(n.w.size(0)) # Mantener el mismo input_size
                nuevo.mutate_from(mejores[0], mejores[1], generation_num=generation_num)
                nuevas_neuronas.append(nuevo)
            else:
                # Si no se poda, mantener la neurona existente
                nuevas_neuronas.append(n)

        # Actualizar la lista de neuronas de la capa
        self.neuronas = nn.ModuleList(nuevas_neuronas)

        # Imprimir resumen de la evolución
        print(f"Número de neuronas podadas/reemplazadas en esta evolución: {len(podadas_info)}")
        for info in podadas_info:
            print(f"  - {info}")
        print(f"Neuronas restantes en la capa oculta: {len(self.neuronas)}\n")


# --- Red completa con una capa oculta genética y una capa de salida ---
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.oculta = Capa(input_size, hidden_size)
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # La capa oculta ahora maneja el lote
        x = self.oculta(x)
        x = F.relu(x)
        return self.salida(x)

    def registrar_ajustes(self):
        self.oculta.registrar_ajustes()

    def evolucionar(self, generation_num):
        self.oculta.evolucionar(generation_num)

# --- Entrenamiento Básico con Evolución ---
input_dim = 4
hidden_dim = 10
output_dim = 1 # Ajustado a la salida de tu dataset sintético

modelo = RedGenetica(input_size=input_dim, hidden_size=hidden_dim, output_size=output_dim)
opt = torch.optim.SGD(modelo.parameters(), lr=0.01)
criterion = nn.MSELoss() # Usamos MSE para el problema de regresión

# --- Conteo Inicial de Neuronas ---
print("--- Conteo de Neuronas al Inicio ---")
print(f"Capa de Entrada: {input_dim} neuronas (features)")
print(f"Capa Oculta: {len(modelo.oculta.neuronas)} neuronas (iniciales)")
print(f"Capa de Salida: {output_dim} neuronas")

# Historial para visualización
losses = []
neuron_adjustments_history = [] # Lista de diccionarios, uno por época
pruned_neurons_count_history = [] # Para el número de neuronas podadas por evolución

for epoch in range(300):
    # Asegúrate de que las neuronas tengan sus gradientes reiniciados
    for n in modelo.oculta.neuronas:
        if n.w.grad is not None:
            n.w.grad.zero_()
        if n.b.grad is not None:
            n.b.grad.zero_()

    # Cada época procesa todo el X_train
    modelo.train()
    opt.zero_grad()
    out = modelo(X_train)
    loss = criterion(out, y_train) # Usar y_train que tiene la misma dimensión de out
    loss.backward()

    # Registrar ajustes ANTES de que el optimizador los aplique y los gradientes se vacíen
    modelo.registrar_ajustes()
    opt.step()

    losses.append(loss.item())

    # Guardar el estado de ajustes de cada neurona
    current_adjustments = {n.id: n.ajustes for n in modelo.oculta.neuronas}
    neuron_adjustments_history.append(current_adjustments)

    if epoch == 100: # Activar la evolución en la época 100
        print("\n--- Activando la Evolución Genética de Neuronas ---")
        modelo.evolucionar(epoch)
        # Tras la evolución, resetear ajustes para la siguiente fase
        for n in modelo.oculta.neuronas:
            n.ajustes = 0 # Las nuevas neuronas inician en 0, las que se mantuvieron también

        # Registro de neuronas podadas (se actualiza dentro de evolucionar, esto es para visualización)
        pruned_neurons_count_history.append(len(modelo.oculta.neuronas)) # Actualizar el conteo de neuronas restantes

    if epoch % 50 == 0:
        modelo.eval()
        with torch.no_grad():
            test_loss = criterion(modelo(X_test), y_test)
        print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}")

# --- Conteo Final de Neuronas ---
print("\n--- Conteo de Neuronas al Final ---")
print(f"Capa de Entrada: {input_dim} neuronas (features)")
print(f"Capa Oculta: {len(modelo.oculta.neuronas)} neuronas (después de evolución)")
print(f"Capa de Salida: {output_dim} neuronas")


### Visualización de Resultados


# --- Visualización de la Pérdida ---
plt.figure(figsize=(12, 6))
plt.plot(losses, label='Pérdida de Entrenamiento (MSE)')
plt.title('Evolución de la Pérdida del Modelo')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.axvline(x=100, color='r', linestyle='--', label='Evolución Activada')
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización de Ajustes por Neurona (antes y después de la evolución) ---
plt.figure(figsize=(14, 7))
# Graficar ajustes de las neuronas en la época 99 (antes de la evolución)
if 99 < len(neuron_adjustments_history):
    adjustments_before_evo = neuron_adjustments_history[99]
    neuron_ids_before = list(adjustments_before_evo.keys())
    adj_counts_before = [adjustments_before_evo[n_id] for n_id in neuron_ids_before]

    plt.subplot(1, 2, 1)
    plt.bar([f"N{nid}" for nid in neuron_ids_before], adj_counts_before, color='skyblue')
    plt.title(f'Ajustes por Neurona Oculta (Época 99 - Antes de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()


# Graficar ajustes de las neuronas en la época 101 (después de la evolución, recién reseteados)
# o en una época posterior para ver la tendencia de las nuevas/existentes
if 101 < len(neuron_adjustments_history):
    adjustments_after_evo = neuron_adjustments_history[101] # O elige una época más tardía, ej. 150
    neuron_ids_after = list(adjustments_after_evo.keys())
    adj_counts_after = [adjustments_after_evo[n_id] for n_id in neuron_ids_after]

    plt.subplot(1, 2, 2)
    plt.bar([f"N{nid}" for nid in neuron_ids_after], adj_counts_after, color='lightgreen')
    plt.title(f'Ajustes por Neurona Oculta (Época 101 - Después de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()

plt.tight_layout()
plt.show()

# Visualización de la evolución del número de neuronas (si la poda redujera el número)
# En este caso, el número es fijo porque se reemplazan, no se eliminan.
# Si hubieras implementado reducción de neuronas, esta gráfica sería útil.
plt.figure(figsize=(8, 5))
plt.plot(pruned_neurons_count_history, label='Número de Neuronas Ocultas')
plt.title('Número de Neuronas Ocultas a lo Largo de las Épocas')
plt.xlabel('Época de Evolución') # Cuando se activó la evolución
plt.ylabel('Número de Neuronas')
plt.show()

print("\n--- Ejecución completada ---")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Fijar semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- Dataset Sintético ---
# Para un problema de regresión o clasificación más interesante
# Generamos datos 2D para una mejor visualización si es necesario.
# Pero dado que tu input_size es 4, usaremos un X de 4 dimensiones.
X_data = torch.randn(100, 4) # 100 muestras, 4 features
# La salida depende de la suma de los cuadrados de los primeros dos features
y_data = (X_data[:, 0]**2 + X_data[:, 1]**2 + X_data[:, 2] + 0.5 * torch.randn(100)).unsqueeze(1) # Salida de 1 dimensión

# Convertir a numpy para usar train_test_split de sklearn
from sklearn.model_selection import train_test_split
X_np = X_data.numpy()
y_np = y_data.numpy()

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train = torch.from_numpy(X_train_np).float()
y_train = torch.from_numpy(y_train_np).float()
X_test = torch.from_numpy(X_test_np).float()
y_test = torch.from_numpy(y_test_np).float()


# --- Clase de Neurona con seguimiento de ajustes ---
class Neurona(nn.Module):
    _next_id = 0 # Contador global para IDs únicos de neuronas
    def __init__(self, input_size):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1))
        self.ajustes = 0 # Contador de ajustes por optimizador
        self.generation = 0 # Generación en la que fue creada/evolucionada
        self.id = Neurona._next_id # Asignar un ID único a cada neurona
        Neurona._next_id += 1


    def forward(self, x):
        # Asegurarse de que x sea compatible con el producto punto
        # Si x es un lote, se esperaría un mm.Linear. Esto es para una sola muestra.
        return torch.dot(self.w, x) + self.b

    def mutate_from(self, parent1, parent2=None, sigma=0.01, generation_num=0):
        with torch.no_grad():
            if parent2:
                # Cruce y mutación: promedio de padres + ruido
                self.w.copy_(0.5 * parent1.w + 0.5 * parent2.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(0.5 * parent1.b + 0.5 * parent2.b + torch.randn_like(self.b) * sigma)
            else:
                # Solo mutación de un padre
                self.w.copy_(parent1.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(parent1.b + torch.randn_like(self.b) * sigma)
        self.ajustes = 0 # Resetear ajustes al ser mutada/clonada
        self.generation = generation_num # Registrar la generación de origen
        self.id = Neurona._next_id # Asignar un nuevo ID si se desea un seguimiento de "nueva entidad"
        Neurona._next_id += 1 # Incrementar el contador global de IDs


# --- Capa de neuronas personalizadas ---
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])
        self.initial_num_neurons = num_neuronas # Guardar el número inicial

    def forward(self, x):
        # x puede ser un lote (batch_size, input_size)
        # Cada neurona procesa cada muestra del lote
        # El resultado será (batch_size, num_neuronas)
        return torch.stack([n(sample) for n in self.neuronas for sample in x]).view(x.size(0), len(self.neuronas))


    def registrar_ajustes(self):
        for n in self.neuronas:
            # Comprobar si el gradiente existe y si tiene algún valor no cero
            # Esto indica que la neurona contribuyó a la pérdida y fue ajustada
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1
            if n.b.grad is not None and torch.any(n.b.grad != 0):
                # Opcional: registrar ajustes también por el sesgo
                pass

    def evolucionar(self, generation_num):
        print(f"\n--- Evolución de neuronas en Capa (Generación {generation_num}) ---")

        # Ordenar por ajustes (menor es mejor)
        # Asegurarse de tener al menos 2 neuronas para selección
        if len(self.neuronas) < 2:
            print("¡Advertencia! Menos de 2 neuronas para la evolución. Saltando.")
            return

        # Filtrar las neuronas que no han sido inicializadas correctamente
        # o que tienen problemas en su 'ajustes' contador
        valid_neuronas = [n for n in self.neuronas if hasattr(n, 'ajustes')]

        if not valid_neuronas:
            print("No hay neuronas válidas para evolucionar.")
            return

        # Asegurarse de que tenemos al menos 2 neuronas válidas para la selección
        if len(valid_neuronas) < 2:
            print("Insuficientes neuronas válidas para seleccionar padres. Saltando evolución.")
            return

        mejores = sorted(valid_neuronas, key=lambda n: n.ajustes)[:2]

        # Listas para auditoría
        nuevas_neuronas = []
        podadas_info = []

        for i, n in enumerate(self.neuronas):
            if n.ajustes > 10: # Criterio de poda: más de 10 ajustes
                # Poda: registrar la neurona actual
                podadas_info.append(f"Neurona ID: {n.id} (Ajustes: {n.ajustes}) podada de la posición {i}.")

                # Reemplazo: crear una nueva neurona mutada de los mejores
                nuevo = Neurona(n.w.size(0)) # Mantener el mismo input_size
                nuevo.mutate_from(mejores[0], mejores[1], generation_num=generation_num)
                nuevas_neuronas.append(nuevo)
            else:
                # Si no se poda, mantener la neurona existente
                nuevas_neuronas.append(n)

        # Actualizar la lista de neuronas de la capa
        self.neuronas = nn.ModuleList(nuevas_neuronas)

        # Imprimir resumen de la evolución
        print(f"Número de neuronas podadas/reemplazadas en esta evolución: {len(podadas_info)}")
        for info in podadas_info:
            print(f"  - {info}")
        print(f"Neuronas restantes en la capa oculta: {len(self.neuronas)}\n")


# --- Red completa con una capa oculta genética y una capa de salida ---
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.oculta = Capa(input_size, hidden_size)
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # La capa oculta ahora maneja el lote
        x = self.oculta(x)
        x = F.relu(x)
        return self.salida(x)

    def registrar_ajustes(self):
        self.oculta.registrar_ajustes()

    def evolucionar(self, generation_num):
        self.oculta.evolucionar(generation_num)

# --- Entrenamiento Básico con Evolución ---
input_dim = 4
hidden_dim = 10
output_dim = 1 # Ajustado a la salida de tu dataset sintético

modelo = RedGenetica(input_size=input_dim, hidden_size=hidden_dim, output_size=output_dim)
opt = torch.optim.SGD(modelo.parameters(), lr=0.01)
criterion = nn.MSELoss() # Usamos MSE para el problema de regresión

# --- Conteo Inicial de Neuronas ---
print("--- Conteo de Neuronas al Inicio ---")
print(f"Capa de Entrada: {input_dim} neuronas (features)")
print(f"Capa Oculta: {len(modelo.oculta.neuronas)} neuronas (iniciales)")
print(f"Capa de Salida: {output_dim} neuronas")

# Historial para visualización
losses = []
neuron_adjustments_history = [] # Lista de diccionarios, uno por época
pruned_neurons_count_history = [] # Para el número de neuronas podadas por evolución

for epoch in range(300):
    # Asegúrate de que las neuronas tengan sus gradientes reiniciados
    for n in modelo.oculta.neuronas:
        if n.w.grad is not None:
            n.w.grad.zero_()
        if n.b.grad is not None:
            n.b.grad.zero_()

    # Cada época procesa todo el X_train
    modelo.train()
    opt.zero_grad()
    out = modelo(X_train)
    loss = criterion(out, y_train) # Usar y_train que tiene la misma dimensión de out
    loss.backward()

    # Registrar ajustes ANTES de que el optimizador los aplique y los gradientes se vacíen
    modelo.registrar_ajustes()
    opt.step()

    losses.append(loss.item())

    # Guardar el estado de ajustes de cada neurona
    current_adjustments = {n.id: n.ajustes for n in modelo.oculta.neuronas}
    neuron_adjustments_history.append(current_adjustments)

    if epoch == 100: # Activar la evolución en la época 100
        print("\n--- Activando la Evolución Genética de Neuronas ---")
        modelo.evolucionar(epoch)
        # Tras la evolución, resetear ajustes para la siguiente fase
        for n in modelo.oculta.neuronas:
            n.ajustes = 0 # Las nuevas neuronas inician en 0, las que se mantuvieron también

        # Registro de neuronas podadas (se actualiza dentro de evolucionar, esto es para visualización)
        pruned_neurons_count_history.append(len(modelo.oculta.neuronas)) # Actualizar el conteo de neuronas restantes

    if epoch % 50 == 0:
        modelo.eval()
        with torch.no_grad():
            test_loss = criterion(modelo(X_test), y_test)
        print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}")

# --- Conteo Final de Neuronas ---
print("\n--- Conteo de Neuronas al Final ---")
print(f"Capa de Entrada: {input_dim} neuronas (features)")
print(f"Capa Oculta: {len(modelo.oculta.neuronas)} neuronas (después de evolución)")
print(f"Capa de Salida: {output_dim} neuronas")


### Visualización de Resultados


# --- Visualización de la Pérdida ---
plt.figure(figsize=(12, 6))
plt.plot(losses, label='Pérdida de Entrenamiento (MSE)')
plt.title('Evolución de la Pérdida del Modelo')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.axvline(x=100, color='r', linestyle='--', label='Evolución Activada')
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización de Ajustes por Neurona (antes y después de la evolución) ---
plt.figure(figsize=(14, 7))
# Graficar ajustes de las neuronas en la época 99 (antes de la evolución)
if 99 < len(neuron_adjustments_history):
    adjustments_before_evo = neuron_adjustments_history[99]
    neuron_ids_before = list(adjustments_before_evo.keys())
    adj_counts_before = [adjustments_before_evo[n_id] for n_id in neuron_ids_before]

    plt.subplot(1, 2, 1)
    plt.bar([f"N{nid}" for nid in neuron_ids_before], adj_counts_before, color='skyblue')
    plt.title(f'Ajustes por Neurona Oculta (Época 99 - Antes de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()


# Graficar ajustes de las neuronas en la época 101 (después de la evolución, recién reseteados)
# o en una época posterior para ver la tendencia de las nuevas/existentes
if 101 < len(neuron_adjustments_history):
    adjustments_after_evo = neuron_adjustments_history[101] # O elige una época más tardía, ej. 150
    neuron_ids_after = list(adjustments_after_evo.keys())
    adj_counts_after = [adjustments_after_evo[n_id] for n_id in neuron_ids_after]

    plt.subplot(1, 2, 2)
    plt.bar([f"N{nid}" for nid in neuron_ids_after], adj_counts_after, color='lightgreen')
    plt.title(f'Ajustes por Neurona Oculta (Época 101 - Después de Evolución)')
    plt.xlabel('ID de Neurona')
    plt.ylabel('Número de Ajustes')
    plt.xticks(rotation=90)
    plt.axhline(y=10, color='red', linestyle='--', label='Umbral de Poda (10 ajustes)')
    plt.legend()

plt.tight_layout()
plt.show()

# Visualización de la evolución del número de neuronas (si la poda redujera el número)
# En este caso, el número es fijo porque se reemplazan, no se eliminan.
# Si hubieras implementado reducción de neuronas, esta gráfica sería útil.
# plt.figure(figsize=(8, 5))
# plt.plot(pruned_neurons_count_history, label='Número de Neuronas Ocultas')
# plt.title('Número de Neuronas Ocultas a lo Largo de las Épocas')
# plt.xlabel('Época de Evolución') # Cuando se activó la evolución
# plt.ylabel('Número de Neuronas')
# plt.show()

print("\n--- Ejecución completada ---")

In [ ]:
import numpy as np

# --- Simulación de una neurona simple para ejemplificar ---
# En un caso real, esto sería una neurona dentro de una red neuronal más grande
class SimpleNeuron:
    def __init__(self, weights_shape):
        self.weights = np.random.randn(*weights_shape) * 0.01 # Pesos aleatorios pequeños
        self.bias = np.random.randn() * 0.01 # Sesgo aleatorio pequeño
        self.activation_count = 0 # Contador para las actualizaciones por gradiente

    def forward(self, inputs):
        # Una activación simple (ej. lineal o ReLU para simplificar)
        return np.dot(inputs, self.weights) + self.bias

    def backward(self, inputs, output_error, learning_rate):
        # Simulación de una actualización por descenso de gradiente
        # Esto sería mucho más complejo en una red real (cálculo de gradientes)
        self.weights -= learning_rate * np.dot(inputs.T, output_error)
        self.bias -= learning_rate * np.sum(output_error)
        self.activation_count += 1 # Incrementa el contador por cada actualización

# --- Algoritmo Genético Simplificado para Neuronas ---

def genetic_algorithm_for_neurons(population_size, input_dim, output_dim, generations, learning_rate, prune_threshold):
    # Crear una población inicial de neuronas
    # En un escenario real, 'input_dim' y 'output_dim' serían las dimensiones de los pesos de la neurona
    population = [SimpleNeuron((input_dim, output_dim)) for _ in range(population_size)]

    for gen in range(generations):
        print(f"\n--- Generación {gen + 1} ---")

        # Simular entrenamiento (actualizaciones por gradiente)
        # Aquí se simula un proceso de entrenamiento donde las neuronas se actualizan.
        # En la vida real, se entrenaría la red neuronal completa.
        for i, neuron in enumerate(population):
            # Simular datos de entrada y error (aleatorios para el ejemplo)
            dummy_inputs = np.random.rand(10, input_dim)
            dummy_output_error = np.random.rand(10, output_dim) - 0.5 # Error aleatorio centrado en cero

            # Simular múltiples actualizaciones para algunas neuronas
            num_updates = np.random.randint(1, 5) # Algunas neuronas se actualizan más que otras
            for _ in range(num_updates):
                neuron.backward(dummy_inputs, dummy_output_error, learning_rate)

            # print(f"Neurona {i}: Actualizaciones de gradiente = {neuron.activation_count}")

        # --- Fase de "Poda" y "Reemplazo" ---
        pruned_neurons_indices = []
        for i, neuron in enumerate(population):
            if neuron.activation_count > prune_threshold:
                pruned_neurons_indices.append(i)
                print(f"  Neurona {i} podada (activaciones: {neuron.activation_count})")

        if not pruned_neurons_indices:
            print("  Ninguna neurona podada en esta generación.")
            continue # Si no se podó nada, continuar a la siguiente generación

        # Seleccionar las "mejores" neuronas (las que menos se activaron)
        # Ordenar por el número de activaciones de menor a mayor
        sorted_neurons = sorted(population, key=lambda n: n.activation_count)

        # Tomar las N mejores para la "reproducción"
        num_to_select = min(len(pruned_neurons_indices), len(sorted_neurons) - len(pruned_neurons_indices)) # No seleccionar las mismas que se podaron
        if num_to_select <= 0:
            print("  No hay suficientes neuronas 'buenas' para reemplazar a las podadas.")
            continue

        best_neurons_for_breeding = sorted_neurons[:num_to_select]

        print(f"  Neuronas seleccionadas para 'cruza' (mejores): {[n.activation_count for n in best_neurons_for_breeding]}")

        # Reemplazar las neuronas podadas con "descendencia" de las mejores
        for i, pruned_idx in enumerate(pruned_neurons_indices):
            if i < len(best_neurons_for_breeding):
                parent1 = best_neurons_for_breeding[i % len(best_neurons_for_breeding)] # Para reciclar si hay más podadas que buenas
                parent2 = best_neurons_for_breeding[(i + 1) % len(best_neurons_for_breeding)]

                # --- "Cruza" (combinación de pesos y sesgos) ---
                # Esto es una forma muy simplificada. Se pueden usar cruzas más complejas.
                new_neuron = SimpleNeuron((input_dim, output_dim))
                new_neuron.weights = (parent1.weights + parent2.weights) / 2 # Promedio simple
                new_neuron.bias = (parent1.bias + parent2.bias) / 2

                # --- "Mutación" (pequeño cambio aleatorio) ---
                mutation_rate = 0.05
                new_neuron.weights += np.random.randn(*new_neuron.weights.shape) * mutation_rate
                new_neuron.bias += np.random.randn() * mutation_rate

                new_neuron.activation_count = 0 # Reiniciar contador para la nueva neurona
                population[pruned_idx] = new_neuron
                print(f"  Neurona {pruned_idx} reemplazada por una nueva.")
            else:
                print(f"  Neurona {pruned_idx} no pudo ser reemplazada (no hay suficientes padres).")

# --- Parámetros de la simulación ---
POPULATION_SIZE = 10 # Número de neuronas en nuestra "población"
INPUT_DIM = 5      # Dimensión de las entradas de la neurona
OUTPUT_DIM = 2     # Dimensión de las salidas de la neurona
GENERATIONS = 5    # Número de "generaciones" del algoritmo genético
LEARNING_RATE = 0.01 # Tasa de aprendizaje para las actualizaciones de gradiente
PRUNE_THRESHOLD = 3 # Si una neurona se actualiza más de este número, se "poda"

# Ejecutar la simulación
genetic_algorithm_for_neurons(POPULATION_SIZE, INPUT_DIM, OUTPUT_DIM, GENERATIONS, LEARNING_RATE, PRUNE_THRESHOLD)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Fijar semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- 1. Generar el Dataset Sintético ---
# Un único feature de entrada (X)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # Un único feature de entrada
# Salida (y) con una relación cúbica y ruido
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 2. Dividir en conjuntos de Entrenamiento y Prueba ---
# Convertir a numpy para usar train_test_split de sklearn
X_np = X.numpy()
y_np = y.numpy()

# Dividir el 80% para entrenamiento y 20% para prueba
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

# Convertir de nuevo a tensores de PyTorch
X_train = torch.from_numpy(X_train_np).float()
y_train = torch.from_numpy(y_train_np).float()
X_test = torch.from_numpy(X_test_np).float()
y_test = torch.from_numpy(y_test_np).float()

# --- 3. Visualizar los conjuntos de Entrenamiento y Prueba ---
plt.figure(figsize=(10, 6))

# Visualizar el conjunto de entrenamiento
plt.scatter(X_train.numpy(), y_train.numpy(), color='blue', label='Datos de Entrenamiento', alpha=0.6)

# Visualizar el conjunto de prueba
plt.scatter(X_test.numpy(), y_test.numpy(), color='red', label='Datos de Prueba', alpha=0.6)

plt.title('Dataset Sintético: Entrenamiento vs. Prueba')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True)
plt.show()

print(f"Tamaño del conjunto de entrenamiento: X={X_train.shape}, y={y_train.shape}")
print(f"Tamaño del conjunto de prueba: X={X_test.shape}, y={y_test.shape}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# Para el AG, usaremos una simulación o una librería básica de AG si no quieres PyGAD
# Para este ejemplo inicial, nos centraremos en la estructura y el conteo.

# Fijar semillas para reproducibilidad
torch.manual_seed(0)
np.random.seed(0)

# --- 1. Generar el Dataset Sintético (como en el paso anterior) ---
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

X_np = X.numpy()
y_np = y.numpy()

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train = torch.from_numpy(X_train_np).float()
y_train = torch.from_numpy(y_train_np).float()
X_test = torch.from_numpy(X_test_np).float()
y_test = torch.from_numpy(y_test_np).float()

# --- 2. Definir una Red Neuronal Simple en PyTorch ---
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size) # Capa de entrada a oculta
        self.relu = nn.ReLU()                         # Función de activación
        self.fc2 = nn.Linear(hidden_size, output_size) # Capa oculta a salida

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Parámetros de la red
input_size = 1     # Un único feature de entrada (X)
hidden_size = 20   # Número de neuronas en la capa oculta
output_size = 1    # Un único valor de salida (y)

# Instanciar el modelo
model = SimpleNN(input_size, hidden_size, output_size)

# --- 3. Contabilizar Neuronas al Inicio del "Entrenamiento" (antes de cualquier AG o GD) ---
print("--- Conteo de Neuronas al Inicio ---")
# Una 'neurona' en una capa densa se corresponde con una salida de esa capa.
# Cada fila de la matriz de pesos de la siguiente capa se conecta a una "neurona" de la capa actual.

# Capa de entrada (implícita, número de features de entrada)
num_input_neurons_initial = input_size
print(f"Capa de Entrada: {num_input_neurons_initial} neuronas (features de entrada)")

# Capa oculta
num_hidden_neurons_initial = model.fc1.out_features
print(f"Capa Oculta: {num_hidden_neurons_initial} neuronas")

# Capa de salida
num_output_neurons_initial = model.fc2.out_features
print(f"Capa de Salida: {num_output_neurons_initial} neuronas")

# --- 4. Simulación del Entrenamiento con Algoritmo Genético (Conceptual) ---
# Aquí es donde el Algoritmo Genético operaría.
# En un GA real, el "cromosoma" sería un vector plano de todos los pesos y sesgos del modelo.
# El GA iteraría, evaluaría el fitness (por ejemplo, MSE en este caso),
# y generaría nuevas "soluciones" (nuevos conjuntos de pesos).

# Para simular, vamos a "entrenar" la red con un optimizador simple (GD),
# pero la idea es que el AG reemplazaría este proceso.

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss() # Para problemas de regresión

epochs = 500 # Un número moderado de épocas para "entrenar"
print("\n--- Simulación de Entrenamiento (aquí iría el AG) ---")
for epoch in range(epochs):
    model.train() # Pone el modelo en modo entrenamiento
    optimizer.zero_grad()
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        model.eval() # Pone el modelo en modo evaluación
        with torch.no_grad():
            test_predictions = model(X_test)
            test_loss = criterion(test_predictions, y_test)
            print(f'Epoch [{epoch+1}/{epochs}], Training Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

# --- 5. Contabilizar Neuronas al Final del Entrenamiento ---
# Si la poda se basa en un criterio de pesos (ej., si los pesos de una neurona son casi cero)
# entonces esta parte es donde examinarías los pesos.

print("\n--- Conteo de Neuronas al Final (Post-Entrenamiento/AG) ---")
# Por defecto, en una red densa estándar, el número de neuronas no cambia
# a menos que implementes explícitamente una poda estructural.

num_input_neurons_final = input_size
print(f"Capa de Entrada: {num_input_neurons_final} neuronas (features de entrada)")

num_hidden_neurons_final = model.fc1.out_features
print(f"Capa Oculta: {num_hidden_neurons_final} neuronas")

num_output_neurons_final = model.fc2.out_features
print(f"Capa de Salida: {num_output_neurons_final} neuronas")

# --- 6. Cuales fueron podadas y en que capa (Conceptual para Neuronas Genéticas) ---

# Concepto de Poda en Neuronas Genéticas:
# En una implementación real de "neuronas genéticas" con poda, la "poda" no es solo
# hacer que los pesos sean cero, sino que implica una eliminación *estructural* o un
# reemplazo. Para esto, necesitarías:

# A. Definir un CRITERIO DE PODA:
#    - "Menor número de activación por el gradiente": Esto es complejo de medir directamente
#      después del entrenamiento. Una proxy podría ser:
#        - Neuronas con activaciones promedio muy bajas (cercanas a cero) en un dataset.
#        - Neuronas con pesos salientes muy pequeños (cercanos a cero).
#        - Neuronas que contribuyen mínimamente a la reducción del error.
#    - Si tu AG "entrena" pesos, podrías buscar neuronas cuyos pesos están muy cerca de cero
#      o cuyos sesgos son insignificantes, lo que indicaría que esa neurona es "redundante"
#      o "inactiva".

# B. Implementación de Poda y Reemplazo:
#    Si identificas neuronas para podar:
#    1. Podrías modificar dinámicamente la arquitectura de tu `SimpleNN`
#       (eliminando la capa y recreándola con menos neuronas, o enmascarando los pesos).
#    2. En el contexto de un AG, la "poda" se traduciría en manipular el genoma:
#       - Los cromosomas que representan redes con "neuronas podadas" podrían tener
#         secciones de pesos/sesgos desactivadas o directamente eliminadas.
#       - El "reemplazo" significaría que, al cruzar o mutar, las "partes"
#         correspondientes a las neuronas podadas se rellenarían con valores
#         "heredados" de las mejores neuronas, o con mutaciones dirigidas.

# Ejemplo CONCETPTUAL de identificación de neuronas "casi muertas" después del entrenamiento
# (esto NO es una poda activa, solo una observación basada en pesos pequeños)
print("\n--- Criterio de Poda Conceptual (Basado en Pesos Bajos) ---")
threshold_for_pruning = 0.01 # Un umbral para considerar un peso como "pequeño"

# Evaluar la capa oculta (fc1)
# Si una neurona tiene todos sus pesos de salida (o entrada) muy pequeños, podría ser un candidato.
# Esto es una simplificación extrema, ya que la "muerte" de una neurona es más compleja.

# Obtenemos los pesos de la primera capa oculta (fc1) y la segunda capa (fc2)
# Los pesos de `fc1` son (hidden_size, input_size) y los de `fc2` son (output_size, hidden_size)
hidden_layer_weights = model.fc1.weight.detach().numpy()
output_layer_weights = model.fc2.weight.detach().numpy()

# Una neurona en la capa oculta podría ser "podada" si sus pesos que van a la capa de salida
# son todos muy pequeños.
# Esto solo es ilustrativo; un criterio real sería más sofisticado.

# Para cada neurona en la capa oculta, revisamos los pesos que salen de ella.
neurons_to_consider_pruned = []
for i in range(hidden_size):
    # Verificamos si todos los pesos que salen de esta neurona oculta hacia la capa de salida
    # son menores que el umbral.
    # Los pesos de fc2 son (output_size, hidden_size). La columna 'i' son los pesos que *entran*
    # a la capa de salida desde la neurona 'i' de la capa oculta.
    # Entonces, los pesos que *salen* de la neurona 'i' de la capa oculta son la i-ésima columna
    # de la matriz de pesos de fc2.
    outgoing_weights_from_hidden_neuron_i = np.abs(output_layer_weights[:, i])

    if np.all(outgoing_weights_from_hidden_neuron_i < threshold_for_pruning):
        neurons_to_consider_pruned.append(i)

if neurons_to_consider_pruned:
    print(f"Neuronas potencialmente 'podadas' en la Capa Oculta (índices): {neurons_to_consider_pruned}")
    print(f"Basado en: Todos los pesos salientes de esa neurona son menores que {threshold_for_pruning}")
    print(f"Número de neuronas 'podadas' conceptualmente: {len(neurons_to_consider_pruned)}")
else:
    print("No se encontraron neuronas 'podadas' conceptualmente con el umbral dado en la capa oculta.")

# Visualización de la predicción final
model.eval()
with torch.no_grad():
    predictions_final = model(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(X_test.numpy(), y_test.numpy(), color='red', label='Datos de Prueba Reales', alpha=0.6)
plt.scatter(X_test.numpy(), predictions_final.numpy(), color='green', label='Predicciones del Modelo', alpha=0.6)
plt.title('Predicciones del Modelo Final')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Datos sintéticos
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15,
                           n_redundant=5, n_classes=2, random_state=42)
X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Parámetros
INPUT_DIM = 20
HIDDEN_DIM = 10
OUTPUT_DIM = 2
INITIAL_EPOCHS = 100
TOTAL_EPOCHS = 900
PRUNE_THRESHOLD = 10

# Neurona con seguimiento de ajustes
class EvolvableNeuron(nn.Module):
    def __init__(self, in_features, identifier):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(1))
        self.adjustments = 0
        self.id = identifier

    def forward(self, x):
        return torch.matmul(x, self.weight) + self.bias

    def mutate_from(self, parent1, parent2):
        with torch.no_grad():
            self.weight.copy_((parent1.weight + parent2.weight) / 2 + torch.randn_like(self.weight) * 0.05)
            self.bias.copy_((parent1.bias + parent2.bias) / 2 + torch.randn_like(self.bias) * 0.05)
            self.adjustments = 0

# Capa personalizada
class EvolvableHiddenLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.neurons = nn.ModuleList([EvolvableNeuron(in_features, f"N{i}") for i in range(out_features)])
        self.out_features = out_features

    def forward(self, x):
        outputs = [torch.relu(neuron(x)) for neuron in self.neurons]
        return torch.stack(outputs, dim=1)

    def track_adjustments(self, grads):
        for i, neuron in enumerate(self.neurons):
            if neuron.weight.grad is not None:
                neuron.adjustments += 1

    def evolve(self):
        survivors = [n for n in self.neurons if n.adjustments <= PRUNE_THRESHOLD]
        sorted_neurons = sorted(self.neurons, key=lambda n: n.adjustments)
        best1, best2 = sorted_neurons[0], sorted_neurons[1]
        while len(survivors) < self.out_features:
            new_id = f"N{np.random.randint(10000)}"
            new_neuron = EvolvableNeuron(best1.weight.shape[0], new_id)
            new_neuron.mutate_from(best1, best2)
            survivors.append(new_neuron)
        self.neurons = nn.ModuleList(survivors)

# Red neuronal completa
class EvolvableNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = EvolvableHiddenLayer(INPUT_DIM, HIDDEN_DIM)
        self.output = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)

    def forward(self, x):
        x = self.hidden(x)
        x = torch.relu(x)
        x = self.output(x)
        return x

# Entrenamiento
model = EvolvableNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []

for epoch in range(TOTAL_EPOCHS):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()

    # Registrar ajustes por neurona (solo capa oculta)
    model.hidden.track_adjustments([p.grad for p in model.parameters()])

    optimizer.step()
    losses.append(loss.item())

    if epoch == INITIAL_EPOCHS:
        model.hidden.evolve()

final_accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
final_accuracy, losses[-10:]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Clase de Neurona con seguimiento de ajustes
class Neurona(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        # Weights shape should be (input_size,) for dot product, but for matrix multiplication
        # in a batched context, it's easier if the weight is (input_size, 1) or the input
        # is handled appropriately. Let's keep it (input_size,) and handle in forward.
        self.w = nn.Parameter(torch.randn(input_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(1)) # Bias is a scalar per neuron
        self.ajustes = 0
        self.id = id(self)

    def forward(self, x):
        # x shape: (batch_size, input_size)
        # w shape: (input_size,)
        # We need to perform element-wise multiplication and sum across input_size for each batch item
        # (batch_size, input_size) * (input_size,) -> (batch_size, input_size)
        # Then sum across dimension 1 -> (batch_size,)
        # Add bias (scalar or (1,)) broadcasting to (batch_size,)
        output = torch.sum(x * self.w, dim=1) + self.b
        return output # Shape: (batch_size,)

    def mutate_from(self, parent1, parent2=None, sigma=0.01):
        with torch.no_grad():
            if parent2:
                self.w.copy_(0.5 * parent1.w + 0.5 * parent2.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(0.5 * parent1.b + 0.5 * parent2.b + torch.randn_like(self.b) * sigma)
            else:
                self.w.copy_(parent1.w + torch.randn_like(self.w) * sigma)
                self.b.copy_(parent1.b + torch.randn_like(self.b) * sigma)
        self.ajustes = 0

# Capa de neuronas personalizadas (adaptada para manejar batches)
class Capa(nn.Module):
    def __init__(self, input_size, num_neuronas):
        super().__init__()
        # ModuleList holds the individual Neurona modules
        self.neuronas = nn.ModuleList([Neurona(input_size) for _ in range(num_neuronas)])
        self.num_neuronas = num_neuronas

    def forward(self, x):
        # x shape: (batch_size, input_size)
        # Output shape should be (batch_size, num_neuronas)

        # Option 1: Process each neuron's output and stack (less efficient)
        # outputs = [neuron(x) for neuron in self.neuronas] # Each neuron(x) returns (batch_size,)
        # return torch.stack(outputs, dim=1) # Stacking along dim 1 gives (batch_size, num_neuronas)

        # Option 2: Combine weights and biases for efficient matrix multiplication
        # This requires Neurona's forward not to do the full dot product,
        # or combine weights/biases at the Capa level. Let's stick to Option 1
        # for now as it respects the individual Neurona module structure,
        # but acknowledge this is less efficient than a single matrix multiply.

        # Option 1 implementation:
        # x shape: (batch_size, input_size)
        # Individual neuron forward returns (batch_size,)
        # Stack along dimension 1 to get (batch_size, num_neuronas)
        # Ensure x has a batch dimension if it's a single sample (input_size,)
        if x.ndim == 1:
            x = x.unsqueeze(0) # Add batch dimension: (input_size,) -> (1, input_size)

        outputs = []
        for neuron in self.neuronas:
             # neuron(x) will return (batch_size,)
             outputs.append(neuron(x))

        # Stack outputs along dimension 1 to get (batch_size, num_neuronas)
        stacked_outputs = torch.stack(outputs, dim=1)

        # If the input was a single sample, remove the added batch dimension from the output
        if x.shape[0] == 1 and x.ndim == 2 and stacked_outputs.ndim == 2: # Check if batch was added
             stacked_outputs = stacked_outputs.squeeze(0) # (1, num_neuronas) -> (num_neuronas,)


        # Re-checking shapes based on original error: input to linear was (10x1), linear was (10,3).
        # This implies input expected was (10, 10) (batch_size, hidden_size), but got (10, 1).
        # The output of Capa should be (batch_size, num_neuronas) == (batch_size, hidden_size).
        # If input is (4), Capa output should be (10,). After unsqueeze(0) in Capa, input is (1,4).
        # Each neuron produces (1,). Stacking 10 of these gives (1, 10).
        # Squeezing gives (10,).
        # Maybe the squeeze(0) should NOT happen if the linear layer expects (batch_size, hidden_size)?
        # Yes, the linear layer expects (batch_size, in_features). If input is (10,), linear assumes (1, 10).
        # The error (10x1) is very strange. It seems the (10,) tensor is being treated as a COLUMN vector (10, 1).

        # Let's try forcing the output to be (batch_size, num_neuronas) explicitly
        # If input x is (4,), unsqueeze(0) -> (1, 4)
        # Neuron output: (1,)
        # Stacked outputs: (num_neuronas, 1) e.g., (10, 1)
        # Permute to (1, num_neuronas) e.g., (1, 10)
        # Ah, torch.stack(outputs) on (1,) tensors gives (num_neuronas, 1).
        # We need (batch_size, num_neuronas). The current neuron(x) returns (batch_size,).
        # So stacking [ (batch_size,), (batch_size,), ... ] along dim=1 gives (batch_size, num_neuronas).
        # This should be correct assuming neuron(x) returns (batch_size,) when x is (batch_size, input_size).

        # Let's rethink Neurona.forward for batch processing:
        # Input x: (batch_size, input_size)
        # self.w: (input_size)
        # self.b: (1,) or scalar
        # Desired output: (batch_size,)
        # This is equivalent to a linear transformation without bias followed by summing, then adding bias.
        # Example: batch size 2, input_size 3, w (3,), b scalar
        # x = [[x11, x12, x13], [x21, x22, x23]]
        # x * w = [[x11*w1, x12*w2, x13*w3], [x21*w1, x22*w2, x23*w3]]
        # torch.sum(x * w, dim=1) = [x11*w1+x12*w2+x13*w3, x21*w1+x22*w2+x23*w3] (batch_size,)
        # This implementation of Neurona.forward seems correct for batches.

        # The issue must be when the input x to Capa is (4,) (single sample).
        # Capa.forward: if x.ndim == 1: x = x.unsqueeze(0) -> x is now (1, 4)
        # outputs = [neuron(x) for neuron in self.neuronas]
        # neuron(x) with x (1, 4) returns torch.sum((1, 4) * (4,), dim=1) + bias.
        # (1, 4) * (4,) -> (1, 4) element-wise product due to broadcasting
        # torch.sum((1, 4), dim=1) -> (1,) tensor
        # So, each neuron(x) returns a (1,) tensor when input x is (1, 4).
        # outputs list contains 10 tensors of shape (1,).
        # torch.stack(outputs, dim=1) -> stacks 10 tensors of shape (1,) along dim 1.
        # The result will be shape (1, 10) -> (batch_size, num_neuronas).

        # This shape (1, 10) IS the correct input shape for self.salida (nn.Linear(10, 3))
        # when batch_size is 1.
        # Multiplication should be (1, 10) @ (10, 3) -> (1, 3).

        # The error message (10x1 and 10x3) implies the input to nn.Linear is seen as (10, 1).
        # This could happen if the output of Capa was (10,) and it was explicitly
        # unsqueezed to (10, 1) somewhere, or if torch.stack in Capa was stacking along dim=0
        # instead of dim=1, producing (num_neuronas, batch_size).
        # Let's check the stack dimension. `torch.stack(outputs, dim=1)` where outputs are (batch_size,)
        # results in (batch_size, len(outputs)). This is correct.

        # Maybe the issue is simply that the input `x` in the loop `x = torch.randn(4)`
        # should have an explicit batch dimension, even if it's 1.
        # Let's change `x = torch.randn(4)` to `x = torch.randn(1, 4)`.
        # Then Capa's unsqueeze(0) is not needed and won't be executed.
        # Capa.forward: x is (1, 4).
        # outputs = [neuron(x) for neuron in self.neuronas]. neuron(x) returns (1,).
        # torch.stack(outputs, dim=1) -> stacks 10 tensors of shape (1,) along dim=1 -> (1, 10).
        # This shape (1, 10) is correctly passed to nn.Linear(10, 3).

        # Let's test this fix.

        # Corrected Capa forward logic:
        original_shape_is_1d = (x.ndim == 1)
        if original_shape_is_1d:
            x = x.unsqueeze(0) # Add batch dimension: (input_size,) -> (1, input_size)

        outputs = []
        for neuron in self.neuronas:
             # neuron(x) will return (batch_size,)
             outputs.append(neuron(x))

        # Stack outputs along dimension 1 to get (batch_size, num_neuronas)
        stacked_outputs = torch.stack(outputs, dim=1)

        # The output of the hidden layer should be (batch_size, hidden_size)
        # The linear layer expects (batch_size, in_features)
        # Our stacked_outputs is (batch_size, num_neuronas), which is (batch_size, hidden_size).
        # This is the correct format for the subsequent nn.Linear layer.
        # Do NOT squeeze here, even if the input was originally 1D.
        # The linear layer should always receive (batch_size, num_features).

        return stacked_outputs # Shape: (batch_size, num_neuronas)


    def registrar_ajustes(self):
        for n in self.neuronas:
            # Check if the gradient exists and is not all zeros
            # Using torch.any is safer than checking against 0 directly for float tensors
            if n.w.grad is not None and torch.any(n.w.grad != 0):
                n.ajustes += 1
            # Also check bias gradient
            if n.b.grad is not None and torch.any(n.b.grad != 0):
                 n.ajustes += 1 # Or count separately if needed


    def evolucionar(self):
        # Ordenar por ajustes (menor es mejor)
        # Ensure there are at least 2 neurons to select as parents
        if len(self.neuronas) < 2:
             print("Warning: Not enough neurons for evolution.")
             return

        mejores = sorted(self.neuronas, key=lambda n: n.ajustes)[:2]
        nuevas = []
        pruned_count = 0
        for i, n in enumerate(self.neuronas):
            # Use a threshold (e.g., 10 adjustments) to decide if a neuron is "podada" (needs replacement)
            if n.ajustes > 10:
                pruned_count += 1
                # Create a new neuron and mutate it from the best parents
                # Ensure the new neuron has the correct input size
                nuevo = Neurona(self.neuronas[0].w.shape[0]) # Get input_size from an existing neuron
                nuevo.mutate_from(mejores[0], mejores[1])
                nuevas.append(nuevo)
                #print(f"  Neurona {n.id} podada (ajustes: {n.ajustes}). Reemplazada por nueva neuron ({nuevo.id}).")
            else:
                # Keep the old neuron if it was not pruned
                nuevas.append(n)

        if pruned_count > 0:
             print(f"  {pruned_count} neurona(s) podada(s) y reemplazada(s).")
        else:
             print("  Ninguna neurona podada en esta generación.")

        # Replace the old list with the new list of neurons
        self.neuronas = nn.ModuleList(nuevas)


# Red completa con una capa oculta genética y una capa de salida
class RedGenetica(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # Pass input_size and num_neuronas to Capa
        self.oculta = Capa(input_size, hidden_size)
        # The output of 'oculta' will be (batch_size, hidden_size)
        # The input to the linear layer should match the hidden_size
        self.salida = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, input_size) or (input_size,) for a single sample
        x = self.oculta(x) # Capa handles adding batch dim if needed, returns (batch_size, hidden_size)
        x = F.relu(x)       # Shape remains (batch_size, hidden_size)
        # Pass (batch_size, hidden_size) to the linear layer
        return self.salida(x) # Should return (batch_size, output_size)

    def registrar_ajustes(self):
        # Registrar ajustes en la capa genética (oculta)
        self.oculta.registrar_ajustes()

    def evolucionar(self):
        # Evolucionar la capa genética (oculta)
        self.oculta.evolucionar()

# Entrenamiento básico con evolución
# input_size=4, hidden_size=10, output_size=3 as per original code
modelo = RedGenetica(input_size=4, hidden_size=10, output_size=3)
opt = torch.optim.SGD(modelo.parameters(), lr=0.01)

# Example training loop - using explicit batches
num_epochs = 300
batch_size = 1 # Keep batch size 1 to match the original loop structure, but use (1, input_size) input
samples_per_epoch = 10 # Number of training samples processed per epoch

print("Iniciando entrenamiento...")
for epoch in range(num_epochs):
    epoch_loss = 0
    for i in range(samples_per_epoch):
        # Generate a single random sample as a batch of size 1
        x = torch.randn(batch_size, 4) # Shape (1, 4)
        y = torch.tensor([[1, 0, 0]], dtype=torch.float) # Shape (1, 3)

        # Ensure y has the correct shape (batch_size, output_size)
        if y.ndim == 1: # If y is (output_size,)
             y = y.unsqueeze(0) # Add batch dimension -> (1, output_size)

        out = modelo(x) # model expects (batch_size, input_size), returns (batch_size, output_size)
        loss = F.mse_loss(out, y)

        opt.zero_grad()
        loss.backward()

        # Registrar ajustes *after* backward pass (gradients are available)
        modelo.registrar_ajustes()

        opt.step()
        epoch_loss += loss.item()

    avg_epoch_loss = epoch_loss / samples_per_epoch


    if epoch == 100:
        print("\nActivando evolución genética en la capa oculta...\n")
        modelo.evolucionar()
        # After evolution, optimizers might need to be re-initialized or handle
        # changed parameters. For SGD, it might be ok, but be cautious with Adam etc.
        # Re-initializing for safety if network structure changed.
        # NOTE: This simple evolution replaces neurons. A real AG might just update weights.
        # If structure changes, optimizer needs updating.
        opt = torch.optim.SGD(modelo.parameters(), lr=0.01)


    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {avg_epoch_loss:.4f}")

print("\nEntrenamiento finalizado.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import copy
import random

# Configuraciones
INPUT_SIZE = 784  # MNIST 28x28
HIDDEN_SIZE = 32
OUTPUT_SIZE = 10
NUM_EPOCHS = 900
GENETIC_START_EPOCH = 100
GENETIC_THRESHOLD = 10
BATCH_SIZE = 64

# Datos
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Capa Oculta Evolutiva
class EvolvableLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weights = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.biases = nn.Parameter(torch.zeros(out_features))

        self.adjust_count = torch.zeros(out_features)
        self.ids = [f'N{i}' for i in range(out_features)]

    def forward(self, x):
        self.last_input = x.detach()
        return F.linear(x, self.weights, self.biases)

    def update_adjust_count(self, weight_grads, bias_grads):
        with torch.no_grad():
            for i in range(self.out_features):
                if weight_grads[i].abs().sum() > 1e-5 or bias_grads[i].abs().sum() > 1e-5:
                    self.adjust_count[i] += 1

    def evolve(self):
        # Selección de las 2 mejores neuronas (menos ajustadas)
        sorted_indices = torch.argsort(self.adjust_count)
        best_idx1 = sorted_indices[0].item()
        best_idx2 = sorted_indices[1].item()

        to_replace = (self.adjust_count >= GENETIC_THRESHOLD).nonzero(as_tuple=False).view(-1)

        for idx in to_replace:
            idx = idx.item()
            parent_idx = random.choice([best_idx1, best_idx2])
            self.weights.data[idx] = self.weights.data[parent_idx] + torch.randn_like(self.weights.data[parent_idx]) * 0.02
            self.biases.data[idx] = self.biases.data[parent_idx] + torch.randn_like(self.biases.data[parent_idx]) * 0.02
            self.adjust_count[idx] = 0
            self.ids[idx] = f'N{idx}_mut'

# Red completa
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = EvolvableLayer(INPUT_SIZE, HIDDEN_SIZE)
        self.output = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.hidden(x))
        return self.output(x)

# Entrenamiento
model = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        # Contador de ajustes por neurona
        model.hidden.update_adjust_count(model.hidden.weights.grad, model.hidden.biases.grad)

        optimizer.step()
        total_loss += loss.item()

    if epoch == GENETIC_START_EPOCH:
        print(f"\n[Genética Activada] Epoch {epoch}")

    if epoch > GENETIC_START_EPOCH:
        model.hidden.evolve()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}, Ajustes promedio: {model.hidden.adjust_count.mean():.2f}")

# Evaluación
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()

print(f"\nPrecisión final en test: {100 * correct / len(test_dataset):.2f}%")